# Token Prepending on Kaggle T4 x2

Notebook self-contained for the paper code. It writes the Python source tree into `/kaggle/working/token_prepending`, installs only the Kaggle-needed packages, then runs Vietnamese STS evaluation by default.

Before running on Kaggle: set **Settings -> Accelerator -> GPU T4 x2** and enable Internet for first-time Hugging Face/dataset downloads. An `.ipynb` cannot force the exact Kaggle hardware by itself, so the runtime check below fails early if two CUDA devices are not visible.

## Review Notes

- TP is implemented by copying the previous layer last-token hidden state into the `<PST>` position for an early layer window.
- Do not use `--tensor_parallel` for TP here: that code path loads a vanilla `AutoModelForCausalLM` and bypasses the custom `senllm` forward methods.
- Default task is `vi-sts`; English SentEval tasks need external `SentEval/data`.
- Batch size defaults to `1` to keep 7B inference stable on 2 x 16 GB T4.

In [ ]:
# Kaggle runtime setup
import os
import sys
import subprocess

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", "/kaggle/working/hf-cache")
os.environ.setdefault("HF_DATASETS_CACHE", "/kaggle/working/hf-cache/datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/kaggle/working/hf-cache/transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

packages = [
    "transformers==4.46.3",
    "accelerate>=0.26.0",
    "datasets>=2.19.0",
    "scipy",
    "scikit-learn",
    "pandas",
    "prettytable",
    "colorama",
    "pyvi==0.1.1",
    "sentencepiece>=0.1.99",
    "pyyaml",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependency setup complete")


In [ ]:
# Verify Kaggle T4 x2 visibility
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Kaggle, choose Settings -> Accelerator -> GPU T4 x2.")

gpu_count = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
print("CUDA devices:", gpu_names)

if gpu_count < 2:
    raise RuntimeError(f"Expected 2 CUDA GPUs for T4 x2, found {gpu_count}.")
if not all("T4" in name for name in gpu_names[:2]):
    print("Warning: first two GPUs are not both named T4. Continuing with visible GPUs.")


In [ ]:
# Write runtime/evaluation files
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/token_prepending")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "evaluate.py": "import re\nimport sys\nimport io, os\n\nDEFAULT_LOCAL_HF_HOME = os.path.join(os.path.dirname(__file__), \".cache\", \"huggingface\")\nos.environ.setdefault(\"HF_HOME\", DEFAULT_LOCAL_HF_HOME)\nos.environ.setdefault(\"HF_DATASETS_CACHE\", os.path.join(DEFAULT_LOCAL_HF_HOME, \"datasets\"))\n\nimport torch\nimport numpy as np\nimport logging\nimport tqdm\nimport fcntl\nimport time\nimport argparse\nfrom prettytable import PrettyTable\nimport transformers\nfrom transformers import LlamaTokenizer\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache\nfrom senllm import LlamaForCausalLM, Qwen2ForCausalLM, Gemma2ForCausalLM\nfrom model_runtime import (\n    DEFAULT_CACHE_DIR,\n    DEFAULT_MAC_7B_MODEL,\n    build_loading_kwargs,\n    move_model_to_device_if_needed,\n    resolve_torch_device,\n)\nfrom colorama import Fore, Style\nimport textwrap\nfrom scipy.stats import spearmanr\nimport numpy as np\nimport yaml\nfrom datasets import load_dataset\nfrom vietnamese_sts import (\n    DEFAULT_VIETNAMESE_STS_DATASET,\n    DEFAULT_VIETNAMESE_STS_SPLIT,\n    cosine_similarity_scores,\n    preprocess_sentence_for_prompt,\n    resolve_sts_columns,\n    select_dataset_split,\n)\nimport warnings\nwarnings.filterwarnings(\"ignore\")\n\n\nif torch.cuda.is_available():\n    print(\"We are using GPU!\")\n    torch.cuda.manual_seed(3407)\n    torch.cuda.manual_seed_all(3407)\n\nCOEFF = float(os.environ.get(\"COEFF\", 1.0))\n\n# Set up logger\nlogging.basicConfig(format='%(asctime)s : %(message)s', level=logging.DEBUG)\n\n# Set PATHs\nPATH_TO_SENTEVAL = './SentEval'\nPATH_TO_DATA = './SentEval/data'\n\n# Import SentEval\nsys.path.insert(0, PATH_TO_SENTEVAL)\nimport senteval\n\ndef print_table(task_names, scores):\n    tb = PrettyTable()\n    tb.field_names = task_names\n    tb.add_row(scores)\n    print(tb)\n\ndef lock_and_write_file(file_path, content):\n    with open(file_path, 'a') as file:\n        while True:\n            try:\n                # Acquire an exclusive lock (non-blocking)\n                fcntl.flock(file, fcntl.LOCK_EX | fcntl.LOCK_NB)\n\n                # Perform your write operations here\n                file.write(content + '\\n')\n                file.flush()\n\n            except IOError as e:\n                print(\"File is locked by another process. Can't write.\")\n                time.sleep(1)\n            finally:\n                # Release the lock\n                fcntl.flock(file, fcntl.LOCK_UN)\n                break\n\ndef load_config_from_yaml(config_file=\"config.yaml\", config_name=None):\n\n    try:\n        with open(config_file, 'r', encoding='utf-8') as f:\n            yaml_config = yaml.safe_load(f)\n    except FileNotFoundError:\n        print(f\"warning: config file {config_file} not found, using command line parameters\")\n        return None\n    except yaml.YAMLError as e:\n        print(f\"error: config file {config_file} format error: {e}\")\n        return None\n    \n    if config_name is None:\n        config_name = yaml_config.get('default_config', 'llama-2-7b')\n    \n    if config_name not in yaml_config.get('models', {}):\n        available_configs = list(yaml_config.get('models', {}).keys())\n        print(f\"error: config '{config_name}' not found\")\n        print(f\"available configs: {available_configs}\")\n        return None\n    \n    # \u83b7\u53d6\u6307\u5b9a\u914d\u7f6e\n    config = yaml_config['models'][config_name].copy()\n    \n    # \u6dfb\u52a0GPU\u914d\u7f6e\n    if 'gpu_config' in yaml_config:\n        config['gpu_config'] = yaml_config['gpu_config']\n    \n    print(f\"\u2713 successfully loaded config: {config_name}\")\n    return config\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\"--config\", type=str, default=None,\n                        help=\"config name, read parameters from config.yaml\")\n    parser.add_argument(\"--config_file\", type=str, default=\"config.yaml\",\n                        help=\"config file path\")\n    \n\n    parser.add_argument(\"--tokenizer_name\", type=str, \n                        default='')\n    parser.add_argument(\"--model_name_or_path\", type=str,\n                        default=DEFAULT_MAC_7B_MODEL,\n                        help=\"Transformers' model name or path\")\n    parser.add_argument(\"--mode\", type=str,\n                        choices=['dev', 'test', 'fasttest'],\n                        default='test',\n                        help=\"What evaluation mode to use (dev: fast mode, dev results; test: full mode, test results); fasttest: fast mode, test results\")\n    parser.add_argument(\"--task_set\", type=str,\n                        choices=['sts', 'transfer', 'full', 'na', 'stsb', 'vi-sts'],\n                        default='sts',\n                        help=\"What set of tasks to evaluate on. If not 'na', this will override '--tasks'\")\n    parser.add_argument('--tensor_parallel', action='store_true')\n    parser.add_argument('--prompt_method', type=str, \n                        default='prompteol', choices=['prompteol', 'metaeol', 'cot', 'ke'], help=\"What prompt method to use.\")\n    parser.add_argument(\"--use_which_plan\", type=str,\n                        choices=['tp', 'vanilla'],\n                        default='tp')\n    parser.add_argument(\"--output_layer\", type=int, \n                        default=-1)\n    parser.add_argument(\"--tp_starting_index\", type=int, \n                        default=1)\n    parser.add_argument(\"--tp_exiting_index\", type=int, \n                        default=99)\n    parser.add_argument(\"--batch_size\", type=int, \n                        default=16)\n    parser.add_argument(\"--device\", type=str,\n                        choices=[\"auto\", \"mps\", \"cuda\", \"cpu\"],\n                        default=\"auto\")\n    parser.add_argument(\"--cache_dir\", type=str,\n                        default=DEFAULT_CACHE_DIR)\n    parser.add_argument(\"--vietnamese_dataset_name\", type=str,\n                        default=DEFAULT_VIETNAMESE_STS_DATASET)\n    parser.add_argument(\"--vietnamese_split\", type=str,\n                        default=DEFAULT_VIETNAMESE_STS_SPLIT)\n\n    args = parser.parse_args()\n    \n    if args.config:\n        config = load_config_from_yaml(args.config_file, args.config)\n        if config is None:\n            print(\"config loading failed, exit program\")\n            sys.exit(1)\n        \n        args.model_name_or_path = config.get('model_name_or_path', args.model_name_or_path)\n        args.use_which_plan = config.get('use_which_plan', args.use_which_plan)\n        args.output_layer = config.get('output_layer', args.output_layer)\n        args.tp_starting_index = config.get('tp_starting_index', args.tp_starting_index)\n        args.tp_exiting_index = config.get('tp_exiting_index', args.tp_exiting_index)\n        args.batch_size = config.get('batch_size', args.batch_size)\n        args.mode = config.get('mode', args.mode)\n        args.task_set = config.get('task_set', args.task_set)\n        args.prompt_method = config.get('prompt_method', args.prompt_method)\n        args.device = config.get('device', args.device)\n        args.cache_dir = config.get('cache_dir', args.cache_dir)\n        args.vietnamese_dataset_name = config.get('vietnamese_dataset_name', args.vietnamese_dataset_name)\n        args.vietnamese_split = config.get('vietnamese_split', args.vietnamese_split)\n        \n        should_configure_cuda = args.device == \"cuda\" or (\n            args.device == \"auto\"\n            and torch.cuda.is_available()\n            and not torch.backends.mps.is_available()\n        )\n        if should_configure_cuda and 'gpu_config' in config and 'cuda_visible_devices' in config['gpu_config']:\n            os.environ['CUDA_VISIBLE_DEVICES'] = config['gpu_config']['cuda_visible_devices']\n            print(f\"\u2713 set GPU devices: {config['gpu_config']['cuda_visible_devices']}\")\n    \n    if not args.model_name_or_path:\n        print(\"error: model path not specified, please use --model_name_or_path parameter or specify it in the config file\")\n        sys.exit(1)\n    hyper_parameters = textwrap.dedent(f\"\"\"\n        {Fore.CYAN}Configuration:{Style.RESET_ALL}\n        {Fore.YELLOW}-------------{Style.RESET_ALL}\n        {Fore.GREEN}Backbone                :{Style.RESET_ALL} {args.model_name_or_path.split('/')[-1]}\n        {Fore.GREEN}Prompt Method           :{Style.RESET_ALL} {args.prompt_method}\n        {Fore.GREEN}Output Layer Index      :{Style.RESET_ALL} {args.output_layer}\n        {Fore.GREEN}Plan                    :{Style.RESET_ALL} {args.use_which_plan}\n        {Fore.GREEN}TP Starting layer Index :{Style.RESET_ALL} {args.tp_starting_index}\n        {Fore.GREEN}TP Exiting layer Index  :{Style.RESET_ALL} {args.tp_exiting_index}\n        {Fore.GREEN}Batch Size              :{Style.RESET_ALL} {args.batch_size}\n        {Fore.GREEN}Task Set                :{Style.RESET_ALL} {args.task_set}\n        {Fore.GREEN}Device                  :{Style.RESET_ALL} {args.device}\n        {Fore.GREEN}Cache Dir               :{Style.RESET_ALL} {args.cache_dir}\n    \"\"\")\n\n    print(hyper_parameters)\n\n    selected_device = resolve_torch_device(args.device)\n    cache_dir = args.cache_dir or None\n    tokenizer_source = args.tokenizer_name or args.model_name_or_path\n    tokenizer = AutoTokenizer.from_pretrained(\n        tokenizer_source,\n        cache_dir=cache_dir,\n        trust_remote_code=True,\n    )\n    tokenizer.pad_token_id = 0  # Set the padding token. we want this to be different from the eos token\n    tokenizer.padding_side = \"left\"  # Allow batched inference\n\n    if args.tensor_parallel:\n        if selected_device.type != \"cuda\":\n            raise RuntimeError(\"--tensor_parallel requires CUDA GPUs.\")\n        import tensor_parallel as tp\n        n_gpus = len(os.environ['CUDA_VISIBLE_DEVICES'].split(','))\n        model = AutoModelForCausalLM.from_pretrained(\n            args.model_name_or_path,\n            cache_dir=cache_dir,\n            low_cpu_mem_usage=True,\n            torch_dtype=torch.float16,\n        )\n        model = tp.tensor_parallel(model, [i for i in range(n_gpus)])\n    else:\n        if args.use_which_plan == 'tp':\n            placeholder_token = '<PST>'\n            tokenizer.add_tokens([placeholder_token])\n            placeholder_token_id = tokenizer.convert_tokens_to_ids(placeholder_token)\n\n        loading_kwargs = build_loading_kwargs(\n            selected_device.type,\n            cache_dir=cache_dir,\n            output_hidden_states=True,\n        )\n\n        if args.use_which_plan == 'tp':\n            loading_kwargs[\"ignore_mismatched_sizes\"] = True\n\n        def load_model_with_mps_fallback(model_cls):\n            try:\n                return model_cls.from_pretrained(args.model_name_or_path, **loading_kwargs)\n            except Exception as exc:\n                is_mps_device_map_error = (\n                    selected_device.type == \"mps\"\n                    and \"device_map\" in loading_kwargs\n                    and \"device_map\" in str(exc).lower()\n                )\n                if not is_mps_device_map_error:\n                    raise\n                print(\"warning: MPS device_map loading failed; retrying then moving model to MPS.\")\n                fallback_kwargs = dict(loading_kwargs)\n                fallback_kwargs.pop(\"device_map\", None)\n                return model_cls.from_pretrained(args.model_name_or_path, **fallback_kwargs)\n\n        if 'llama' in args.model_name_or_path.lower():\n            model = load_model_with_mps_fallback(LlamaForCausalLM)\n            model.model.plan = args.use_which_plan\n            model.model.tp_starting_index = args.tp_starting_index\n            model.model.tp_exiting_index = args.tp_exiting_index\n            if args.use_which_plan == 'tp':\n                model.model.placeholder_token_id = placeholder_token_id\n        elif 'qwen2' in args.model_name_or_path.lower():\n            model = load_model_with_mps_fallback(Qwen2ForCausalLM)\n            model.model.plan = args.use_which_plan\n            model.model.tp_starting_index = args.tp_starting_index\n            model.model.tp_exiting_index = args.tp_exiting_index\n            if args.use_which_plan == 'tp':\n                model.model.placeholder_token_id = placeholder_token_id\n        elif 'gemma' in args.model_name_or_path.lower():\n            model = load_model_with_mps_fallback(Gemma2ForCausalLM)\n            model.model.plan = args.use_which_plan\n            model.model.tp_starting_index = args.tp_starting_index\n            model.model.tp_exiting_index = args.tp_exiting_index\n            if args.use_which_plan == 'tp':\n                model.model.placeholder_token_id = placeholder_token_id\n        else:\n            raise ValueError(f\"Cannot find such {args.model_name_or_path.lower()} model!\")\n\n\n    if args.use_which_plan == 'tp':        \n        if hasattr(model, 'lm_head') and hasattr(model.lm_head, '_hf_hook'):\n            import accelerate\n            accelerate.hooks.remove_hook_from_module(model.lm_head)\n\n        model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=None, mean_resizing=False)\n        \n        # update internal vocab size tracking for the custom senllm model\n        if hasattr(model, 'vocab_size'):\n            model.vocab_size = len(tokenizer)\n        if hasattr(model.config, 'vocab_size'):\n            model.config.vocab_size = len(tokenizer)\n        if hasattr(model.model, 'vocab_size'):\n            model.model.vocab_size = len(tokenizer)\n            \n        embedding_layer = model.get_input_embeddings()\n        \n        if hasattr(model, 'lm_head'):\n            model.lm_head.to(embedding_layer.weight.device)\n            \n        embedding_layer.weight.requires_grad_(False)\n        \n        num_dim = embedding_layer.weight.shape[1]\n        device = embedding_layer.weight.device\n        \n        with torch.no_grad():\n            embedding_layer.weight[placeholder_token_id] = torch.randn(num_dim, device=device)\n        embedding_layer.weight.requires_grad_(True)\n\n    model = move_model_to_device_if_needed(model, selected_device)\n    model.eval()\n\n    device = selected_device\n\n    # Set up the tasks\n    if args.task_set == 'sts':\n        args.tasks = ['STS12', 'STS13', 'STS14', 'STS15', 'STS16', 'STSBenchmark', 'SICKRelatedness']\n        # args.tasks = ['STS16', 'STSBenchmark']\n        if args.mode == 'dev':\n            args.tasks = ['STSBenchmark-dev']\n    elif args.task_set == 'transfer':\n        args.tasks = ['MR', 'CR', 'MPQA', 'SUBJ', 'SST2', 'TREC', 'MRPC']\n    elif args.task_set == 'full':\n        args.tasks = ['STS12', 'STS13', 'STS14', 'STS15', 'STS16', 'STSBenchmark', 'SICKRelatedness']\n        args.tasks += ['MR', 'CR', 'MPQA', 'SUBJ', 'SST2', 'TREC', 'MRPC']\n    elif args.task_set == 'stsb':\n        args.tasks = ['STSBenchmark']\n    elif args.task_set == 'vi-sts':\n        args.tasks = ['VietnameseSTS']\n    # Set params for SentEval\n    if args.mode == 'dev' or args.mode == 'fasttest':\n        # Fast mode\n        params = {'task_path': PATH_TO_DATA, 'usepytorch': True, 'kfold': 5, 'batch_size': 32}\n        params['classifier'] = {'nhid': 0, 'optim': 'rmsprop', 'batch_size': 32,\n                                         'tenacity': 3, 'epoch_size': 2}\n    elif args.mode == 'test':\n        # Full mode\n        params = {'task_path': PATH_TO_DATA, 'usepytorch': True, 'kfold': 10, 'batch_size':args.batch_size}\n        params['classifier'] = {'nhid': 0, 'optim': 'adam', 'batch_size': 64,\n                                         'tenacity': 5, 'epoch_size': 4}\n    else:\n        raise NotImplementedError\n\n    # SentEval prepare and batcher\n    def prepare(params, samples):\n        return\n\n    if args.prompt_method == \"metaeol\":\n        if args.use_which_plan == 'tp':\n            task_prompts = [\"In this task, you're presented with a text excerpt. Your task is to categorize the excerpt into a broad category such as 'Education', 'Technology', 'Health', 'Business', 'Environment', 'Politics', or 'Culture'. These categories help in organizing content for better accessibility and targeting. For this task, this sentence : <PST> \\\"*sent 0*\\\" should be classified under one general category in one word:\\\"\",\n                            \"In this task, you're given a statement and you need to determine whether it's presenting an 'Opinion' or a 'Fact'. This distinction is vital for information verification, educational purposes, and content analysis. For this task, this sentence : <PST> \\\"*sent 0*\\\" discriminates between opinion and fact in one word:\\\"\",\n                            \"In this task, you're given a review from an online platform. Your task is to generate a rating for the product based on the review on a scale of 1-5, where 1 means 'extremely negative' and 5 means 'extremely positive'. For this task, this sentence : <PST> \\\"*sent 0*\\\" reflects the sentiment in one word:\\\"\",\n                            \"In this task, you're reading a personal diary entry. Your task is to identify the predominant emotion expressed, such as joy, sadness, anger, fear, or love. For this task, this sentence : <PST> \\\"*sent 0*\\\" conveys the emotion in one word:\\\"\",\n                            \"In this task, you're presented with two sentences. Your task is to assess whether the sentences convey the same meaning. Use 'identical', 'similar', 'different', or 'unrelated' to describe the relationship. To enhance the performance of this task, this sentence : <PST> \\\"*sent 0*\\\" means in one word:\\\"\",\n                            \"In this task, you're given a sentence and a phrase. Your task is to determine if the phrase can be a contextual synonym within the given sentence. Options include 'yes', 'no', or 'partially'. To enhance the performance of this task, this sentence : <PST> \\\"*sent 0*\\\" means in one word:\\\"\",\n                            \"In this task, you're examining a news article. Your task is to extract the most critical fact from the article. For this task, this sentence : <PST> \\\"*sent 0*\\\" encapsulates the key fact in one word:\\\"\",\n                            \"In this task, you're reviewing a scientific abstract. Your task is to identify the main entities (e.g., proteins, diseases) and their relations (e.g., causes, treats). For this task, this sentence : <PST> \\\"*sent 0*\\\" highlights the primary entity or relation in one word:\\\"\",\n                            ]\n        else:\n            task_prompts = [\"In this task, you're presented with a text excerpt. Your task is to categorize the excerpt into a broad category such as 'Education', 'Technology', 'Health', 'Business', 'Environment', 'Politics', or 'Culture'. These categories help in organizing content for better accessibility and targeting. For this task, this sentence : \\\"*sent 0*\\\" should be classified under one general category in one word:\\\"\",\n                            \"In this task, you're given a statement and you need to determine whether it's presenting an 'Opinion' or a 'Fact'. This distinction is vital for information verification, educational purposes, and content analysis. For this task, this sentence : \\\"*sent 0*\\\" discriminates between opinion and fact in one word:\\\"\",\n                            \"In this task, you're given a review from an online platform. Your task is to generate a rating for the product based on the review on a scale of 1-5, where 1 means 'extremely negative' and 5 means 'extremely positive'. For this task, this sentence : \\\"*sent 0*\\\" reflects the sentiment in one word:\\\"\",\n                            \"In this task, you're reading a personal diary entry. Your task is to identify the predominant emotion expressed, such as joy, sadness, anger, fear, or love. For this task, this sentence : \\\"*sent 0*\\\" conveys the emotion in one word:\\\"\",\n                            \"In this task, you're presented with two sentences. Your task is to assess whether the sentences convey the same meaning. Use 'identical', 'similar', 'different', or 'unrelated' to describe the relationship. To enhance the performance of this task, this sentence : \\\"*sent 0*\\\" means in one word:\\\"\",\n                            \"In this task, you're given a sentence and a phrase. Your task is to determine if the phrase can be a contextual synonym within the given sentence. Options include 'yes', 'no', or 'partially'. To enhance the performance of this task, this sentence : \\\"*sent 0*\\\" means in one word:\\\"\",\n                            \"In this task, you're examining a news article. Your task is to extract the most critical fact from the article. For this task, this sentence : \\\"*sent 0*\\\" encapsulates the key fact in one word:\\\"\",\n                            \"In this task, you're reviewing a scientific abstract. Your task is to identify the main entities (e.g., proteins, diseases) and their relations (e.g., causes, treats). For this task, this sentence : \\\"*sent 0*\\\" highlights the primary entity or relation in one word:\\\"\",\n                            ]\n    elif args.prompt_method == \"prompteol\":\n        if args.use_which_plan == 'tp':\n            task_prompts = ['This sentence : <PST> \\\"*sent 0*\\\" means in one word:\\\"']\n        else:\n            task_prompts = [\"This sentence : \\\"*sent 0*\\\" means in one word:\\\"\"]\n    elif args.prompt_method == \"cot\":\n        if args.use_which_plan == 'tp':\n            task_prompts = ['After thinking step by step , this sentence : <PST> \\\"*sent 0*\\\" means in one word:\\\"']\n        else:\n            task_prompts = ['After thinking step by step , this sentence : \\\"*sent 0*\\\" means in one word:\\\"']\n    elif args.prompt_method == \"ke\":\n        if args.use_which_plan == 'tp':\n            task_prompts = ['The essence of a sentence is often captured by its main subjects and actions, while descriptive terms provide additional but less central details. With this in mind , this sentence : <PST> \\\"*sent 0*\\\" means in one word:\\\"']\n        else:    \n            task_prompts = ['The essence of a sentence is often captured by its main subjects and actions, while descriptive terms provide additional but less central details. With this in mind , this sentence : \\\"*sent 0*\\\" means in one word:\\\"']\n\n    print(task_prompts)\n\n    def encode_sentences_for_model(sentences, max_length=None, use_vi_tokenizer=False):\n        if max_length == 500:\n            sentences = [tokenizer.decode(tokenizer.encode(s, add_special_tokens=False)[:max_length]) for s in sentences]\n            max_length = 512\n\n        new_sentences = []\n        for i, s in enumerate(sentences):\n            s = preprocess_sentence_for_prompt(s, use_vi_tokenizer=use_vi_tokenizer)\n            for prompt in task_prompts:\n                new_sentences.append(prompt.replace('*sent 0*', s).strip())\n        sentences = new_sentences\n\n        batch = tokenizer.batch_encode_plus(\n            sentences,\n            return_tensors='pt',\n            padding=True,\n            max_length=max_length,\n            truncation=max_length is not None\n        )\n\n        # Move to the correct device\n        for k in batch:\n            batch[k] = batch[k].to(device) if batch[k] is not None else None\n        # Get raw embeddings\n        with torch.no_grad():\n            raw_outputs = model(output_hidden_states=True, return_dict=True, **batch)\n            hidden_states = raw_outputs.hidden_states\n            outputs = hidden_states[args.output_layer][:, -1, :]\n            outputs = outputs.view(-1, len(task_prompts), outputs.size()[1]).mean(dim=1) # Average the embeddings from different tasks \n\n            if outputs.dtype == torch.bfloat16:\n                # bfloat16 not support for .numpy()\n                outputs = outputs.float()\n\n\n            return outputs.cpu()\n\n    def batcher(params, batch, max_length=None):\n        # Handle rare token encoding issues in the dataset\n        if len(batch) >= 1 and len(batch[0]) >= 1 and isinstance(batch[0][0], bytes):\n            batch = [[word.decode('utf-8') for word in s] for s in batch]\n\n        sentences = [' '.join(s) for s in batch]\n        return encode_sentences_for_model(sentences, max_length=max_length, use_vi_tokenizer=False)\n\n    def evaluate_vietnamese_sts():\n        dataset_dict = load_dataset(args.vietnamese_dataset_name, cache_dir=cache_dir)\n        dataset = select_dataset_split(dataset_dict, args.vietnamese_split)\n        sentence1_field, sentence2_field, score_field = resolve_sts_columns(dataset.column_names)\n\n        predictions = []\n        labels = []\n        for start in tqdm.trange(0, len(dataset), args.batch_size, desc=\"Vietnamese STS\"):\n            end = min(start + args.batch_size, len(dataset))\n            batch = dataset[start:end]\n            embeddings_a = encode_sentences_for_model(\n                batch[sentence1_field],\n                use_vi_tokenizer=True,\n            ).numpy()\n            embeddings_b = encode_sentences_for_model(\n                batch[sentence2_field],\n                use_vi_tokenizer=True,\n            ).numpy()\n            predictions.extend(cosine_similarity_scores(embeddings_a, embeddings_b).tolist())\n            labels.extend([float(score) for score in batch[score_field]])\n\n        correlation = spearmanr(labels, predictions)\n        print_table(\n            [\"Dataset\", \"Split\", \"Examples\", \"Spearman\"],\n            [\n                args.vietnamese_dataset_name,\n                args.vietnamese_split,\n                str(len(dataset)),\n                \"%.2f\" % (correlation.correlation * 100),\n            ],\n        )\n        return correlation\n\n    results = {}\n\n    if args.task_set == 'vi-sts':\n        results['VietnameseSTS'] = evaluate_vietnamese_sts()\n        return\n\n    for task in args.tasks:\n        se = senteval.engine.SE(params, batcher, prepare)\n        result = se.eval(task)\n        results[task] = result\n\n    # Print evaluation results\n    if args.mode == 'dev':\n        print(\"------ %s ------\" % (args.mode))\n\n        task_names = []\n        scores = []\n        for task in ['STSBenchmark-dev']:\n            task_names.append(task)\n            if task in results:\n                scores.append(\"%.2f\" % (results[task]['dev']['spearman'][0] * 100))\n            else:\n                scores.append(\"0.00\")\n        print_table(task_names, scores)\n\n        task_names = []\n        scores = []\n        for task in ['MR', 'CR', 'SUBJ', 'MPQA', 'SST2', 'TREC', 'MRPC']:\n            task_names.append(task)\n            if task in results:\n                scores.append(\"%.2f\" % (results[task]['devacc']))    \n            else:\n                scores.append(\"0.00\")\n        task_names.append(\"Avg.\")\n        scores.append(\"%.2f\" % (sum([float(score) for score in scores]) / len(scores)))\n        print_table(task_names, scores)\n\n\n    elif args.mode == 'test' or args.mode == 'fasttest':\n        print(\"------ %s ------\" % (args.mode))\n\n        task_names = []\n        scores = []\n        for task in ['STS12', 'STS13', 'STS14', 'STS15', 'STS16', 'STSBenchmark', 'SICKRelatedness']:\n            task_names.append(task)\n            if task in results:\n                if task in ['STS12', 'STS13', 'STS14', 'STS15', 'STS16']:\n                    scores.append(\"%.2f\" % (results[task]['all']['spearman']['all'] * 100))\n                else:\n                    scores.append(\"%.2f\" % (results[task]['test']['spearman'].correlation * 100))\n            else:\n                scores.append(\"0.00\")\n        task_names.append(\"Avg.\")\n        scores.append(\"%.2f\" % (sum([float(score) for score in scores]) / len(scores)))\n        print_table(task_names, scores)\n        #\n        # write results and template to file\n        if args.task_set != 'transfer':\n            with open('./sts-enhance-results', 'a') as f:\n                model_name = args.model_name_or_path.split('/')[-1]\n                f.write(model_name + ' ' + str(COEFF) + ' ' + str(args.tp_starting_index) + ' ' + ' '.join([str(s) for s in scores]) + '\\n')\n\n        task_names = []\n        scores = []\n        for task in ['MR', 'CR', 'SUBJ', 'MPQA', 'SST2', 'TREC', 'MRPC']:\n            task_names.append(task)\n            if task in results:\n                scores.append(\"%.2f\" % (results[task]['acc']))\n            else:\n                scores.append(\"0.00\")\n        task_names.append(\"Avg.\")\n        scores.append(\"%.2f\" % (sum([float(score) for score in scores]) / len(scores)))\n        print_table(task_names, scores)\n\nif __name__ == \"__main__\":\n    main()\n",
    "model_runtime.py": "import torch\n\n\nDEFAULT_MAC_7B_MODEL = \"Qwen/Qwen2.5-7B\"\nDEFAULT_CACHE_DIR = \".cache/huggingface\"\n\n\ndef resolve_torch_device(device_name: str = \"auto\") -> torch.device:\n    if device_name == \"auto\":\n        if torch.backends.mps.is_available():\n            return torch.device(\"mps\")\n        if torch.cuda.is_available():\n            return torch.device(\"cuda\")\n        return torch.device(\"cpu\")\n\n    device = torch.device(device_name)\n    if device.type == \"mps\" and not torch.backends.mps.is_available():\n        raise RuntimeError(\"MPS was requested, but torch.backends.mps.is_available() is False.\")\n    if device.type == \"cuda\" and not torch.cuda.is_available():\n        raise RuntimeError(\"CUDA was requested, but torch.cuda.is_available() is False.\")\n    return device\n\n\ndef build_loading_kwargs(\n    device_type: str,\n    cache_dir=None,\n    output_hidden_states: bool = True,\n    trust_remote_code: bool = True,\n    torch_dtype=\"auto\",\n):\n    kwargs = {\n        \"output_hidden_states\": output_hidden_states,\n        \"trust_remote_code\": trust_remote_code,\n        \"low_cpu_mem_usage\": True,\n    }\n    if cache_dir:\n        kwargs[\"cache_dir\"] = cache_dir\n    if torch_dtype:\n        kwargs[\"torch_dtype\"] = torch_dtype\n\n    if device_type == \"mps\":\n        kwargs[\"device_map\"] = \"mps\"\n    elif device_type == \"cuda\":\n        kwargs[\"device_map\"] = \"auto\"\n\n    return kwargs\n\n\ndef move_model_to_device_if_needed(model, device: torch.device):\n    if device.type == \"cuda\":\n        return model\n    if hasattr(model, \"hf_device_map\"):\n        return model\n    return model.to(device)\n",
    "vietnamese_sts.py": "from typing import Iterable, Optional, Tuple\n\nimport numpy as np\n\n\nDEFAULT_VIETNAMESE_STS_DATASET = \"nemixo/stsbenchmark-sts-vietnamese\"\nDEFAULT_VIETNAMESE_STS_SPLIT = \"test\"\n\n\ndef preprocess_sentence_for_prompt(text, use_vi_tokenizer: bool = True, vi_tokenizer=None) -> str:\n    sentence = \"\" if text is None else str(text)\n    if use_vi_tokenizer:\n        if vi_tokenizer is None:\n            from pyvi import ViTokenizer\n\n            vi_tokenizer = ViTokenizer\n        sentence = vi_tokenizer.tokenize(sentence)\n\n    if sentence and sentence[-1] not in \".?\\\"'\":\n        sentence += \".\"\n    sentence = sentence.replace('\"', \"'\")\n    if sentence and sentence[-1] == \"?\":\n        sentence = sentence[:-1] + \".\"\n    return sentence\n\n\ndef cosine_similarity_scores(embeddings_a, embeddings_b) -> np.ndarray:\n    a = np.asarray(embeddings_a, dtype=np.float64)\n    b = np.asarray(embeddings_b, dtype=np.float64)\n    numerator = np.sum(a * b, axis=1)\n    denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)\n    return numerator / np.clip(denominator, 1e-12, None)\n\n\ndef resolve_sts_columns(column_names: Iterable[str]) -> Tuple[str, str, str]:\n    names = set(column_names)\n    candidates = [\n        (\"sentence1\", \"sentence2\", \"score\"),\n        (\"sentence1\", \"sentence2\", \"label\"),\n        (\"sent1\", \"sent2\", \"score\"),\n        (\"text1\", \"text2\", \"score\"),\n    ]\n    for sentence1, sentence2, score in candidates:\n        if {sentence1, sentence2, score}.issubset(names):\n            return sentence1, sentence2, score\n    raise ValueError(\n        \"Cannot infer STS columns. Expected one of: \"\n        \"sentence1/sentence2/score, sentence1/sentence2/label, \"\n        \"sent1/sent2/score, text1/text2/score.\"\n    )\n\n\ndef select_dataset_split(dataset, split_name: Optional[str]):\n    if not hasattr(dataset, \"keys\"):\n        return dataset\n\n    if split_name in dataset:\n        return dataset[split_name]\n\n    available_splits = list(dataset.keys())\n    preferred_splits = (\"test\", \"validation\", \"dev\", \"train\")\n    for fallback in preferred_splits:\n        if fallback in dataset:\n            print(\n                f\"warning: split '{split_name}' not found; using '{fallback}'. \"\n                f\"available splits: {available_splits}\"\n            )\n            return dataset[fallback]\n\n    raise ValueError(\n        f\"split '{split_name}' not found. available splits: {available_splits}\"\n    )\n",
}

for rel_path, content in FILES.items():
    path = PROJECT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

print(f"Wrote {len(FILES)} files to {PROJECT_DIR}")


In [ ]:
# Write custom senllm model files
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/token_prepending")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "senllm/__init__.py": "from .modeling_llama import LlamaForCausalLM\nfrom .modeling_qwen2 import Qwen2ForCausalLM\nfrom .modeling_gemma2 import Gemma2ForCausalLM",
    "senllm/utils.py": "import torch\n\n\ndef scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None) -> torch.Tensor:\n    L, S = query.size(-2), key.size(-2)\n    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale\n    attn_bias = torch.zeros(L, S, dtype=query.dtype)\n    if is_causal:\n        assert attn_mask is None\n        temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)\n        attn_bias.masked_fill_(temp_mask.logical_not(), float(\"-inf\"))\n        attn_bias.to(query.dtype)\n\n    if attn_mask is not None:\n        if attn_mask.dtype == torch.bool:\n            attn_bias.masked_fill_(attn_mask.logical_not(), float(\"-inf\"))\n        else:\n            attn_bias += attn_mask\n    attn_weight = query @ key.transpose(-2, -1) * scale_factor\n    attn_weight += attn_bias\n    attn_weight = torch.softmax(attn_weight, dim=-1)\n    attn_weight = torch.dropout(attn_weight, dropout_p, train=True)\n    return attn_weight @ value\n\nclass LlamaSdpaAttention(LlamaAttention):\n    \"\"\"\n    Llama attention module using torch.nn.functional.scaled_dot_product_attention. This module inherits from\n    `LlamaAttention` as the weights of the module stays untouched. The only changes are on the forward pass to adapt to\n    SDPA API.\n    \"\"\"\n\n    # Adapted from LlamaAttention.forward\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: bool = False,\n        use_cache: bool = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        sentence_embedding: Optional[torch.Tensor] = None,\n        **kwargs,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:\n        if output_attentions:\n            # TODO: Improve this warning with e.g. `model.config.attn_implementation = \"manual\"` once this is implemented.\n            logger.warning_once(\n                \"LlamaModel is using LlamaSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, \"\n                'but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation=\"eager\"` when loading the model.'\n            )\n            return super().forward(\n                hidden_states=hidden_states,\n                attention_mask=attention_mask,\n                position_ids=position_ids,\n                past_key_value=past_key_value,\n                output_attentions=output_attentions,\n                use_cache=use_cache,\n                cache_position=cache_position,\n            )\n\n        bsz, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        se_value_states = self.v_proj(sentence_embedding)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        se_value_states = se_value_states.view(bsz, 1, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        cos, sin = self.rotary_emb(value_states, position_ids)\n        # cos_ = torch.ones_like(cos)\n        # sin_ = torch.ones_like(sin)\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n        # print(\"we don't use pos emb!\")\n        if past_key_value is not None:\n            # sin and cos are specific to RoPE models; cache_position needed for the static cache\n            cache_kwargs = {\"sin\": sin, \"cos\": cos, \"cache_position\": cache_position}\n            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)\n\n        key_states = repeat_kv(key_states, self.num_key_value_groups)\n        value_states = repeat_kv(value_states, self.num_key_value_groups)\n\n        causal_mask = attention_mask\n        if attention_mask is not None:\n            causal_mask = causal_mask[:, :, :, : key_states.shape[-2]]\n\n        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,\n        # Reference: https://github.com/pytorch/pytorch/issues/112577.\n        if query_states.device.type == \"cuda\" and causal_mask is not None:\n            query_states = query_states.contiguous()\n            key_states = key_states.contiguous()\n            value_states = value_states.contiguous()\n\n        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment\n        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.\n        is_causal = True if causal_mask is None and q_len > 1 else False\n\n        attn_output = torch.nn.functional.scaled_dot_product_attention(\n            query_states,\n            key_states,\n            value_states,\n            attn_mask=causal_mask,\n            dropout_p=self.attention_dropout if self.training else 0.0,\n            is_causal=is_causal,\n        )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.view(bsz, q_len, -1)\n\n        attn_output = self.o_proj(attn_output)\n\n        return attn_output, None, past_key_value",
    "senllm/modeling_qwen2.py": "import os\nimport math\nfrom typing import List, Optional, Tuple, Union\n\nimport torch\nimport torch.utils.checkpoint\nfrom torch import nn\nfrom torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss\n\nfrom transformers.models.qwen2.modeling_qwen2 import *\nfrom transformers.activations import ACT2FN\nfrom transformers.cache_utils import Cache, DynamicCache, SlidingWindowCache, StaticCache\nfrom transformers.generation import GenerationMixin\nfrom transformers.modeling_attn_mask_utils import AttentionMaskConverter\nfrom transformers.modeling_outputs import (\n    BaseModelOutputWithPast,\n    CausalLMOutputWithPast,\n    QuestionAnsweringModelOutput,\n    SequenceClassifierOutputWithPast,\n    TokenClassifierOutput,\n)\nfrom transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.utils import (\n    add_code_sample_docstrings,\n    add_start_docstrings,\n    add_start_docstrings_to_model_forward,\n    is_flash_attn_2_available,\n    is_flash_attn_greater_or_equal_2_10,\n    logging,\n    replace_return_docstrings,\n)\nfrom transformers.models.qwen2.configuration_qwen2 import Qwen2Config\n\n\nlogger = logging.get_logger(__name__)\n\n\n_CHECKPOINT_FOR_DOC = \"Qwen/Qwen2-7B\"\n_CONFIG_FOR_DOC = \"Qwen2Config\"\n\nCAL_IDX = int(os.environ.get(\"CAL_IDX\", 0))\nBETA = float(os.environ.get(\"BETA\", 0))\nCOEFF = float(os.environ.get(\"COEFF\", 1.0))\n\n\ndef find_token_indices(input_ids, token=32000):\n    # \u65ad\u8a00 input_ids \u4e2d\u6240\u6709\u5e8f\u5217\u90fd\u5305\u542b\u5143\u7d20 32000\n    assert (input_ids == token).any(dim=1).all(), f\"Not all sequences contain the token {token}\"\n    \n    # \u83b7\u53d6\u7b2c\u4e00\u4e2a\u5339\u914d 32000 \u7684\u7d22\u5f15\n    mask = (input_ids == token)\n    # \u8f6c\u6362\u4e3a\u6d6e\u70b9\u578b\u4ee5\u4fbf\u4f7f\u7528argmax\n    mask_float = mask.float()\n    # \u8ba1\u7b97\u7b2c\u4e00\u4e2a\u5339\u914d\u7684\u7d22\u5f15\n    first_match_indices = mask_float.argmax(dim=1)\n    \n    return first_match_indices\n\n\n\nQWEN2_ATTENTION_CLASSES = {\n    \"eager\": Qwen2Attention,\n    \"flash_attention_2\": Qwen2FlashAttention2,\n    \"sdpa\": Qwen2SdpaAttention,\n}\n\n\nclass Qwen2DecoderLayer(nn.Module):\n    def __init__(self, config: Qwen2Config, layer_idx: int):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n\n        if config.sliding_window and config._attn_implementation != \"flash_attention_2\":\n            logger.warning_once(\n                f\"Sliding Window Attention is enabled but not implemented for `{config._attn_implementation}`; \"\n                \"unexpected results may be encountered.\"\n            )\n        self.self_attn = QWEN2_ATTENTION_CLASSES[config._attn_implementation](config, layer_idx)\n\n        self.mlp = Qwen2MLP(config)\n        self.input_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Tuple[torch.Tensor]] = None,\n        output_attentions: Optional[bool] = False,\n        use_cache: Optional[bool] = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n        **kwargs,\n    ) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]:\n        \"\"\"\n        Args:\n            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`\n            attention_mask (`torch.FloatTensor`, *optional*): attention mask of size\n                `(batch, sequence_length)` where padding elements are indicated by 0.\n            output_attentions (`bool`, *optional*):\n                Whether or not to return the attentions tensors of all attention layers. See `attentions` under\n                returned tensors for more detail.\n            use_cache (`bool`, *optional*):\n                If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding\n                (see `past_key_values`).\n            past_key_value (`Tuple(torch.FloatTensor)`, *optional*): cached past key and value projection states\n            cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):\n                Indices depicting the position of the input sequence tokens in the sequence.\n            position_embeddings (`Tuple[torch.FloatTensor, torch.FloatTensor]`, *optional*):\n                Tuple containing the cosine and sine positional embeddings of shape `(batch_size, seq_len, head_dim)`,\n                with `head_dim` being the embedding dimension of each attention head.\n            kwargs (`dict`, *optional*):\n                Arbitrary kwargs to be ignored, used for FSDP and other methods that injects code\n                into the model\n        \"\"\"\n\n        residual = hidden_states\n\n        hidden_states = self.input_layernorm(hidden_states)\n\n        # Self Attention\n        hidden_states, self_attn_weights, present_key_value = self.self_attn(\n            hidden_states=hidden_states,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_value=past_key_value,\n            output_attentions=output_attentions,\n            use_cache=use_cache,\n            cache_position=cache_position,\n            position_embeddings=position_embeddings,\n        )\n        hidden_states = residual + hidden_states\n\n        # Fully Connected\n        residual = hidden_states\n        hidden_states = self.post_attention_layernorm(hidden_states)\n        hidden_states = self.mlp(hidden_states)\n        hidden_states = residual + hidden_states\n\n        outputs = (hidden_states,)\n\n        if output_attentions:\n            outputs += (self_attn_weights,)\n\n        if use_cache:\n            outputs += (present_key_value,)\n\n        return outputs\n\n\nQWEN2_START_DOCSTRING = r\"\"\"\n    This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the\n    library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads\n    etc.)\n\n    This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.\n    Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage\n    and behavior.\n\n    Parameters:\n        config ([`Qwen2Config`]):\n            Model configuration class with all the parameters of the model. Initializing with a config file does not\n            load the weights associated with the model, only the configuration. Check out the\n            [`~PreTrainedModel.from_pretrained`] method to load the model weights.\n\"\"\"\n\n\n@add_start_docstrings(\n    \"The bare Qwen2 Model outputting raw hidden-states without any specific head on top.\",\n    QWEN2_START_DOCSTRING,\n)\nclass Qwen2PreTrainedModel(PreTrainedModel):\n    config_class = Qwen2Config\n    base_model_prefix = \"model\"\n    supports_gradient_checkpointing = True\n    _no_split_modules = [\"Qwen2DecoderLayer\"]\n    _skip_keys_device_placement = \"past_key_values\"\n    _supports_flash_attn_2 = True\n    _supports_sdpa = True\n    _supports_cache_class = True\n    _supports_quantized_cache = True\n    _supports_static_cache = True\n\n    def _init_weights(self, module):\n        std = self.config.initializer_range\n        if isinstance(module, nn.Linear):\n            module.weight.data.normal_(mean=0.0, std=std)\n            if module.bias is not None:\n                module.bias.data.zero_()\n        elif isinstance(module, nn.Embedding):\n            module.weight.data.normal_(mean=0.0, std=std)\n            if module.padding_idx is not None:\n                module.weight.data[module.padding_idx].zero_()\n\n\nQWEN2_INPUTS_DOCSTRING = r\"\"\"\n    Args:\n        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):\n            Indices of input sequence tokens in the vocabulary. Padding will be ignored by default should you provide\n            it.\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            [What are input IDs?](../glossary#input-ids)\n        attention_mask (`torch.Tensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Mask to avoid performing attention on padding token indices. Mask values selected in `[0, 1]`:\n\n            - 1 for tokens that are **not masked**,\n            - 0 for tokens that are **masked**.\n\n            [What are attention masks?](../glossary#attention-mask)\n\n            Indices can be obtained using [`AutoTokenizer`]. See [`PreTrainedTokenizer.encode`] and\n            [`PreTrainedTokenizer.__call__`] for details.\n\n            If `past_key_values` is used, optionally only the last `decoder_input_ids` have to be input (see\n            `past_key_values`).\n\n            If you want to change padding behavior, you should read [`modeling_opt._prepare_decoder_attention_mask`]\n            and modify to your needs. See diagram 1 in [the paper](https://arxiv.org/abs/1910.13461) for more\n            information on the default strategy.\n\n            - 1 indicates the head is **not masked**,\n            - 0 indicates the head is **masked**.\n        position_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n            Indices of positions of each input sequence tokens in the position embeddings. Selected in the range `[0,\n            config.n_positions - 1]`.\n\n            [What are position IDs?](../glossary#position-ids)\n        past_key_values (`Cache` or `tuple(tuple(torch.FloatTensor))`, *optional*):\n            Pre-computed hidden-states (key and values in the self-attention blocks and in the cross-attention\n            blocks) that can be used to speed up sequential decoding. This typically consists in the `past_key_values`\n            returned by the model at a previous stage of decoding, when `use_cache=True` or `config.use_cache=True`.\n\n            Two formats are allowed:\n            - a [`~cache_utils.Cache`] instance, see our\n            [kv cache guide](https://huggingface.co/docs/transformers/en/kv_cache);\n            - Tuple of `tuple(torch.FloatTensor)` of length `config.n_layers`, with each tuple having 2 tensors of\n            shape `(batch_size, num_heads, sequence_length, embed_size_per_head)`). This is also known as the legacy\n            cache format.\n\n            The model will output the same cache format that is fed as input. If no `past_key_values` are passed, the\n            legacy cache format will be returned.\n\n            If `past_key_values` are used, the user can optionally input only the last `input_ids` (those that don't\n            have their past key value states given to this model) of shape `(batch_size, 1)` instead of all `input_ids`\n            of shape `(batch_size, sequence_length)`.\n        inputs_embeds (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`, *optional*):\n            Optionally, instead of passing `input_ids` you can choose to directly pass an embedded representation. This\n            is useful if you want more control over how to convert `input_ids` indices into associated vectors than the\n            model's internal embedding lookup matrix.\n        use_cache (`bool`, *optional*):\n            If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding (see\n            `past_key_values`).\n        output_attentions (`bool`, *optional*):\n            Whether or not to return the attentions tensors of all attention layers. See `attentions` under returned\n            tensors for more detail.\n        output_hidden_states (`bool`, *optional*):\n            Whether or not to return the hidden states of all layers. See `hidden_states` under returned tensors for\n            more detail.\n        return_dict (`bool`, *optional*):\n            Whether or not to return a [`~utils.ModelOutput`] instead of a plain tuple.\n        cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):\n            Indices depicting the position of the input sequence tokens in the sequence. Contrarily to `position_ids`,\n            this tensor is not affected by padding. It is used to update the cache in the correct position and to infer\n            the complete sequence length.\n\"\"\"\n\n\n@add_start_docstrings(\n    \"The bare Qwen2 Model outputting raw hidden-states without any specific head on top.\",\n    QWEN2_START_DOCSTRING,\n)\nclass Qwen2Model(Qwen2PreTrainedModel):\n    \"\"\"\n    Transformer decoder consisting of *config.num_hidden_layers* layers. Each layer is a [`Qwen2DecoderLayer`]\n\n    Args:\n        config: Qwen2Config\n    \"\"\"\n\n    def __init__(self, config: Qwen2Config):\n        super().__init__(config)\n        self.padding_idx = config.pad_token_id\n        self.vocab_size = config.vocab_size\n\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)\n        self.layers = nn.ModuleList(\n            [Qwen2DecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]\n        )\n        self._attn_implementation = config._attn_implementation\n        self.norm = Qwen2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.rotary_emb = Qwen2RotaryEmbedding(config=config)\n\n        self.gradient_checkpointing = False\n        # Initialize weights and apply final processing\n        self.post_init()\n\n        self.plan = 'vanilla'\n        self.tp_starting_index = 1\n        self.tp_exiting_index = 99\n        self.placeholder_token_id = config.vocab_size - 1\n\n    def get_input_embeddings(self):\n        return self.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.embed_tokens = value\n\n    @add_start_docstrings_to_model_forward(QWEN2_INPUTS_DOCSTRING)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[List[torch.FloatTensor]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPast]:\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        use_cache = use_cache if use_cache is not None else self.config.use_cache\n\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        if (input_ids is None) ^ (inputs_embeds is not None):\n            raise ValueError(\"You must specify exactly one of input_ids or inputs_embeds\")\n\n        if self.gradient_checkpointing and self.training:\n            if use_cache:\n                logger.warning_once(\n                    \"`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...\"\n                )\n                use_cache = False\n\n        # kept for BC (non `Cache` `past_key_values` inputs)\n        return_legacy_cache = False\n        if use_cache and not isinstance(past_key_values, Cache):\n            return_legacy_cache = True\n            if past_key_values is None:\n                past_key_values = DynamicCache()\n            else:\n                past_key_values = DynamicCache.from_legacy_cache(past_key_values)\n                logger.warning_once(\n                    \"We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and \"\n                    \"will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class \"\n                    \"(https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)\"\n                )\n        \n\n        if self.plan == 'tp':\n            pst_token_indices = find_token_indices(input_ids, token=self.placeholder_token_id)\n\n        if inputs_embeds is None:\n            inputs_embeds = self.embed_tokens(input_ids)\n\n        if cache_position is None:\n            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0\n            cache_position = torch.arange(\n                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device\n            )\n        if position_ids is None:\n            position_ids = cache_position.unsqueeze(0)\n\n        causal_mask = self._update_causal_mask(\n            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions\n        )\n\n        hidden_states = inputs_embeds\n\n        # create position embeddings to be shared across the decoder layers\n        position_embeddings = self.rotary_emb(hidden_states, position_ids)\n\n        # decoder layers\n        all_hidden_states = () if output_hidden_states else None\n        all_self_attns = () if output_attentions else None\n        next_decoder_cache = None\n\n        for index, decoder_layer in enumerate(self.layers):\n            if output_hidden_states:\n                all_hidden_states += (hidden_states,)\n\n            if self.gradient_checkpointing and self.training:\n                layer_outputs = self._gradient_checkpointing_func(\n                    decoder_layer.__call__,\n                    hidden_states,\n                    causal_mask,\n                    position_ids,\n                    past_key_values,\n                    output_attentions,\n                    use_cache,\n                    cache_position,\n                    position_embeddings,\n                )\n            else:\n                if self.plan == \"vanilla\":\n                    layer_outputs = decoder_layer(\n                        hidden_states,\n                        attention_mask=causal_mask,\n                        position_ids=position_ids,\n                        past_key_value=past_key_values,\n                        output_attentions=output_attentions,\n                        use_cache=use_cache,\n                        cache_position=cache_position,\n                        position_embeddings=position_embeddings,\n                    )\n                elif self.plan == \"tp\":\n                    layer_index = self.tp_starting_index\n                    exiting_index = self.tp_exiting_index\n                    if index < layer_index:\n                        layer_outputs = decoder_layer(\n                                            hidden_states,\n                                            attention_mask=causal_mask,\n                                            position_ids=position_ids,\n                                            past_key_value=past_key_values,\n                                            output_attentions=output_attentions,\n                                            use_cache=use_cache,\n                                            cache_position=cache_position,\n                                            position_embeddings=position_embeddings,\n                                        )\n                    elif index >= layer_index and index < exiting_index:\n                        B = hidden_states.shape[0]\n                        previous_sentence_embeddings = hidden_states[:, -1, :].clone()\n                        hidden_states[torch.arange(B), pst_token_indices, :] = previous_sentence_embeddings\n                        layer_outputs = decoder_layer(\n                                            hidden_states,\n                                            attention_mask=causal_mask,\n                                            position_ids=position_ids,\n                                            past_key_value=past_key_values,\n                                            output_attentions=output_attentions,\n                                            use_cache=use_cache,\n                                            cache_position=cache_position,\n                                            position_embeddings=position_embeddings,\n                                        )\n                    elif index >= exiting_index:\n                        layer_outputs = decoder_layer(\n                                            hidden_states,\n                                            attention_mask=causal_mask,\n                                            position_ids=position_ids,\n                                            past_key_value=past_key_values,\n                                            output_attentions=output_attentions,\n                                            use_cache=use_cache,\n                                            cache_position=cache_position,\n                                            position_embeddings=position_embeddings,\n                                        )\n                    else:\n                        raise ValueError(\"layer index not right!\")\n                    \n                else:\n                    raise ValueError(f\"The {self.plan} plan have not yet been implemented!\")\n\n            hidden_states = layer_outputs[0]\n\n            if use_cache:\n                next_decoder_cache = layer_outputs[2 if output_attentions else 1]\n\n            if output_attentions:\n                all_self_attns += (layer_outputs[1],)\n\n        hidden_states = self.norm(hidden_states)\n\n        # add hidden states from the last decoder layer\n        if output_hidden_states:\n            all_hidden_states += (hidden_states,)\n\n        next_cache = next_decoder_cache if use_cache else None\n        if return_legacy_cache:\n            next_cache = next_cache.to_legacy_cache()\n\n        if not return_dict:\n            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)\n        return BaseModelOutputWithPast(\n            last_hidden_state=hidden_states,\n            past_key_values=next_cache,\n            hidden_states=all_hidden_states,\n            attentions=all_self_attns,\n        )\n\n    # Copied from transformers.models.phi3.modeling_phi3.Phi3Model._update_causal_mask\n    def _update_causal_mask(\n        self,\n        attention_mask: torch.Tensor,\n        input_tensor: torch.Tensor,\n        cache_position: torch.Tensor,\n        past_key_values: Cache,\n        output_attentions: bool,\n    ):\n        if self.config._attn_implementation == \"flash_attention_2\":\n            if attention_mask is not None and 0.0 in attention_mask:\n                return attention_mask\n            return None\n\n        # For SDPA, when possible, we will rely on its `is_causal` argument instead of its `attn_mask` argument, in\n        # order to dispatch on Flash Attention 2. This feature is not compatible with static cache, as SDPA will fail\n        # to infer the attention mask.\n        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0\n        using_static_cache = isinstance(past_key_values, StaticCache)\n        using_sliding_window_cache = isinstance(past_key_values, SlidingWindowCache)\n\n        # When output attentions is True, sdpa implementation's forward method calls the eager implementation's forward\n        if (\n            self.config._attn_implementation == \"sdpa\"\n            and not (using_static_cache or using_sliding_window_cache)\n            and not output_attentions\n        ):\n            if AttentionMaskConverter._ignore_causal_mask_sdpa(\n                attention_mask,\n                inputs_embeds=input_tensor,\n                past_key_values_length=past_seen_tokens,\n                sliding_window=self.config.sliding_window,\n                is_training=self.training,\n            ):\n                return None\n\n        dtype, device = input_tensor.dtype, input_tensor.device\n        min_dtype = torch.finfo(dtype).min\n        sequence_length = input_tensor.shape[1]\n        # SlidingWindowCache or StaticCache\n        if using_sliding_window_cache or using_static_cache:\n            target_length = past_key_values.get_max_cache_shape()\n        # DynamicCache or no cache\n        else:\n            target_length = (\n                attention_mask.shape[-1]\n                if isinstance(attention_mask, torch.Tensor)\n                else past_seen_tokens + sequence_length + 1\n            )\n\n        # In case the provided `attention` mask is 2D, we generate a causal mask here (4D).\n        causal_mask = self._prepare_4d_causal_attention_mask_with_cache_position(\n            attention_mask,\n            sequence_length=sequence_length,\n            target_length=target_length,\n            dtype=dtype,\n            device=device,\n            cache_position=cache_position,\n            batch_size=input_tensor.shape[0],\n            config=self.config,\n            past_key_values=past_key_values,\n        )\n\n        if (\n            self.config._attn_implementation == \"sdpa\"\n            and attention_mask is not None\n            and attention_mask.device.type == \"cuda\"\n            and not output_attentions\n        ):\n            # Attend to all tokens in fully masked rows in the causal_mask, for example the relevant first rows when\n            # using left padding. This is required by F.scaled_dot_product_attention memory-efficient attention path.\n            # Details: https://github.com/pytorch/pytorch/issues/110213\n            causal_mask = AttentionMaskConverter._unmask_unattended(causal_mask, min_dtype)\n\n        return causal_mask\n\n    @staticmethod\n    # Copied from transformers.models.mistral.modeling_mistral.MistralModel._prepare_4d_causal_attention_mask_with_cache_position with Mistral->Qwen2\n    def _prepare_4d_causal_attention_mask_with_cache_position(\n        attention_mask: torch.Tensor,\n        sequence_length: int,\n        target_length: int,\n        dtype: torch.dtype,\n        device: torch.device,\n        cache_position: torch.Tensor,\n        batch_size: int,\n        config: Qwen2Config,\n        past_key_values: Cache,\n    ):\n        \"\"\"\n        Creates a causal 4D mask of shape `(batch_size, 1, query_length, key_value_length)` from a 2D mask of shape\n        `(batch_size, key_value_length)`, or if the input `attention_mask` is already 4D, do nothing.\n\n        Args:\n            attention_mask (`torch.Tensor`):\n                A 2D attention mask of shape `(batch_size, key_value_length)` or a 4D attention mask of shape `(batch_size, 1, query_length, key_value_length)`.\n            sequence_length (`int`):\n                The sequence length being processed.\n            target_length (`int`):\n                The target length: when generating with static cache, the mask should be as long as the static cache, to account for the 0 padding, the part of the cache that is not filled yet.\n            dtype (`torch.dtype`):\n                The dtype to use for the 4D attention mask.\n            device (`torch.device`):\n                The device to plcae the 4D attention mask on.\n            cache_position (`torch.Tensor`):\n                Indices depicting the position of the input sequence tokens in the sequence.\n            batch_size (`torch.Tensor`):\n                Batch size.\n            config (`Qwen2Config`):\n                The model's configuration class\n            past_key_values (`Cache`):\n                The cache class that is being used currently to generate\n        \"\"\"\n        if attention_mask is not None and attention_mask.dim() == 4:\n            # In this case we assume that the mask comes already in inverted form and requires no inversion or slicing.\n            causal_mask = attention_mask\n        else:\n            min_dtype = torch.finfo(dtype).min\n            causal_mask = torch.full(\n                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=device\n            )\n            diagonal_attend_mask = torch.arange(target_length, device=device) > cache_position.reshape(-1, 1)\n            if config.sliding_window is not None:\n                # if we have sliding window, we should not attend to tokens beyond sliding window length, so we mask them out also\n                # the check is needed to verify is current checkpoint was trained with sliding window or not\n                if not isinstance(past_key_values, SlidingWindowCache) or sequence_length > target_length:\n                    sliding_attend_mask = torch.arange(target_length, device=device) <= (\n                        cache_position.reshape(-1, 1) - config.sliding_window\n                    )\n                    diagonal_attend_mask.bitwise_or_(sliding_attend_mask)\n            causal_mask *= diagonal_attend_mask\n            causal_mask = causal_mask[None, None, :, :].expand(batch_size, 1, -1, -1)\n            if attention_mask is not None:\n                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit\n                if attention_mask.shape[-1] > target_length:\n                    attention_mask = attention_mask[:, :target_length]\n                mask_length = attention_mask.shape[-1]\n                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :]\n                padding_mask = padding_mask == 0\n                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(\n                    padding_mask, min_dtype\n                )\n        return causal_mask\n\n\nclass Qwen2ForCausalLM(Qwen2PreTrainedModel, GenerationMixin):\n    _tied_weights_keys = [\"lm_head.weight\"]\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.model = Qwen2Model(config)\n        self.vocab_size = config.vocab_size\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def set_decoder(self, decoder):\n        self.model = decoder\n\n    def get_decoder(self):\n        return self.model\n\n    @add_start_docstrings_to_model_forward(QWEN2_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=CausalLMOutputWithPast, config_class=_CONFIG_FOR_DOC)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[List[torch.FloatTensor]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        labels: Optional[torch.LongTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n        num_logits_to_keep: int = 0,\n        **loss_kwargs,\n    ) -> Union[Tuple, CausalLMOutputWithPast]:\n        r\"\"\"\n        Args:\n            labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n                Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,\n                config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored\n                (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.\n\n            num_logits_to_keep (`int`, *optional*):\n                Calculate logits for the last `num_logits_to_keep` tokens. If `0`, calculate logits for all\n                `input_ids` (special case). Only last token logits are needed for generation, and calculating them only for that\n                token can save memory, which becomes pretty significant for long sequences or large vocabulary size.\n\n        Returns:\n\n        Example:\n\n        ```python\n        >>> from transformers import AutoTokenizer, Qwen2ForCausalLM\n\n        >>> model = Qwen2ForCausalLM.from_pretrained(PATH_TO_CONVERTED_WEIGHTS)\n        >>> tokenizer = AutoTokenizer.from_pretrained(PATH_TO_CONVERTED_TOKENIZER)\n\n        >>> prompt = \"Hey, are you conscious? Can you talk to me?\"\n        >>> inputs = tokenizer(prompt, return_tensors=\"pt\")\n\n        >>> # Generate\n        >>> generate_ids = model.generate(inputs.input_ids, max_length=30)\n        >>> tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]\n        \"Hey, are you conscious? Can you talk to me?\\nI'm not conscious, but I can talk to you.\"\n        ```\"\"\"\n\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        # decoder outputs consists of (dec_features, layer_state, dec_hidden, dec_attn)\n        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_values=past_key_values,\n            inputs_embeds=inputs_embeds,\n            use_cache=use_cache,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            cache_position=cache_position,\n        )\n\n        hidden_states = outputs[0]\n        # Only compute necessary logits, and do not upcast them to float if we are not computing the loss\n        logits = self.lm_head(hidden_states[:, -num_logits_to_keep:, :])\n\n        loss = None\n        if labels is not None:\n            loss = self.loss_function(logits, labels, self.vocab_size, **loss_kwargs)\n\n        if not return_dict:\n            output = (logits,) + outputs[1:]\n            return (loss,) + output if loss is not None else output\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=outputs.past_key_values,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n",
    "senllm/modeling_llama.py": "import os\nimport logging\nimport math\nimport copy\nfrom typing import List, Optional, Tuple, Union\n\nimport torch\nimport torch.nn.functional as F\nimport torch.utils.checkpoint\nfrom torch import nn\nfrom torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss\n\nfrom transformers.models.llama.modeling_llama import *\nfrom transformers.activations import ACT2FN\nfrom transformers.cache_utils import Cache, DynamicCache, StaticCache\nfrom transformers.modeling_attn_mask_utils import AttentionMaskConverter\n\nfrom transformers.modeling_outputs import (\n    BaseModelOutputWithPast,\n    CausalLMOutputWithPast,\n    QuestionAnsweringModelOutput,\n    SequenceClassifierOutputWithPast,\n    TokenClassifierOutput,\n)\n\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.pytorch_utils import ALL_LAYERNORM_LAYERS\nfrom transformers.utils import (\n    add_start_docstrings,\n    add_start_docstrings_to_model_forward,\n    logging,\n    replace_return_docstrings,\n)\nfrom transformers.models.llama.configuration_llama import LlamaConfig\n\n\n\n\nCAL_IDX = int(os.environ.get(\"CAL_IDX\", 0))\nBETA = float(os.environ.get(\"BETA\", 0))\nCOEFF = float(os.environ.get(\"COEFF\", 1.0))\n\nlogger = logging.get_logger(__name__)\n\n_CONFIG_FOR_DOC = \"LlamaConfig\"\n\n\n\n\ndef find_token_indices(input_ids, token=32000):\n    # \u65ad\u8a00 input_ids \u4e2d\u6240\u6709\u5e8f\u5217\u90fd\u5305\u542b\u5143\u7d20 32000\n    assert (input_ids == token).any(dim=1).all(), f\"Not all sequences contain the token {token}\"\n    \n    mask = (input_ids == token)\n    mask_float = mask.float()\n    first_match_indices = mask_float.argmax(dim=1)\n    \n    return first_match_indices\n\n\ndef scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=0.0,\n        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:\n    L, S = query.size(-2), key.size(-2)\n    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale\n    attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)\n    if is_causal:\n        assert attn_mask is None\n        temp_mask = torch.ones(L, S, dtype=torch.bool, device=query.device).tril(diagonal=0)\n        attn_bias.masked_fill_(temp_mask.logical_not(), float(\"-inf\"))\n        attn_bias.to(query.dtype)\n\n    if attn_mask is not None:\n        if attn_mask.dtype == torch.bool:\n            attn_bias.masked_fill_(attn_mask.logical_not(), float(\"-inf\"))\n        else:\n            attn_bias = attn_mask + attn_bias\n\n    if enable_gqa:\n        key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)\n        value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)\n\n    attn_weight = query @ key.transpose(-2, -1) * scale_factor\n    attn_weight += attn_bias\n    attn_weight = torch.softmax(attn_weight, dim=-1)\n    attn_weight = torch.dropout(attn_weight, dropout_p, train=True)\n    \n    return attn_weight @ value\n\n\n\nclass LlamaAttention(nn.Module):\n    \"\"\"Multi-headed attention from 'Attention Is All You Need' paper\"\"\"\n\n    def __init__(self, config: LlamaConfig, layer_idx: Optional[int] = None):\n        super().__init__()\n        self.config = config\n        self.layer_idx = layer_idx\n        if layer_idx is None:\n            logger.warning_once(\n                f\"Instantiating {self.__class__.__name__} without passing a `layer_idx` is not recommended and will \"\n                \"lead to errors during the forward call if caching is used. Please make sure to provide a `layer_idx` \"\n                \"when creating this class.\"\n            )\n\n        self.attention_dropout = config.attention_dropout\n        self.hidden_size = config.hidden_size\n        self.num_heads = config.num_attention_heads\n        self.head_dim = getattr(config, \"head_dim\", self.hidden_size // self.num_heads)\n        self.num_key_value_heads = config.num_key_value_heads\n        self.num_key_value_groups = self.num_heads // self.num_key_value_heads\n        self.max_position_embeddings = config.max_position_embeddings\n        self.rope_theta = config.rope_theta\n        self.is_causal = True\n\n        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)\n        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)\n        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)\n        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=config.attention_bias)\n\n        # TODO (joao): remove in v4.46 (RoPE is computed in the model, not in the decoder layers)\n        self.rotary_emb = LlamaRotaryEmbedding(config=self.config)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: bool = False,\n        use_cache: bool = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n        **kwargs,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:\n        bsz, q_len, _ = hidden_states.size()\n\n        if self.config.pretraining_tp > 1:\n            key_value_slicing = (self.num_key_value_heads * self.head_dim) // self.config.pretraining_tp\n            query_slices = self.q_proj.weight.split(\n                (self.num_heads * self.head_dim) // self.config.pretraining_tp, dim=0\n            )\n            key_slices = self.k_proj.weight.split(key_value_slicing, dim=0)\n            value_slices = self.v_proj.weight.split(key_value_slicing, dim=0)\n\n            query_states = [F.linear(hidden_states, query_slices[i]) for i in range(self.config.pretraining_tp)]\n            query_states = torch.cat(query_states, dim=-1)\n\n            key_states = [F.linear(hidden_states, key_slices[i]) for i in range(self.config.pretraining_tp)]\n            key_states = torch.cat(key_states, dim=-1)\n\n            value_states = [F.linear(hidden_states, value_slices[i]) for i in range(self.config.pretraining_tp)]\n            value_states = torch.cat(value_states, dim=-1)\n\n        else:\n            query_states = self.q_proj(hidden_states)\n            key_states = self.k_proj(hidden_states)\n            value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        if position_embeddings is None:\n            logger.warning_once(\n                \"The attention layers in this model are transitioning from computing the RoPE embeddings internally \"\n                \"through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed \"\n                \"`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be \"\n                \"removed and `position_embeddings` will be mandatory.\"\n            )\n            cos, sin = self.rotary_emb(value_states, position_ids)\n        else:\n            cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        if past_key_value is not None:\n            # sin and cos are specific to RoPE models; cache_position needed for the static cache\n            cache_kwargs = {\"sin\": sin, \"cos\": cos, \"cache_position\": cache_position}\n            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)\n\n        key_states = repeat_kv(key_states, self.num_key_value_groups)\n        value_states = repeat_kv(value_states, self.num_key_value_groups)\n        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / math.sqrt(self.head_dim)\n\n        if attention_mask is not None:  # no matter the length, we just slice it\n            causal_mask = attention_mask[:, :, :, : key_states.shape[-2]]\n            attn_weights = attn_weights + causal_mask\n\n        # upcast attention to fp32\n        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n        attn_weights = nn.functional.dropout(attn_weights, p=self.attention_dropout, training=self.training)\n        attn_output = torch.matmul(attn_weights, value_states)\n\n        if attn_output.size() != (bsz, self.num_heads, q_len, self.head_dim):\n            raise ValueError(\n                f\"`attn_output` should be of size {(bsz, self.num_heads, q_len, self.head_dim)}, but is\"\n                f\" {attn_output.size()}\"\n            )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n\n        attn_output = attn_output.reshape(bsz, q_len, -1)\n\n        if self.config.pretraining_tp > 1:\n            attn_output = attn_output.split(self.hidden_size // self.config.pretraining_tp, dim=2)\n            o_proj_slices = self.o_proj.weight.split(self.hidden_size // self.config.pretraining_tp, dim=1)\n            attn_output = sum([F.linear(attn_output[i], o_proj_slices[i]) for i in range(self.config.pretraining_tp)])\n        else:\n            attn_output = self.o_proj(attn_output)\n\n        if not output_attentions:\n            attn_weights = None\n\n        return attn_output, attn_weights, past_key_value\n\n\n\nclass LlamaSdpaAttention(LlamaAttention):\n    \"\"\"\n    Llama attention module using torch.nn.functional.scaled_dot_product_attention. This module inherits from\n    `LlamaAttention` as the weights of the module stays untouched. The only changes are on the forward pass to adapt to\n    SDPA API.\n    \"\"\"\n\n    # Adapted from LlamaAttention.forward\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: bool = False,\n        use_cache: bool = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.46\n        **kwargs,\n    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:\n        if output_attentions:\n            # TODO: Improve this warning with e.g. `model.config.attn_implementation = \"manual\"` once this is implemented.\n            logger.warning_once(\n                \"LlamaModel is using LlamaSdpaAttention, but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to the manual attention implementation, \"\n                'but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation=\"eager\"` when loading the model.'\n            )\n            return super().forward(\n                hidden_states=hidden_states,\n                attention_mask=attention_mask,\n                position_ids=position_ids,\n                past_key_value=past_key_value,\n                output_attentions=output_attentions,\n                use_cache=use_cache,\n                cache_position=cache_position,\n                position_embeddings=position_embeddings,\n            )\n\n        bsz, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        if position_embeddings is None:\n            logger.warning_once(\n                \"The attention layers in this model are transitioning from computing the RoPE embeddings internally \"\n                \"through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed \"\n                \"`position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be \"\n                \"removed and `position_embeddings` will be mandatory.\"\n            )\n            cos, sin = self.rotary_emb(value_states, position_ids)\n        else:\n            cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        if past_key_value is not None:\n            # sin and cos are specific to RoPE models; cache_position needed for the static cache\n            cache_kwargs = {\"sin\": sin, \"cos\": cos, \"cache_position\": cache_position}\n            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)\n\n        key_states = repeat_kv(key_states, self.num_key_value_groups)\n        value_states = repeat_kv(value_states, self.num_key_value_groups)\n\n        causal_mask = attention_mask\n        if attention_mask is not None:\n            causal_mask = causal_mask[:, :, :, : key_states.shape[-2]]\n\n        # SDPA with memory-efficient backend is currently (torch==2.1.2) bugged with non-contiguous inputs with custom attn_mask,\n        # Reference: https://github.com/pytorch/pytorch/issues/112577.\n        if query_states.device.type == \"cuda\" and causal_mask is not None:\n            query_states = query_states.contiguous()\n            key_states = key_states.contiguous()\n            value_states = value_states.contiguous()\n\n        # We dispatch to SDPA's Flash Attention or Efficient kernels via this `is_causal` if statement instead of an inline conditional assignment\n        # in SDPA to support both torch.compile's dynamic shapes and full graph options. An inline conditional prevents dynamic shapes from compiling.\n        is_causal = True if causal_mask is None and q_len > 1 else False\n\n        attn_output = torch.nn.functional.scaled_dot_product_attention(\n            query_states,\n            key_states,\n            value_states,\n            attn_mask=causal_mask,\n            dropout_p=self.attention_dropout if self.training else 0.0,\n            is_causal=is_causal,\n        )\n\n        attn_output = attn_output.transpose(1, 2).contiguous()\n        attn_output = attn_output.view(bsz, q_len, -1)\n\n        attn_output = self.o_proj(attn_output)\n\n        return attn_output, None, past_key_value\n\n\n\nLLAMA_ATTENTION_CLASSES = {\n    \"eager\": LlamaAttention,\n    \"flash_attention_2\": LlamaFlashAttention2,\n    \"sdpa\": LlamaSdpaAttention,\n}\n\n\nclass LlamaDecoderLayer(nn.Module):\n    def __init__(self, config: LlamaConfig, layer_idx: int):\n        super().__init__()\n        self.hidden_size = config.hidden_size\n\n        # self.self_attn = LLAMA_ATTENTION_CLASSES[config._attn_implementation](config=config, layer_idx=layer_idx)\n        self.self_attn = LLAMA_ATTENTION_CLASSES[\"sdpa\"](config=config, layer_idx=layer_idx)\n\n        self.mlp = LlamaMLP(config)\n        self.input_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.post_attention_layernorm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n\n    def forward(\n        self,\n        hidden_states: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_value: Optional[Cache] = None,\n        output_attentions: Optional[bool] = False,\n        use_cache: Optional[bool] = False,\n        cache_position: Optional[torch.LongTensor] = None,\n        position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,  # will become mandatory in v4.45\n        pst_token_indices: Optional[torch.Tensor] = None,\n        layer_index: Optional[int] = None,\n        first_token_indices: Optional[torch.Tensor] = None,\n        **kwargs,\n    ) -> Tuple[torch.FloatTensor, Optional[Tuple[torch.FloatTensor, torch.FloatTensor]]]:\n        \"\"\"\n        Args:\n            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`\n            attention_mask (`torch.FloatTensor`, *optional*):\n                attention mask of size `(batch_size, sequence_length)` if flash attention is used or `(batch_size, 1,\n                query_sequence_length, key_sequence_length)` if default attention is used.\n            output_attentions (`bool`, *optional*):\n                Whether or not to return the attentions tensors of all attention layers. See `attentions` under\n                returned tensors for more detail.\n            use_cache (`bool`, *optional*):\n                If set to `True`, `past_key_values` key value states are returned and can be used to speed up decoding\n                (see `past_key_values`).\n            past_key_value (`Tuple(torch.FloatTensor)`, *optional*): cached past key and value projection states\n            cache_position (`torch.LongTensor` of shape `(sequence_length)`, *optional*):\n                Indices depicting the position of the input sequence tokens in the sequence\n            position_embeddings (`Tuple[torch.FloatTensor, torch.FloatTensor]`, *optional*):\n                Tuple containing the cosine and sine positional embeddings of shape `(batch_size, seq_len, head_dim)`,\n                with `head_dim` being the embedding dimension of each attention head.\n            kwargs (`dict`, *optional*):\n                Arbitrary kwargs to be ignored, used for FSDP and other methods that injects code\n                into the model\n        \"\"\"\n        residual = hidden_states\n\n        hidden_states = self.input_layernorm(hidden_states)\n\n        # Self Attention\n        hidden_states, self_attn_weights, present_key_value = self.self_attn(\n            hidden_states=hidden_states,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_value=past_key_value,\n            output_attentions=output_attentions,\n            use_cache=use_cache,\n            cache_position=cache_position,\n            position_embeddings=position_embeddings,\n            **kwargs,\n        )\n\n        hidden_states = residual + hidden_states\n        # Fully Connected\n        residual = hidden_states\n        hidden_states = self.post_attention_layernorm(hidden_states)\n        hidden_states = self.mlp(hidden_states)\n\n        hidden_states = residual + hidden_states\n\n        outputs = (hidden_states,)\n\n        if output_attentions:\n            outputs += (self_attn_weights,)\n\n        if use_cache:\n            outputs += (present_key_value,)\n\n        return outputs\n\n\n@add_start_docstrings(\n    \"The bare LLaMA Model outputting raw hidden-states without any specific head on top.\",\n    LLAMA_START_DOCSTRING,\n)\nclass LlamaModel(LlamaPreTrainedModel):\n    \"\"\"\n    Transformer decoder consisting of *config.num_hidden_layers* layers. Each layer is a [`LlamaDecoderLayer`]\n\n    Args:\n        config: LlamaConfig\n    \"\"\"\n\n    def __init__(self, config: LlamaConfig):\n        super().__init__(config)\n        self.padding_idx = config.pad_token_id\n        self.vocab_size = config.vocab_size\n\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)\n        self.layers = nn.ModuleList(\n            [LlamaDecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]\n        )\n        self.norm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.rotary_emb = LlamaRotaryEmbedding(config=config)\n        self.gradient_checkpointing = False\n\n        # Initialize weights and apply final processing\n        self.post_init()\n        self.plan = 'vanilla'\n        self.tp_starting_index = 1\n        self.tp_exiting_index = 99\n\n    def get_input_embeddings(self):\n        return self.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.embed_tokens = value\n\n    @add_start_docstrings_to_model_forward(LLAMA_INPUTS_DOCSTRING)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPast]:\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        use_cache = use_cache if use_cache is not None else self.config.use_cache\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        if (input_ids is None) ^ (inputs_embeds is not None):\n            raise ValueError(\n                \"You cannot specify both input_ids and inputs_embeds at the same time, and must specify either one\"\n            )\n\n        if self.gradient_checkpointing and self.training and use_cache:\n            logger.warning_once(\n                \"`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.\"\n            )\n            use_cache = False\n        \n        if self.plan == \"tp\":\n            pst_token_indices = find_token_indices(input_ids, token=32000)\n        \n        if inputs_embeds is None:\n            inputs_embeds = self.embed_tokens(input_ids)\n\n        return_legacy_cache = False\n        if (\n            use_cache and not isinstance(past_key_values, Cache) and not self.training\n        ):  # kept for BC (non `Cache` `past_key_values` inputs)\n            return_legacy_cache = True\n            past_key_values = DynamicCache.from_legacy_cache(past_key_values)\n            logger.warning_once(\n                \"We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. \"\n                \"Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)\"\n            )\n\n        if cache_position is None:\n            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0\n            cache_position = torch.arange(\n                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device\n            )\n        if position_ids is None:\n            position_ids = cache_position.unsqueeze(0)\n\n        causal_mask = self._update_causal_mask(\n            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions\n        )\n        \n\n        hidden_states = inputs_embeds\n\n        # create position embeddings to be shared across the decoder layers\n        position_embeddings = self.rotary_emb(hidden_states, position_ids)\n\n        # decoder layers\n        all_hidden_states = () if output_hidden_states else None\n        all_self_attns = () if output_attentions else None\n        next_decoder_cache = None\n\n        for index, decoder_layer in enumerate(self.layers):\n            if output_hidden_states:\n                all_hidden_states += (hidden_states,)\n\n            if self.gradient_checkpointing and self.training:\n                layer_outputs = self._gradient_checkpointing_func(\n                    decoder_layer.__call__,\n                    hidden_states,\n                    causal_mask,\n                    position_ids,\n                    past_key_values,\n                    output_attentions,\n                    use_cache,\n                    cache_position,\n                    position_embeddings,\n                )\n            else:    \n                if self.plan == \"vanilla\":\n                    layer_outputs = decoder_layer(\n                        hidden_states,\n                        attention_mask=causal_mask,\n                        position_ids=position_ids,\n                        past_key_value=past_key_values,\n                        output_attentions=output_attentions,\n                        use_cache=use_cache,\n                        cache_position=cache_position,\n                        position_embeddings=position_embeddings,\n                    )\n                elif self.plan == \"tp\":\n                    layer_index = self.tp_starting_index\n                    exiting_index = self.tp_exiting_index\n                    assert layer_index < exiting_index\n                    if index < layer_index:\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                            position_embeddings=position_embeddings,\n                            pst_token_indices=pst_token_indices,\n                            layer_index=index,\n                            first_token_indices=first_token_indices,\n                        )\n                    elif index >= layer_index and index < exiting_index:\n                        B = hidden_states.shape[0]\n                        previous_sentence_embeddings = hidden_states[:, -1, :].clone()\n                        hidden_states[torch.arange(B), pst_token_indices, :] = previous_sentence_embeddings\n\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                            position_embeddings=position_embeddings,\n                            pst_token_indices=pst_token_indices,\n                            layer_index=index,\n                            first_token_indices=first_token_indices,\n                        )\n                    elif index >= exiting_index:\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                            position_embeddings=position_embeddings,\n                            pst_token_indices=pst_token_indices,\n                            layer_index=index,\n                            first_token_indices=first_token_indices,\n                        )\n                    else:\n                        raise ValueError(\"layer index error!\")\n                \n                else:\n                    raise ValueError(f\"The {self.plan} plan have not yet been implemented!\")\n\n            hidden_states = layer_outputs[0]\n\n            if use_cache:\n                next_decoder_cache = layer_outputs[2 if output_attentions else 1]\n\n            if output_attentions:\n                all_self_attns += (layer_outputs[1],)\n\n        hidden_states = self.norm(hidden_states)\n\n        # add hidden states from the last decoder layer\n        if output_hidden_states:\n            all_hidden_states += (hidden_states,)\n\n        next_cache = next_decoder_cache if use_cache else None\n        if return_legacy_cache:\n            next_cache = next_cache.to_legacy_cache()\n\n        if not return_dict:\n            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)\n        return BaseModelOutputWithPast(\n            last_hidden_state=hidden_states,\n            past_key_values=next_cache,\n            hidden_states=all_hidden_states,\n            attentions=all_self_attns,\n        )\n\n    def _update_causal_mask(\n        self,\n        attention_mask: torch.Tensor,\n        input_tensor: torch.Tensor,\n        cache_position: torch.Tensor,\n        past_key_values: Cache,\n        output_attentions: bool,\n    ):\n        # TODO: As of torch==2.2.0, the `attention_mask` passed to the model in `generate` is 2D and of dynamic length even when the static\n        # KV cache is used. This is an issue for torch.compile which then recaptures cudagraphs at each decode steps due to the dynamic shapes.\n        # (`recording cudagraph tree for symint key 13`, etc.), which is VERY slow. A workaround is `@torch.compiler.disable`, but this prevents using\n        # `fullgraph=True`. See more context in https://github.com/huggingface/transformers/pull/29114\n\n        if self.config._attn_implementation == \"flash_attention_2\":\n            if attention_mask is not None and 0.0 in attention_mask:\n                return attention_mask\n            return None\n\n        # For SDPA, when possible, we will rely on its `is_causal` argument instead of its `attn_mask` argument, in\n        # order to dispatch on Flash Attention 2. This feature is not compatible with static cache, as SDPA will fail\n        # to infer the attention mask.\n        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0\n        using_static_cache = isinstance(past_key_values, StaticCache)\n\n        # When output attentions is True, sdpa implementation's forward method calls the eager implementation's forward\n        if self.config._attn_implementation == \"sdpa\" and not using_static_cache and not output_attentions:\n            if AttentionMaskConverter._ignore_causal_mask_sdpa(\n                attention_mask,\n                inputs_embeds=input_tensor,\n                past_key_values_length=past_seen_tokens,\n                is_training=self.training,\n            ):\n                return None\n\n        dtype, device = input_tensor.dtype, input_tensor.device\n        min_dtype = torch.finfo(dtype).min\n        sequence_length = input_tensor.shape[1]\n        if using_static_cache:\n            target_length = past_key_values.get_max_length()\n        else:\n            target_length = (\n                attention_mask.shape[-1]\n                if isinstance(attention_mask, torch.Tensor)\n                else past_seen_tokens + sequence_length + 1\n            )\n\n        if attention_mask is not None and attention_mask.dim() == 4:\n            # in this case we assume that the mask comes already in inverted form and requires no inversion or slicing\n            if attention_mask.max() != 0:\n                raise ValueError(\"Custom 4D attention mask should be passed in inverted form with max==0`\")\n            causal_mask = attention_mask\n        else:\n            causal_mask = torch.full(\n                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=device\n            )\n            if sequence_length != 1:\n                causal_mask = torch.triu(causal_mask, diagonal=1)\n            causal_mask *= torch.arange(target_length, device=device) > cache_position.reshape(-1, 1)\n            causal_mask = causal_mask[None, None, :, :].expand(input_tensor.shape[0], 1, -1, -1)\n            if attention_mask is not None:\n                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit\n                mask_length = attention_mask.shape[-1]\n                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :]\n                padding_mask = padding_mask == 0\n                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(\n                    padding_mask, min_dtype\n                )\n        if (\n            self.config._attn_implementation == \"sdpa\"\n            and attention_mask is not None\n            and attention_mask.device.type == \"cuda\"\n            and not output_attentions\n        ):\n            # Attend to all tokens in fully masked rows in the causal_mask, for example the relevant first rows when\n            # using left padding. This is required by F.scaled_dot_product_attention memory-efficient attention path.\n            # Details: https://github.com/pytorch/pytorch/issues/110213\n            causal_mask = AttentionMaskConverter._unmask_unattended(causal_mask, min_dtype)\n\n        return causal_mask\n\n\nclass LlamaForCausalLM(LlamaPreTrainedModel):\n    _tied_weights_keys = [\"lm_head.weight\"]\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.model = LlamaModel(config)\n        self.vocab_size = config.vocab_size\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def set_decoder(self, decoder):\n        self.model = decoder\n\n    def get_decoder(self):\n        return self.model\n\n    @add_start_docstrings_to_model_forward(LLAMA_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=CausalLMOutputWithPast, config_class=_CONFIG_FOR_DOC)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        labels: Optional[torch.LongTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, CausalLMOutputWithPast]:\n        r\"\"\"\n        Args:\n            labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n                Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,\n                config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored\n                (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.\n\n        Returns:\n\n        Example:\n\n        ```python\n        >>> from transformers import AutoTokenizer, LlamaForCausalLM\n\n        >>> model = LlamaForCausalLM.from_pretrained(\"meta-llama/Llama-2-7b-hf\")\n        >>> tokenizer = AutoTokenizer.from_pretrained(\"meta-llama/Llama-2-7b-hf\")\n\n        >>> prompt = \"Hey, are you conscious? Can you talk to me?\"\n        >>> inputs = tokenizer(prompt, return_tensors=\"pt\")\n\n        >>> # Generate\n        >>> generate_ids = model.generate(inputs.input_ids, max_length=30)\n        >>> tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]\n        \"Hey, are you conscious? Can you talk to me?\\nI'm not conscious, but I can talk to you.\"\n        ```\"\"\"\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        # decoder outputs consists of (dec_features, layer_state, dec_hidden, dec_attn)\n        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_values=past_key_values,\n            inputs_embeds=inputs_embeds,\n            use_cache=use_cache,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            cache_position=cache_position,\n        )\n\n        hidden_states = outputs[0]\n        if self.config.pretraining_tp > 1:\n            lm_head_slices = self.lm_head.weight.split(self.vocab_size // self.config.pretraining_tp, dim=0)\n            logits = [F.linear(hidden_states, lm_head_slices[i]) for i in range(self.config.pretraining_tp)]\n            logits = torch.cat(logits, dim=-1)\n        else:\n            logits = self.lm_head(hidden_states)\n        logits = logits.float()\n\n        loss = None\n        if labels is not None:\n            # Shift so that tokens < n predict n\n            shift_logits = logits[..., :-1, :].contiguous()\n            shift_labels = labels[..., 1:].contiguous()\n            # Flatten the tokens\n            loss_fct = CrossEntropyLoss()\n            shift_logits = shift_logits.view(-1, self.config.vocab_size)\n            shift_labels = shift_labels.view(-1)\n            # Enable model parallelism\n            shift_labels = shift_labels.to(shift_logits.device)\n            loss = loss_fct(shift_logits, shift_labels)\n\n        if not return_dict:\n            output = (logits,) + outputs[1:]\n            return (loss,) + output if loss is not None else output\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=outputs.past_key_values,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n\n    def prepare_inputs_for_generation(\n        self,\n        input_ids,\n        past_key_values=None,\n        attention_mask=None,\n        inputs_embeds=None,\n        cache_position=None,\n        position_ids=None,\n        use_cache=True,\n        **kwargs,\n    ):\n        # If we have cache: let's slice `input_ids` through `cache_position`, to keep only the unprocessed tokens\n        # Exception 1: when passing input_embeds, input_ids may be missing entries\n        # Exception 2: some generation methods do special slicing of input_ids, so we don't need to do it here\n        if past_key_values is not None:\n            if inputs_embeds is not None:  # Exception 1\n                input_ids = input_ids[:, -cache_position.shape[0] :]\n            elif input_ids.shape[1] != cache_position.shape[0]:  # Default case (the \"else\", a no op, is Exception 2)\n                input_ids = input_ids[:, cache_position]\n\n        if attention_mask is not None and position_ids is None:\n            # create position_ids on the fly for batch generation\n            position_ids = attention_mask.long().cumsum(-1) - 1\n            position_ids.masked_fill_(attention_mask == 0, 1)\n            if past_key_values:\n                position_ids = position_ids[:, -input_ids.shape[1] :]\n\n        # if `inputs_embeds` are passed, we only want to use them in the 1st generation step\n        if inputs_embeds is not None and cache_position[0] == 0:\n            model_inputs = {\"inputs_embeds\": inputs_embeds}\n        else:\n            model_inputs = {\"input_ids\": input_ids.contiguous()}  # `contiguous()` needed for compilation use cases\n\n        model_inputs.update(\n            {\n                \"position_ids\": position_ids,\n                \"cache_position\": cache_position,\n                \"past_key_values\": past_key_values,\n                \"use_cache\": use_cache,\n                \"attention_mask\": attention_mask,\n            }\n        )\n        return model_inputs\n",
    "senllm/modeling_gemma2.py": "from typing import List, Optional, Tuple, Union\n\nimport torch\nimport torch.utils.checkpoint\nfrom torch import nn\nfrom torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss\n\nfrom transformers.activations import ACT2FN\nfrom transformers.cache_utils import Cache\nfrom transformers.modeling_outputs import (\n    BaseModelOutputWithPast,\n    CausalLMOutputWithPast,\n    SequenceClassifierOutputWithPast,\n    TokenClassifierOutput,\n)\nfrom transformers.modeling_utils import PreTrainedModel\nfrom transformers.utils import (\n    add_start_docstrings,\n    add_start_docstrings_to_model_forward,\n    is_flash_attn_2_available,\n    is_flash_attn_greater_or_equal,\n    is_flash_attn_greater_or_equal_2_10,\n    logging,\n    replace_return_docstrings,\n)\nfrom transformers.models.gemma2.configuration_gemma2 import Gemma2Config\nfrom transformers.models.gemma2.modeling_gemma2 import *\n\n\ndef find_token_indices(input_ids, token=32000):\n    # \u65ad\u8a00 input_ids \u4e2d\u6240\u6709\u5e8f\u5217\u90fd\u5305\u542b\u5143\u7d20 32000\n    assert (input_ids == token).any(dim=1).all(), f\"Not all sequences contain the token {token}\"\n    \n    # \u83b7\u53d6\u7b2c\u4e00\u4e2a\u5339\u914d 32000 \u7684\u7d22\u5f15\n    mask = (input_ids == token)\n    # \u8f6c\u6362\u4e3a\u6d6e\u70b9\u578b\u4ee5\u4fbf\u4f7f\u7528argmax\n    mask_float = mask.float()\n    # \u8ba1\u7b97\u7b2c\u4e00\u4e2a\u5339\u914d\u7684\u7d22\u5f15\n    first_match_indices = mask_float.argmax(dim=1)\n    \n    return first_match_indices\n\n_CONFIG_FOR_DOC = \"Gemma2Config\"\n\n@add_start_docstrings(\n    \"The bare Gemma2 Model outputting raw hidden-states without any specific head on top.\",\n    GEMMA2_START_DOCSTRING,\n)\nclass Gemma2Model(Gemma2PreTrainedModel):\n    \"\"\"\n    Transformer decoder consisting of *config.num_hidden_layers* layers. Each layer is a [`Gemma2DecoderLayer`]\n\n    Args:\n        config: Gemma2Config\n    \"\"\"\n\n    def __init__(self, config: Gemma2Config):\n        super().__init__(config)\n        self.padding_idx = config.pad_token_id\n        self.vocab_size = config.vocab_size\n\n        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)\n        self.layers = nn.ModuleList(\n            [Gemma2DecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]\n        )\n        self.norm = Gemma2RMSNorm(config.hidden_size, eps=config.rms_norm_eps)\n        self.gradient_checkpointing = False\n\n        # Initialize weights and apply final processing\n        self.post_init()\n        self.plan = 'tp'\n        self.tp_starting_index = 1\n        self.tp_exiting_index = 99\n\n    def get_input_embeddings(self):\n        return self.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.embed_tokens = value\n\n    @add_start_docstrings_to_model_forward(GEMMA2_INPUTS_DOCSTRING)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, BaseModelOutputWithPast]:\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        use_cache = use_cache if use_cache is not None else self.config.use_cache\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        if (input_ids is None) ^ (inputs_embeds is not None):\n            raise ValueError(\n                \"You cannot specify both input_ids and inputs_embeds at the same time, and must specify either one\"\n            )\n\n        if self.gradient_checkpointing and self.training and use_cache:\n            logger.warning_once(\n                \"`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.\"\n            )\n            use_cache = False\n        \n        if self.plan == \"tp\":\n            pst_token_indices = find_token_indices(input_ids, token=256000)\n\n        if inputs_embeds is None:\n            inputs_embeds = self.embed_tokens(input_ids)\n\n        if cache_position is None:\n            cache_position = torch.arange(0, inputs_embeds.shape[1], device=inputs_embeds.device)\n\n        if position_ids is None:\n            position_ids = cache_position.unsqueeze(0)\n\n        causal_mask = self._update_causal_mask(\n            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions\n        )\n\n        # embed positions\n        hidden_states = inputs_embeds\n\n        # normalized\n        # Gemma2 downcasts the below to float16, causing sqrt(3072)=55.4256 to become 55.5\n        # See https://github.com/huggingface/transformers/pull/29402\n        normalizer = torch.tensor(self.config.hidden_size**0.5, dtype=hidden_states.dtype)\n        hidden_states = hidden_states * normalizer\n\n        all_hidden_states = () if output_hidden_states else None\n        all_self_attns = () if output_attentions else None\n\n        for index, decoder_layer in enumerate(self.layers):\n            if output_hidden_states:\n                all_hidden_states += (hidden_states,)\n\n            if self.gradient_checkpointing and self.training:\n                layer_outputs = self._gradient_checkpointing_func(\n                    decoder_layer.__call__,\n                    hidden_states,\n                    causal_mask,\n                    position_ids,\n                    past_key_values,\n                    output_attentions,\n                    use_cache,\n                    cache_position,\n                )\n            else:\n                if self.plan == \"vanilla\":\n                    layer_outputs = decoder_layer(\n                        hidden_states,\n                        attention_mask=causal_mask,\n                        position_ids=position_ids,\n                        past_key_value=past_key_values,\n                        output_attentions=output_attentions,\n                        use_cache=use_cache,\n                        cache_position=cache_position,\n                    )\n                elif self.plan == \"tp\":\n                    layer_index = self.tp_starting_index\n                    exiting_index = self.tp_exiting_index\n                    assert layer_index < exiting_index\n                    if index < layer_index:\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                        )\n                    elif index >= layer_index and index < exiting_index:\n                        B = hidden_states.shape[0]\n                        previous_sentence_embeddings = hidden_states[:, -1, :].clone()\n                        hidden_states[torch.arange(B), pst_token_indices, :] = previous_sentence_embeddings\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                        )\n                    elif index >= exiting_index:\n                        layer_outputs = decoder_layer(\n                            hidden_states,\n                            attention_mask=causal_mask,\n                            position_ids=position_ids,\n                            past_key_value=past_key_values,\n                            output_attentions=output_attentions,\n                            use_cache=use_cache,\n                            cache_position=cache_position,\n                        )\n                    else:\n                        raise ValueError(\"layer index error!\")\n\n                else:\n                    raise ValueError(\"Other plans are not supported now!\")\n\n\n            hidden_states = layer_outputs[0]\n\n            if output_attentions:\n                all_self_attns += (layer_outputs[1],)\n\n        hidden_states = self.norm(hidden_states)\n\n        # add hidden states from the last decoder layer\n        if output_hidden_states:\n            all_hidden_states += (hidden_states,)\n\n        next_cache = past_key_values if use_cache else None\n\n        if not return_dict:\n            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)\n        return BaseModelOutputWithPast(\n            last_hidden_state=hidden_states,\n            past_key_values=next_cache,\n            hidden_states=all_hidden_states,\n            attentions=all_self_attns,\n        )\n\n    def _update_causal_mask(\n        self,\n        attention_mask: torch.Tensor,\n        input_tensor: torch.Tensor,\n        cache_position: torch.Tensor,\n        past_key_values: Cache,\n        output_attentions: bool,\n    ):\n        if self.config._attn_implementation == \"flash_attention_2\":\n            if attention_mask is not None and 0.0 in attention_mask:\n                return attention_mask\n            return None\n\n        dtype, device = input_tensor.dtype, input_tensor.device\n        min_dtype = torch.finfo(dtype).min\n        sequence_length = input_tensor.shape[1]\n        if past_key_values is not None:\n            target_length = past_key_values.get_max_length()\n        else:\n            target_length = attention_mask.shape[-1] if attention_mask is not None else input_tensor.shape[1]\n\n        if attention_mask is not None and attention_mask.dim() == 4:\n            # in this case we assume that the mask comes already in inverted form and requires no inversion or slicing\n            if attention_mask.max() != 0:\n                raise ValueError(\"Custom 4D attention mask should be passed in inverted form with max==0`\")\n            causal_mask = attention_mask\n        else:\n            causal_mask = torch.full(\n                (sequence_length, target_length), fill_value=min_dtype, dtype=dtype, device=device\n            )\n            if sequence_length != 1:\n                causal_mask = torch.triu(causal_mask, diagonal=1)\n            causal_mask *= torch.arange(target_length, device=device) > cache_position.reshape(-1, 1)\n            causal_mask = causal_mask[None, None, :, :].expand(input_tensor.shape[0], 1, -1, -1)\n            if attention_mask is not None:\n                causal_mask = causal_mask.clone()  # copy to contiguous memory for in-place edit\n                mask_length = attention_mask.shape[-1]\n                padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :]\n                padding_mask = padding_mask == 0\n                causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(\n                    padding_mask, min_dtype\n                )\n        return causal_mask\n\n\nclass Gemma2ForCausalLM(Gemma2PreTrainedModel):\n    _tied_weights_keys = [\"lm_head.weight\"]\n\n    def __init__(self, config):\n        super().__init__(config)\n        self.model = Gemma2Model(config)\n        self.vocab_size = config.vocab_size\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n\n        # Initialize weights and apply final processing\n        self.post_init()\n\n    def get_input_embeddings(self):\n        return self.model.embed_tokens\n\n    def set_input_embeddings(self, value):\n        self.model.embed_tokens = value\n\n    def get_output_embeddings(self):\n        return self.lm_head\n\n    def set_output_embeddings(self, new_embeddings):\n        self.lm_head = new_embeddings\n\n    def set_decoder(self, decoder):\n        self.model = decoder\n\n    def get_decoder(self):\n        return self.model\n\n    @add_start_docstrings_to_model_forward(GEMMA2_INPUTS_DOCSTRING)\n    @replace_return_docstrings(output_type=CausalLMOutputWithPast, config_class=_CONFIG_FOR_DOC)\n    def forward(\n        self,\n        input_ids: torch.LongTensor = None,\n        attention_mask: Optional[torch.Tensor] = None,\n        position_ids: Optional[torch.LongTensor] = None,\n        past_key_values: Optional[Union[Cache, List[torch.FloatTensor]]] = None,\n        inputs_embeds: Optional[torch.FloatTensor] = None,\n        labels: Optional[torch.LongTensor] = None,\n        use_cache: Optional[bool] = None,\n        output_attentions: Optional[bool] = None,\n        output_hidden_states: Optional[bool] = None,\n        return_dict: Optional[bool] = None,\n        cache_position: Optional[torch.LongTensor] = None,\n    ) -> Union[Tuple, CausalLMOutputWithPast]:\n        r\"\"\"\n        Args:\n            labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):\n                Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,\n                config.vocab_size]` or -100 (see `input_ids` docstring). Tokens with indices set to `-100` are ignored\n                (masked), the loss is only computed for the tokens with labels in `[0, ..., config.vocab_size]`.\n\n        Returns:\n\n        Example:\n\n        ```python\n        >>> from transformers import AutoTokenizer, GemmaForCausalLM\n\n        >>> model = GemmaForCausalLM.from_pretrained(\"google/gemma-2-9b\")\n        >>> tokenizer = AutoTokenizer.from_pretrained(\"google/gemma-2-9b\")\n\n        >>> prompt = \"What is your favorite condiment?\"\n        >>> inputs = tokenizer(prompt, return_tensors=\"pt\")\n\n        >>> # Generate\n        >>> generate_ids = model.generate(inputs.input_ids, max_length=30)\n        >>> tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]\n        \"What is your favorite condiment?\"\n        ```\"\"\"\n        if self.training and self.config._attn_implementation != \"eager\":\n            logger.warning_once(\n                \"It is strongly recommended to train Gemma2 models with the `eager` attention implementation \"\n                f\"instead of `{self.config._attn_implementation}`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.\"\n            )\n        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions\n        output_hidden_states = (\n            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states\n        )\n        return_dict = return_dict if return_dict is not None else self.config.use_return_dict\n\n        # decoder outputs consists of (dec_features, layer_state, dec_hidden, dec_attn)\n        outputs = self.model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            position_ids=position_ids,\n            past_key_values=past_key_values,\n            inputs_embeds=inputs_embeds,\n            use_cache=use_cache,\n            output_attentions=output_attentions,\n            output_hidden_states=output_hidden_states,\n            return_dict=return_dict,\n            cache_position=cache_position,\n        )\n\n        hidden_states = outputs[0]\n        logits = self.lm_head(hidden_states)\n        if self.config.final_logit_softcapping is not None:\n            logits = logits / self.config.final_logit_softcapping\n            logits = torch.tanh(logits)\n            logits = logits * self.config.final_logit_softcapping\n\n        logits = logits.float()\n        loss = None\n        if labels is not None:\n            # Shift so that tokens < n predict n\n            shift_logits = logits[..., :-1, :].contiguous()\n            shift_labels = labels[..., 1:].contiguous()\n            # Flatten the tokens\n            loss_fct = CrossEntropyLoss()\n            shift_logits = shift_logits.view(-1, self.config.vocab_size)\n            shift_labels = shift_labels.view(-1)\n            # Enable model parallelism\n            shift_labels = shift_labels.to(shift_logits.device)\n            loss = loss_fct(shift_logits, shift_labels)\n\n        if not return_dict:\n            output = (logits,) + outputs[1:]\n            return (loss,) + output if loss is not None else output\n\n        return CausalLMOutputWithPast(\n            loss=loss,\n            logits=logits,\n            past_key_values=outputs.past_key_values,\n            hidden_states=outputs.hidden_states,\n            attentions=outputs.attentions,\n        )\n\n    def prepare_inputs_for_generation(\n        self,\n        input_ids,\n        past_key_values=None,\n        attention_mask=None,\n        inputs_embeds=None,\n        cache_position=None,\n        position_ids=None,\n        use_cache=True,\n        **kwargs,\n    ):\n        # If we have cache: let's slice `input_ids` through `cache_position`, to keep only the unprocessed tokens\n        # Exception 1: when passing input_embeds, input_ids may be missing entries\n        # Exception 2: some generation methods do special slicing of input_ids, so we don't need to do it here\n        if past_key_values is not None:\n            if inputs_embeds is not None:  # Exception 1\n                input_ids = input_ids[:, -cache_position.shape[0] :]\n            elif input_ids.shape[1] != cache_position.shape[0]:  # Default case (the \"else\", a no op, is Exception 2)\n                input_ids = input_ids[:, cache_position]\n\n        if attention_mask is not None and position_ids is None:\n            # create position_ids on the fly for batch generation\n            position_ids = attention_mask.long().cumsum(-1) - 1\n            position_ids.masked_fill_(attention_mask == 0, 1)\n            if past_key_values:\n                position_ids = position_ids[:, -input_ids.shape[1] :]\n\n        # if `inputs_embeds` are passed, we only want to use them in the 1st generation step\n        if inputs_embeds is not None and cache_position[0] == 0:\n            model_inputs = {\"inputs_embeds\": inputs_embeds}\n        else:\n            model_inputs = {\"input_ids\": input_ids.contiguous()}  # `contiguous()` needed for compilation use cases\n\n        model_inputs.update(\n            {\n                \"position_ids\": position_ids,\n                \"cache_position\": cache_position,\n                \"past_key_values\": past_key_values,\n                \"use_cache\": use_cache,\n                \"attention_mask\": attention_mask,\n            }\n        )\n        return model_inputs\n",
}

for rel_path, content in FILES.items():
    path = PROJECT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

print(f"Wrote {len(FILES)} files to {PROJECT_DIR}")


In [ ]:
# Write minimal SentEval package files
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/token_prepending")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "SentEval/setup.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\nimport io\nfrom setuptools import setup, find_packages\n\nwith io.open('./README.md', encoding='utf-8') as f:\n    readme = f.read()\n\nsetup(\n    name='SentEval',\n    version='0.1.0',\n    url='https://github.com/facebookresearch/SentEval',\n    packages=find_packages(exclude=['examples']),\n    license='Attribution-NonCommercial 4.0 International',\n    long_description=readme,\n)\n",
    "SentEval/senteval/__init__.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\nfrom __future__ import absolute_import\n\nfrom senteval.engine import SE\n",
    "SentEval/senteval/binary.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nBinary classifier and corresponding datasets : MR, CR, SUBJ, MPQA\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport io\nimport os\nimport numpy as np\nimport logging\n\nfrom senteval.tools.validation import InnerKFoldClassifier\n\n\nclass BinaryClassifierEval(object):\n    def __init__(self, pos, neg, seed=1111):\n        self.seed = seed\n        self.samples, self.labels = pos + neg, [1] * len(pos) + [0] * len(neg)\n        self.n_samples = len(self.samples)\n\n    def do_prepare(self, params, prepare):\n        # prepare is given the whole text\n        return prepare(params, self.samples)\n        # prepare puts everything it outputs in \"params\" : params.word2id etc\n        # Those output will be further used by \"batcher\".\n\n    def loadFile(self, fpath):\n        with io.open(fpath, 'r', encoding='latin-1') as f:\n            return [line.split() for line in f.read().splitlines()]\n\n    def run(self, params, batcher):\n        enc_input = []\n        # Sort to reduce padding\n        sorted_corpus = sorted(zip(self.samples, self.labels),\n                               key=lambda z: (len(z[0]), z[1]))\n        sorted_samples = [x for (x, y) in sorted_corpus]\n        sorted_labels = [y for (x, y) in sorted_corpus]\n        logging.info('Generating sentence embeddings')\n        for ii in range(0, self.n_samples, params.batch_size):\n            batch = sorted_samples[ii:ii + params.batch_size]\n            embeddings = batcher(params, batch)\n            enc_input.append(embeddings)\n        enc_input = np.vstack(enc_input)\n        logging.info('Generated sentence embeddings')\n\n        config = {'nclasses': 2, 'seed': self.seed,\n                  'usepytorch': params.usepytorch,\n                  'classifier': params.classifier,\n                  'nhid': params.nhid, 'kfold': params.kfold}\n        clf = InnerKFoldClassifier(enc_input, np.array(sorted_labels), config)\n        devacc, testacc = clf.run()\n        logging.debug('Dev acc : {0} Test acc : {1}\\n'.format(devacc, testacc))\n        return {'devacc': devacc, 'acc': testacc, 'ndev': self.n_samples,\n                'ntest': self.n_samples}\n\n\nclass CREval(BinaryClassifierEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : CR *****\\n\\n')\n        pos = self.loadFile(os.path.join(task_path, 'custrev.pos'))\n        neg = self.loadFile(os.path.join(task_path, 'custrev.neg'))\n        super(self.__class__, self).__init__(pos, neg, seed)\n\n\nclass MREval(BinaryClassifierEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : MR *****\\n\\n')\n        pos = self.loadFile(os.path.join(task_path, 'rt-polarity.pos'))\n        neg = self.loadFile(os.path.join(task_path, 'rt-polarity.neg'))\n        super(self.__class__, self).__init__(pos, neg, seed)\n\n\nclass SUBJEval(BinaryClassifierEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : SUBJ *****\\n\\n')\n        obj = self.loadFile(os.path.join(task_path, 'subj.objective'))\n        subj = self.loadFile(os.path.join(task_path, 'subj.subjective'))\n        super(self.__class__, self).__init__(obj, subj, seed)\n\n\nclass MPQAEval(BinaryClassifierEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : MPQA *****\\n\\n')\n        pos = self.loadFile(os.path.join(task_path, 'mpqa.pos'))\n        neg = self.loadFile(os.path.join(task_path, 'mpqa.neg'))\n        super(self.__class__, self).__init__(pos, neg, seed)\n",
    "SentEval/senteval/engine.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\n\nGeneric sentence evaluation scripts wrapper\n\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nfrom senteval import utils\nfrom senteval.binary import CREval, MREval, MPQAEval, SUBJEval\nfrom senteval.snli import SNLIEval\nfrom senteval.trec import TRECEval\nfrom senteval.sick import SICKEntailmentEval, SICKEval\nfrom senteval.mrpc import MRPCEval\nfrom senteval.sts import STS12Eval, STS13Eval, STS14Eval, STS15Eval, STS16Eval, STSBenchmarkEval, SICKRelatednessEval, STSBenchmarkFinetune, STSBenchmarkEvalDev\nfrom senteval.sst import SSTEval\nfrom senteval.rank import ImageCaptionRetrievalEval\nfrom senteval.probing import *\n\nclass SE(object):\n    def __init__(self, params, batcher, prepare=None):\n        # parameters\n        params = utils.dotdict(params)\n        params.usepytorch = True if 'usepytorch' not in params else params.usepytorch\n        params.seed = 1111 if 'seed' not in params else params.seed\n\n        params.batch_size = 128 if 'batch_size' not in params else params.batch_size\n        params.nhid = 0 if 'nhid' not in params else params.nhid\n        params.kfold = 5 if 'kfold' not in params else params.kfold\n\n        if 'classifier' not in params or not params['classifier']:\n            params.classifier = {'nhid': 0}\n\n        assert 'nhid' in params.classifier, 'Set number of hidden units in classifier config!!'\n\n        self.params = params\n\n        # batcher and prepare\n        self.batcher = batcher\n        self.prepare = prepare if prepare else lambda x, y: None\n\n        self.list_tasks = ['CR', 'MR', 'MPQA', 'SUBJ', 'SST2', 'SST5', 'TREC', 'MRPC',\n                           'SICKRelatedness', 'SICKEntailment', 'STSBenchmark',\n                           'SNLI', 'ImageCaptionRetrieval', 'STS12', 'STS13',\n                           'STS14', 'STS15', 'STS16',\n                           'Length', 'WordContent', 'Depth', 'TopConstituents',\n                           'BigramShift', 'Tense', 'SubjNumber', 'ObjNumber',\n                           'OddManOut', 'CoordinationInversion', 'SICKRelatedness-finetune', 'STSBenchmark-finetune', 'STSBenchmark-fix', 'STSBenchmark-dev']\n\n    def eval(self, name):\n        # evaluate on evaluation [name], either takes string or list of strings\n        if (isinstance(name, list)):\n            self.results = {x: self.eval(x) for x in name}\n            return self.results\n\n        tpath = self.params.task_path\n        assert name in self.list_tasks, str(name) + ' not in ' + str(self.list_tasks)\n\n        # Original SentEval tasks\n        if name == 'CR':\n            self.evaluation = CREval(tpath + '/downstream/CR', seed=self.params.seed)\n        elif name == 'MR':\n            self.evaluation = MREval(tpath + '/downstream/MR', seed=self.params.seed)\n        elif name == 'MPQA':\n            self.evaluation = MPQAEval(tpath + '/downstream/MPQA', seed=self.params.seed)\n        elif name == 'SUBJ':\n            self.evaluation = SUBJEval(tpath + '/downstream/SUBJ', seed=self.params.seed)\n        elif name == 'SST2':\n            self.evaluation = SSTEval(tpath + '/downstream/SST/binary', nclasses=2, seed=self.params.seed)\n        elif name == 'SST5':\n            self.evaluation = SSTEval(tpath + '/downstream/SST/fine', nclasses=5, seed=self.params.seed)\n        elif name == 'TREC':\n            self.evaluation = TRECEval(tpath + '/downstream/TREC', seed=self.params.seed)\n        elif name == 'MRPC':\n            self.evaluation = MRPCEval(tpath + '/downstream/MRPC', seed=self.params.seed)\n        elif name == 'SICKRelatedness':\n            self.evaluation = SICKRelatednessEval(tpath + '/downstream/SICK', seed=self.params.seed)\n        elif name == 'STSBenchmark':\n            self.evaluation = STSBenchmarkEval(tpath + '/downstream/STS/STSBenchmark', seed=self.params.seed)\n        elif name == 'STSBenchmark-dev':\n            self.evaluation = STSBenchmarkEvalDev(tpath + '/downstream/STS/STSBenchmark', seed=self.params.seed)\n        elif name == 'STSBenchmark-fix':\n            self.evaluation = STSBenchmarkEval(tpath + '/downstream/STS/STSBenchmark-fix', seed=self.params.seed)\n        elif name == 'STSBenchmark-finetune':\n            self.evaluation = STSBenchmarkFinetune(tpath + '/downstream/STS/STSBenchmark', seed=self.params.seed)\n        elif name == 'SICKRelatedness-finetune':\n            self.evaluation = SICKEval(tpath + '/downstream/SICK', seed=self.params.seed)\n        elif name == 'SICKEntailment':\n            self.evaluation = SICKEntailmentEval(tpath + '/downstream/SICK', seed=self.params.seed)\n        elif name == 'SNLI':\n            self.evaluation = SNLIEval(tpath + '/downstream/SNLI', seed=self.params.seed)\n        elif name in ['STS12', 'STS13', 'STS14', 'STS15', 'STS16']:\n            fpath = name + '-en-test'\n            self.evaluation = eval(name + 'Eval')(tpath + '/downstream/STS/' + fpath, seed=self.params.seed)\n        elif name == 'ImageCaptionRetrieval':\n            self.evaluation = ImageCaptionRetrievalEval(tpath + '/downstream/COCO', seed=self.params.seed)\n\n        # Probing Tasks\n        elif name == 'Length':\n                self.evaluation = LengthEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'WordContent':\n                self.evaluation = WordContentEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'Depth':\n                self.evaluation = DepthEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'TopConstituents':\n                self.evaluation = TopConstituentsEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'BigramShift':\n                self.evaluation = BigramShiftEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'Tense':\n                self.evaluation = TenseEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'SubjNumber':\n                self.evaluation = SubjNumberEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'ObjNumber':\n                self.evaluation = ObjNumberEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'OddManOut':\n                self.evaluation = OddManOutEval(tpath + '/probing', seed=self.params.seed)\n        elif name == 'CoordinationInversion':\n                self.evaluation = CoordinationInversionEval(tpath + '/probing', seed=self.params.seed)\n\n        self.params.current_task = name\n        self.evaluation.do_prepare(self.params, self.prepare)\n\n        self.results = self.evaluation.run(self.params, self.batcher)\n\n        return self.results\n",
    "SentEval/senteval/mrpc.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nMRPC : Microsoft Research Paraphrase (detection) Corpus\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport logging\nimport numpy as np\nimport io\n\nfrom senteval.tools.validation import KFoldClassifier\n\nfrom sklearn.metrics import f1_score\n\n\nclass MRPCEval(object):\n    def __init__(self, task_path, seed=1111):\n        logging.info('***** Transfer task : MRPC *****\\n\\n')\n        self.seed = seed\n        train = self.loadFile(os.path.join(task_path,\n                              'msr_paraphrase_train.txt'))\n        test = self.loadFile(os.path.join(task_path,\n                             'msr_paraphrase_test.txt'))\n        self.mrpc_data = {'train': train, 'test': test}\n\n    def do_prepare(self, params, prepare):\n        # TODO : Should we separate samples in \"train, test\"?\n        samples = self.mrpc_data['train']['X_A'] + \\\n                  self.mrpc_data['train']['X_B'] + \\\n                  self.mrpc_data['test']['X_A'] + self.mrpc_data['test']['X_B']\n        return prepare(params, samples)\n\n    def loadFile(self, fpath):\n        mrpc_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                text = line.strip().split('\\t')\n                mrpc_data['X_A'].append(text[3].split())\n                mrpc_data['X_B'].append(text[4].split())\n                mrpc_data['y'].append(text[0])\n\n        mrpc_data['X_A'] = mrpc_data['X_A'][1:]\n        mrpc_data['X_B'] = mrpc_data['X_B'][1:]\n        mrpc_data['y'] = [int(s) for s in mrpc_data['y'][1:]]\n        return mrpc_data\n\n    def run(self, params, batcher):\n        mrpc_embed = {'train': {}, 'test': {}}\n\n        for key in self.mrpc_data:\n            logging.info('Computing embedding for {0}'.format(key))\n            # Sort to reduce padding\n            text_data = {}\n            sorted_corpus = sorted(zip(self.mrpc_data[key]['X_A'],\n                                       self.mrpc_data[key]['X_B'],\n                                       self.mrpc_data[key]['y']),\n                                   key=lambda z: (len(z[0]), len(z[1]), z[2]))\n\n            text_data['A'] = [x for (x, y, z) in sorted_corpus]\n            text_data['B'] = [y for (x, y, z) in sorted_corpus]\n            text_data['y'] = [z for (x, y, z) in sorted_corpus]\n\n            for txt_type in ['A', 'B']:\n                mrpc_embed[key][txt_type] = []\n                for ii in range(0, len(text_data['y']), params.batch_size):\n                    batch = text_data[txt_type][ii:ii + params.batch_size]\n                    embeddings = batcher(params, batch)\n                    mrpc_embed[key][txt_type].append(embeddings)\n                mrpc_embed[key][txt_type] = np.vstack(mrpc_embed[key][txt_type])\n            mrpc_embed[key]['y'] = np.array(text_data['y'])\n            logging.info('Computed {0} embeddings'.format(key))\n\n        # Train\n        trainA = mrpc_embed['train']['A']\n        trainB = mrpc_embed['train']['B']\n        trainF = np.c_[np.abs(trainA - trainB), trainA * trainB]\n        trainY = mrpc_embed['train']['y']\n\n        # Test\n        testA = mrpc_embed['test']['A']\n        testB = mrpc_embed['test']['B']\n        testF = np.c_[np.abs(testA - testB), testA * testB]\n        testY = mrpc_embed['test']['y']\n\n        config = {'nclasses': 2, 'seed': self.seed,\n                  'usepytorch': params.usepytorch,\n                  'classifier': params.classifier,\n                  'nhid': params.nhid, 'kfold': params.kfold}\n        clf = KFoldClassifier(train={'X': trainF, 'y': trainY},\n                              test={'X': testF, 'y': testY}, config=config)\n\n        devacc, testacc, yhat = clf.run()\n        testf1 = round(100*f1_score(testY, yhat), 2)\n        logging.debug('Dev acc : {0} Test acc {1}; Test F1 {2} for MRPC.\\n'\n                      .format(devacc, testacc, testf1))\n        return {'devacc': devacc, 'acc': testacc, 'f1': testf1,\n                'ndev': len(trainA), 'ntest': len(testA)}\n",
    "SentEval/senteval/probing.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nprobing tasks\n'''\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport io\nimport copy\nimport logging\nimport numpy as np\n\nfrom senteval.tools.validation import SplitClassifier\n\n\nclass PROBINGEval(object):\n    def __init__(self, task, task_path, seed=1111):\n        self.seed = seed\n        self.task = task\n        logging.debug('***** (Probing) Transfer task : %s classification *****', self.task.upper())\n        self.task_data = {'train': {'X': [], 'y': []},\n                          'dev': {'X': [], 'y': []},\n                          'test': {'X': [], 'y': []}}\n        self.loadFile(task_path)\n        logging.info('Loaded %s train - %s dev - %s test for %s' %\n                     (len(self.task_data['train']['y']), len(self.task_data['dev']['y']),\n                      len(self.task_data['test']['y']), self.task))\n\n    def do_prepare(self, params, prepare):\n        samples = self.task_data['train']['X'] + self.task_data['dev']['X'] + \\\n                  self.task_data['test']['X']\n        return prepare(params, samples)\n\n    def loadFile(self, fpath):\n        self.tok2split = {'tr': 'train', 'va': 'dev', 'te': 'test'}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                line = line.rstrip().split('\\t')\n                self.task_data[self.tok2split[line[0]]]['X'].append(line[-1].split())\n                self.task_data[self.tok2split[line[0]]]['y'].append(line[1])\n\n        labels = sorted(np.unique(self.task_data['train']['y']))\n        self.tok2label = dict(zip(labels, range(len(labels))))\n        self.nclasses = len(self.tok2label)\n\n        for split in self.task_data:\n            for i, y in enumerate(self.task_data[split]['y']):\n                self.task_data[split]['y'][i] = self.tok2label[y]\n\n    def run(self, params, batcher):\n        task_embed = {'train': {}, 'dev': {}, 'test': {}}\n        bsize = params.batch_size\n        logging.info('Computing embeddings for train/dev/test')\n        for key in self.task_data:\n            # Sort to reduce padding\n            sorted_data = sorted(zip(self.task_data[key]['X'],\n                                     self.task_data[key]['y']),\n                                 key=lambda z: (len(z[0]), z[1]))\n            self.task_data[key]['X'], self.task_data[key]['y'] = map(list, zip(*sorted_data))\n\n            task_embed[key]['X'] = []\n            for ii in range(0, len(self.task_data[key]['y']), bsize):\n                batch = self.task_data[key]['X'][ii:ii + bsize]\n                embeddings = batcher(params, batch)\n                task_embed[key]['X'].append(embeddings)\n            task_embed[key]['X'] = np.vstack(task_embed[key]['X'])\n            task_embed[key]['y'] = np.array(self.task_data[key]['y'])\n        logging.info('Computed embeddings')\n\n        config_classifier = {'nclasses': self.nclasses, 'seed': self.seed,\n                             'usepytorch': params.usepytorch,\n                             'classifier': params.classifier}\n\n        if self.task == \"WordContent\" and params.classifier['nhid'] > 0:\n            config_classifier = copy.deepcopy(config_classifier)\n            config_classifier['classifier']['nhid'] = 0\n            print(params.classifier['nhid'])\n\n        clf = SplitClassifier(X={'train': task_embed['train']['X'],\n                                 'valid': task_embed['dev']['X'],\n                                 'test': task_embed['test']['X']},\n                              y={'train': task_embed['train']['y'],\n                                 'valid': task_embed['dev']['y'],\n                                 'test': task_embed['test']['y']},\n                              config=config_classifier)\n\n        devacc, testacc = clf.run()\n        logging.debug('\\nDev acc : %.1f Test acc : %.1f for %s classification\\n' % (devacc, testacc, self.task.upper()))\n\n        return {'devacc': devacc, 'acc': testacc,\n                'ndev': len(task_embed['dev']['X']),\n                'ntest': len(task_embed['test']['X'])}\n\n\"\"\"\nSurface Information\n\"\"\"\nclass LengthEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'sentence_length.txt')\n        # labels: bins\n        PROBINGEval.__init__(self, 'Length', task_path, seed)\n\nclass WordContentEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'word_content.txt')\n        # labels: 200 target words\n        PROBINGEval.__init__(self, 'WordContent', task_path, seed)\n\n\"\"\"\nLatent Structural Information\n\"\"\"\nclass DepthEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'tree_depth.txt')\n        # labels: bins\n        PROBINGEval.__init__(self, 'Depth', task_path, seed)\n\nclass TopConstituentsEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'top_constituents.txt')\n        # labels: 'PP_NP_VP_.' .. (20 classes)\n        PROBINGEval.__init__(self, 'TopConstituents', task_path, seed)\n\nclass BigramShiftEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'bigram_shift.txt')\n        # labels: 0 or 1\n        PROBINGEval.__init__(self, 'BigramShift', task_path, seed)\n\n# TODO: Voice?\n\n\"\"\"\nLatent Semantic Information\n\"\"\"\n\nclass TenseEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'past_present.txt')\n        # labels: 'PRES', 'PAST'\n        PROBINGEval.__init__(self, 'Tense', task_path, seed)\n\nclass SubjNumberEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'subj_number.txt')\n        # labels: 'NN', 'NNS'\n        PROBINGEval.__init__(self, 'SubjNumber', task_path, seed)\n\nclass ObjNumberEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'obj_number.txt')\n        # labels: 'NN', 'NNS'\n        PROBINGEval.__init__(self, 'ObjNumber', task_path, seed)\n\nclass OddManOutEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'odd_man_out.txt')\n        # labels: 'O', 'C'\n        PROBINGEval.__init__(self, 'OddManOut', task_path, seed)\n\nclass CoordinationInversionEval(PROBINGEval):\n    def __init__(self, task_path, seed=1111):\n        task_path = os.path.join(task_path, 'coordination_inversion.txt')\n        # labels: 'O', 'I'\n        PROBINGEval.__init__(self, 'CoordinationInversion', task_path, seed)\n",
    "SentEval/senteval/rank.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nImage-Caption Retrieval with COCO dataset\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport sys\nimport logging\nimport numpy as np\n\ntry:\n    import cPickle as pickle\nexcept ImportError:\n    import pickle\n\nfrom senteval.tools.ranking import ImageSentenceRankingPytorch\n\n\nclass ImageCaptionRetrievalEval(object):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task: Image Caption Retrieval *****\\n\\n')\n\n        # Get captions and image features\n        self.seed = seed\n        train, dev, test = self.loadFile(task_path)\n        self.coco_data = {'train': train, 'dev': dev, 'test': test}\n\n    def do_prepare(self, params, prepare):\n        samples = self.coco_data['train']['sent'] + \\\n                  self.coco_data['dev']['sent'] + \\\n                  self.coco_data['test']['sent']\n        prepare(params, samples)\n\n    def loadFile(self, fpath):\n        coco = {}\n\n        for split in ['train', 'valid', 'test']:\n            list_sent = []\n            list_img_feat = []\n            if sys.version_info < (3, 0):\n                with open(os.path.join(fpath, split + '.pkl')) as f:\n                    cocodata = pickle.load(f)\n            else:\n                with open(os.path.join(fpath, split + '.pkl'), 'rb') as f:\n                    cocodata = pickle.load(f, encoding='latin1')\n\n            for imgkey in range(len(cocodata['features'])):\n                assert len(cocodata['image_to_caption_ids'][imgkey]) >= 5, \\\n                       cocodata['image_to_caption_ids'][imgkey]\n                for captkey in cocodata['image_to_caption_ids'][imgkey][0:5]:\n                    sent = cocodata['captions'][captkey]['cleaned_caption']\n                    sent += ' .'  # add punctuation to end of sentence in COCO\n                    list_sent.append(sent.encode('utf-8').split())\n                    list_img_feat.append(cocodata['features'][imgkey])\n            assert len(list_sent) == len(list_img_feat) and \\\n                len(list_sent) % 5 == 0\n            list_img_feat = np.array(list_img_feat).astype('float32')\n            coco[split] = {'sent': list_sent, 'imgfeat': list_img_feat}\n        return coco['train'], coco['valid'], coco['test']\n\n    def run(self, params, batcher):\n        coco_embed = {'train': {'sentfeat': [], 'imgfeat': []},\n                      'dev': {'sentfeat': [], 'imgfeat': []},\n                      'test': {'sentfeat': [], 'imgfeat': []}}\n\n        for key in self.coco_data:\n            logging.info('Computing embedding for {0}'.format(key))\n            # Sort to reduce padding\n            self.coco_data[key]['sent'] = np.array(self.coco_data[key]['sent'])\n            self.coco_data[key]['sent'], idx_sort = np.sort(self.coco_data[key]['sent']), np.argsort(self.coco_data[key]['sent'])\n            idx_unsort = np.argsort(idx_sort)\n\n            coco_embed[key]['X'] = []\n            nsent = len(self.coco_data[key]['sent'])\n            for ii in range(0, nsent, params.batch_size):\n                batch = self.coco_data[key]['sent'][ii:ii + params.batch_size]\n                embeddings = batcher(params, batch)\n                coco_embed[key]['sentfeat'].append(embeddings)\n            coco_embed[key]['sentfeat'] = np.vstack(coco_embed[key]['sentfeat'])[idx_unsort]\n            coco_embed[key]['imgfeat'] = np.array(self.coco_data[key]['imgfeat'])\n            logging.info('Computed {0} embeddings'.format(key))\n\n        config = {'seed': self.seed, 'projdim': 1000, 'margin': 0.2}\n        clf = ImageSentenceRankingPytorch(train=coco_embed['train'],\n                                          valid=coco_embed['dev'],\n                                          test=coco_embed['test'],\n                                          config=config)\n\n        bestdevscore, r1_i2t, r5_i2t, r10_i2t, medr_i2t, \\\n            r1_t2i, r5_t2i, r10_t2i, medr_t2i = clf.run()\n\n        logging.debug(\"\\nTest scores | Image to text: \\\n            {0}, {1}, {2}, {3}\".format(r1_i2t, r5_i2t, r10_i2t, medr_i2t))\n        logging.debug(\"Test scores | Text to image: \\\n            {0}, {1}, {2}, {3}\\n\".format(r1_t2i, r5_t2i, r10_t2i, medr_t2i))\n\n        return {'devacc': bestdevscore,\n                'acc': [(r1_i2t, r5_i2t, r10_i2t, medr_i2t),\n                        (r1_t2i, r5_t2i, r10_t2i, medr_t2i)],\n                'ndev': len(coco_embed['dev']['sentfeat']),\n                'ntest': len(coco_embed['test']['sentfeat'])}\n",
    "SentEval/senteval/sick.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nSICK Relatedness and Entailment\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport io\nimport logging\nimport numpy as np\n\nfrom sklearn.metrics import mean_squared_error\nfrom scipy.stats import pearsonr, spearmanr\n\nfrom senteval.tools.relatedness import RelatednessPytorch\nfrom senteval.tools.validation import SplitClassifier\n\nclass SICKEval(object):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : SICK-Relatedness*****\\n\\n')\n        self.seed = seed\n        train = self.loadFile(os.path.join(task_path, 'SICK_train.txt'))\n        dev = self.loadFile(os.path.join(task_path, 'SICK_trial.txt'))\n        test = self.loadFile(os.path.join(task_path, 'SICK_test_annotated.txt'))\n        self.sick_data = {'train': train, 'dev': dev, 'test': test}\n\n    def do_prepare(self, params, prepare):\n        samples = self.sick_data['train']['X_A'] + \\\n                  self.sick_data['train']['X_B'] + \\\n                  self.sick_data['dev']['X_A'] + \\\n                  self.sick_data['dev']['X_B'] + \\\n                  self.sick_data['test']['X_A'] + self.sick_data['test']['X_B']\n        return prepare(params, samples)\n\n    def loadFile(self, fpath):\n        skipFirstLine = True\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                if skipFirstLine:\n                    skipFirstLine = False\n                else:\n                    text = line.strip().split('\\t')\n                    sick_data['X_A'].append(text[1].split())\n                    sick_data['X_B'].append(text[2].split())\n                    sick_data['y'].append(text[3])\n\n        sick_data['y'] = [float(s) for s in sick_data['y']]\n        return sick_data\n\n    def run(self, params, batcher):\n        sick_embed = {'train': {}, 'dev': {}, 'test': {}}\n        bsize = params.batch_size\n\n        for key in self.sick_data:\n            logging.info('Computing embedding for {0}'.format(key))\n            # Sort to reduce padding\n            sorted_corpus = sorted(zip(self.sick_data[key]['X_A'],\n                                       self.sick_data[key]['X_B'],\n                                       self.sick_data[key]['y']),\n                                   key=lambda z: (len(z[0]), len(z[1]), z[2]))\n\n            self.sick_data[key]['X_A'] = [x for (x, y, z) in sorted_corpus]\n            self.sick_data[key]['X_B'] = [y for (x, y, z) in sorted_corpus]\n            self.sick_data[key]['y'] = [z for (x, y, z) in sorted_corpus]\n\n            for txt_type in ['X_A', 'X_B']:\n                sick_embed[key][txt_type] = []\n                for ii in range(0, len(self.sick_data[key]['y']), bsize):\n                    batch = self.sick_data[key][txt_type][ii:ii + bsize]\n                    embeddings = batcher(params, batch)\n                    sick_embed[key][txt_type].append(embeddings)\n                sick_embed[key][txt_type] = np.vstack(sick_embed[key][txt_type])\n            sick_embed[key]['y'] = np.array(self.sick_data[key]['y'])\n            logging.info('Computed {0} embeddings'.format(key))\n\n        # Train\n        trainA = sick_embed['train']['X_A']\n        trainB = sick_embed['train']['X_B']\n        trainF = np.c_[np.abs(trainA - trainB), trainA * trainB]\n        trainY = self.encode_labels(self.sick_data['train']['y'])\n\n        # Dev\n        devA = sick_embed['dev']['X_A']\n        devB = sick_embed['dev']['X_B']\n        devF = np.c_[np.abs(devA - devB), devA * devB]\n        devY = self.encode_labels(self.sick_data['dev']['y'])\n\n        # Test\n        testA = sick_embed['test']['X_A']\n        testB = sick_embed['test']['X_B']\n        testF = np.c_[np.abs(testA - testB), testA * testB]\n        testY = self.encode_labels(self.sick_data['test']['y'])\n\n        config = {'seed': self.seed, 'nclasses': 5}\n        clf = RelatednessPytorch(train={'X': trainF, 'y': trainY},\n                                 valid={'X': devF, 'y': devY},\n                                 test={'X': testF, 'y': testY},\n                                 devscores=self.sick_data['dev']['y'],\n                                 config=config)\n\n        devspr, yhat = clf.run()\n\n        pr = pearsonr(yhat, self.sick_data['test']['y'])[0]\n        sr = spearmanr(yhat, self.sick_data['test']['y'])[0]\n        pr = 0 if pr != pr else pr\n        sr = 0 if sr != sr else sr\n        se = mean_squared_error(yhat, self.sick_data['test']['y'])\n        logging.debug('Dev : Spearman {0}'.format(devspr))\n        logging.debug('Test : Pearson {0} Spearman {1} MSE {2} \\\n                       for SICK Relatedness\\n'.format(pr, sr, se))\n\n        return {'devspearman': devspr, 'pearson': pr, 'spearman': sr, 'mse': se,\n                'yhat': yhat, 'ndev': len(devA), 'ntest': len(testA)}\n\n    def encode_labels(self, labels, nclass=5):\n        \"\"\"\n        Label encoding from Tree LSTM paper (Tai, Socher, Manning)\n        \"\"\"\n        Y = np.zeros((len(labels), nclass)).astype('float32')\n        for j, y in enumerate(labels):\n            for i in range(nclass):\n                if i+1 == np.floor(y) + 1:\n                    Y[j, i] = y - np.floor(y)\n                if i+1 == np.floor(y):\n                    Y[j, i] = np.floor(y) - y + 1\n        return Y\n\n\nclass SICKEntailmentEval(SICKEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('***** Transfer task : SICK-Entailment*****\\n\\n')\n        self.seed = seed\n        train = self.loadFile(os.path.join(task_path, 'SICK_train.txt'))\n        dev = self.loadFile(os.path.join(task_path, 'SICK_trial.txt'))\n        test = self.loadFile(os.path.join(task_path, 'SICK_test_annotated.txt'))\n        self.sick_data = {'train': train, 'dev': dev, 'test': test}\n\n    def loadFile(self, fpath):\n        label2id = {'CONTRADICTION': 0, 'NEUTRAL': 1, 'ENTAILMENT': 2}\n        skipFirstLine = True\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                if skipFirstLine:\n                    skipFirstLine = False\n                else:\n                    text = line.strip().split('\\t')\n                    sick_data['X_A'].append(text[1].split())\n                    sick_data['X_B'].append(text[2].split())\n                    sick_data['y'].append(text[4])\n        sick_data['y'] = [label2id[s] for s in sick_data['y']]\n        return sick_data\n\n    def run(self, params, batcher):\n        sick_embed = {'train': {}, 'dev': {}, 'test': {}}\n        bsize = params.batch_size\n\n        for key in self.sick_data:\n            logging.info('Computing embedding for {0}'.format(key))\n            # Sort to reduce padding\n            sorted_corpus = sorted(zip(self.sick_data[key]['X_A'],\n                                       self.sick_data[key]['X_B'],\n                                       self.sick_data[key]['y']),\n                                   key=lambda z: (len(z[0]), len(z[1]), z[2]))\n\n            self.sick_data[key]['X_A'] = [x for (x, y, z) in sorted_corpus]\n            self.sick_data[key]['X_B'] = [y for (x, y, z) in sorted_corpus]\n            self.sick_data[key]['y'] = [z for (x, y, z) in sorted_corpus]\n\n            for txt_type in ['X_A', 'X_B']:\n                sick_embed[key][txt_type] = []\n                for ii in range(0, len(self.sick_data[key]['y']), bsize):\n                    batch = self.sick_data[key][txt_type][ii:ii + bsize]\n                    embeddings = batcher(params, batch)\n                    sick_embed[key][txt_type].append(embeddings)\n                sick_embed[key][txt_type] = np.vstack(sick_embed[key][txt_type])\n            logging.info('Computed {0} embeddings'.format(key))\n\n        # Train\n        trainA = sick_embed['train']['X_A']\n        trainB = sick_embed['train']['X_B']\n        trainF = np.c_[np.abs(trainA - trainB), trainA * trainB]\n        trainY = np.array(self.sick_data['train']['y'])\n\n        # Dev\n        devA = sick_embed['dev']['X_A']\n        devB = sick_embed['dev']['X_B']\n        devF = np.c_[np.abs(devA - devB), devA * devB]\n        devY = np.array(self.sick_data['dev']['y'])\n\n        # Test\n        testA = sick_embed['test']['X_A']\n        testB = sick_embed['test']['X_B']\n        testF = np.c_[np.abs(testA - testB), testA * testB]\n        testY = np.array(self.sick_data['test']['y'])\n\n        config = {'nclasses': 3, 'seed': self.seed,\n                  'usepytorch': params.usepytorch,\n                  'classifier': params.classifier,\n                  'nhid': params.nhid}\n        clf = SplitClassifier(X={'train': trainF, 'valid': devF, 'test': testF},\n                              y={'train': trainY, 'valid': devY, 'test': testY},\n                              config=config)\n\n        devacc, testacc = clf.run()\n        logging.debug('\\nDev acc : {0} Test acc : {1} for \\\n                       SICK entailment\\n'.format(devacc, testacc))\n        return {'devacc': devacc, 'acc': testacc,\n                'ndev': len(devA), 'ntest': len(testA)}\n",
    "SentEval/senteval/snli.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nSNLI - Entailment\n'''\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport codecs\nimport os\nimport io\nimport copy\nimport logging\nimport numpy as np\n\nfrom senteval.tools.validation import SplitClassifier\n\n\nclass SNLIEval(object):\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : SNLI Entailment*****\\n\\n')\n        self.seed = seed\n        train1 = self.loadFile(os.path.join(taskpath, 's1.train'))\n        train2 = self.loadFile(os.path.join(taskpath, 's2.train'))\n\n        trainlabels = io.open(os.path.join(taskpath, 'labels.train'),\n                              encoding='utf-8').read().splitlines()\n\n        valid1 = self.loadFile(os.path.join(taskpath, 's1.dev'))\n        valid2 = self.loadFile(os.path.join(taskpath, 's2.dev'))\n        validlabels = io.open(os.path.join(taskpath, 'labels.dev'),\n                              encoding='utf-8').read().splitlines()\n\n        test1 = self.loadFile(os.path.join(taskpath, 's1.test'))\n        test2 = self.loadFile(os.path.join(taskpath, 's2.test'))\n        testlabels = io.open(os.path.join(taskpath, 'labels.test'),\n                             encoding='utf-8').read().splitlines()\n\n        # sort data (by s2 first) to reduce padding\n        sorted_train = sorted(zip(train2, train1, trainlabels),\n                              key=lambda z: (len(z[0]), len(z[1]), z[2]))\n        train2, train1, trainlabels = map(list, zip(*sorted_train))\n\n        sorted_valid = sorted(zip(valid2, valid1, validlabels),\n                              key=lambda z: (len(z[0]), len(z[1]), z[2]))\n        valid2, valid1, validlabels = map(list, zip(*sorted_valid))\n\n        sorted_test = sorted(zip(test2, test1, testlabels),\n                             key=lambda z: (len(z[0]), len(z[1]), z[2]))\n        test2, test1, testlabels = map(list, zip(*sorted_test))\n\n        self.samples = train1 + train2 + valid1 + valid2 + test1 + test2\n        self.data = {'train': (train1, train2, trainlabels),\n                     'valid': (valid1, valid2, validlabels),\n                     'test': (test1, test2, testlabels)\n                     }\n\n    def do_prepare(self, params, prepare):\n        return prepare(params, self.samples)\n\n    def loadFile(self, fpath):\n        with codecs.open(fpath, 'rb', 'latin-1') as f:\n            return [line.split() for line in\n                    f.read().splitlines()]\n\n    def run(self, params, batcher):\n        self.X, self.y = {}, {}\n        dico_label = {'entailment': 0,  'neutral': 1, 'contradiction': 2}\n        for key in self.data:\n            if key not in self.X:\n                self.X[key] = []\n            if key not in self.y:\n                self.y[key] = []\n\n            input1, input2, mylabels = self.data[key]\n            enc_input = []\n            n_labels = len(mylabels)\n            for ii in range(0, n_labels, params.batch_size):\n                batch1 = input1[ii:ii + params.batch_size]\n                batch2 = input2[ii:ii + params.batch_size]\n\n                if len(batch1) == len(batch2) and len(batch1) > 0:\n                    enc1 = batcher(params, batch1)\n                    enc2 = batcher(params, batch2)\n                    enc_input.append(np.hstack((enc1, enc2, enc1 * enc2,\n                                                np.abs(enc1 - enc2))))\n                if (ii*params.batch_size) % (20000*params.batch_size) == 0:\n                    logging.info(\"PROGRESS (encoding): %.2f%%\" %\n                                 (100 * ii / n_labels))\n            self.X[key] = np.vstack(enc_input)\n            self.y[key] = [dico_label[y] for y in mylabels]\n\n        config = {'nclasses': 3, 'seed': self.seed,\n                  'usepytorch': params.usepytorch,\n                  'cudaEfficient': True,\n                  'nhid': params.nhid, 'noreg': True}\n\n        config_classifier = copy.deepcopy(params.classifier)\n        config_classifier['max_epoch'] = 15\n        config_classifier['epoch_size'] = 1\n        config['classifier'] = config_classifier\n\n        clf = SplitClassifier(self.X, self.y, config)\n        devacc, testacc = clf.run()\n        logging.debug('Dev acc : {0} Test acc : {1} for SNLI\\n'\n                      .format(devacc, testacc))\n        return {'devacc': devacc, 'acc': testacc,\n                'ndev': len(self.data['valid'][0]),\n                'ntest': len(self.data['test'][0])}\n",
    "SentEval/senteval/sst.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nSST - binary classification\n'''\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport io\nimport logging\nimport numpy as np\n\nfrom senteval.tools.validation import SplitClassifier\n\n\nclass SSTEval(object):\n    def __init__(self, task_path, nclasses=2, seed=1111):\n        self.seed = seed\n\n        # binary of fine-grained\n        assert nclasses in [2, 5]\n        self.nclasses = nclasses\n        self.task_name = 'Binary' if self.nclasses == 2 else 'Fine-Grained'\n        logging.debug('***** Transfer task : SST %s classification *****\\n\\n', self.task_name)\n\n        train = self.loadFile(os.path.join(task_path, 'sentiment-train'))\n        dev = self.loadFile(os.path.join(task_path, 'sentiment-dev'))\n        test = self.loadFile(os.path.join(task_path, 'sentiment-test'))\n        self.sst_data = {'train': train, 'dev': dev, 'test': test}\n\n    def do_prepare(self, params, prepare):\n        samples = self.sst_data['train']['X'] + self.sst_data['dev']['X'] + \\\n                  self.sst_data['test']['X']\n        return prepare(params, samples)\n\n    def loadFile(self, fpath):\n        sst_data = {'X': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                if self.nclasses == 2:\n                    sample = line.strip().split('\\t')\n                    sst_data['y'].append(int(sample[1]))\n                    sst_data['X'].append(sample[0].split())\n                elif self.nclasses == 5:\n                    sample = line.strip().split(' ', 1)\n                    sst_data['y'].append(int(sample[0]))\n                    sst_data['X'].append(sample[1].split())\n        assert max(sst_data['y']) == self.nclasses - 1\n        return sst_data\n\n    def run(self, params, batcher):\n        sst_embed = {'train': {}, 'dev': {}, 'test': {}}\n        bsize = params.batch_size\n\n        for key in self.sst_data:\n            logging.info('Computing embedding for {0}'.format(key))\n            # Sort to reduce padding\n            sorted_data = sorted(zip(self.sst_data[key]['X'],\n                                     self.sst_data[key]['y']),\n                                 key=lambda z: (len(z[0]), z[1]))\n            self.sst_data[key]['X'], self.sst_data[key]['y'] = map(list, zip(*sorted_data))\n\n            sst_embed[key]['X'] = []\n            for ii in range(0, len(self.sst_data[key]['y']), bsize):\n                batch = self.sst_data[key]['X'][ii:ii + bsize]\n                embeddings = batcher(params, batch)\n                sst_embed[key]['X'].append(embeddings)\n            sst_embed[key]['X'] = np.vstack(sst_embed[key]['X'])\n            sst_embed[key]['y'] = np.array(self.sst_data[key]['y'])\n            logging.info('Computed {0} embeddings'.format(key))\n\n        config_classifier = {'nclasses': self.nclasses, 'seed': self.seed,\n                             'usepytorch': params.usepytorch,\n                             'classifier': params.classifier}\n\n        clf = SplitClassifier(X={'train': sst_embed['train']['X'],\n                                 'valid': sst_embed['dev']['X'],\n                                 'test': sst_embed['test']['X']},\n                              y={'train': sst_embed['train']['y'],\n                                 'valid': sst_embed['dev']['y'],\n                                 'test': sst_embed['test']['y']},\n                              config=config_classifier)\n\n        devacc, testacc = clf.run()\n        logging.debug('\\nDev acc : {0} Test acc : {1} for \\\n            SST {2} classification\\n'.format(devacc, testacc, self.task_name))\n\n        return {'devacc': devacc, 'acc': testacc,\n                'ndev': len(sst_embed['dev']['X']),\n                'ntest': len(sst_embed['test']['X'])}\n",
    "SentEval/senteval/sts.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nSTS-{2012,2013,2014,2015,2016} (unsupervised) and\nSTS-benchmark (supervised) tasks\n'''\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nfrom tqdm import tqdm\nimport os\nimport io\nimport torch\nimport numpy as np\nimport logging\n\nfrom scipy.stats import spearmanr, pearsonr\n\nfrom senteval.utils import cosine\nfrom senteval.sick import SICKEval\n\nimport json\n\nclass STSEval(object):\n    def loadFile(self, fpath):\n        self.data = {}\n        self.samples = []\n\n        for dataset in self.datasets:\n            sent1, sent2 = zip(*[l.split(\"\\t\") for l in\n                               io.open(fpath + '/STS.input.%s.txt' % dataset,\n                                       encoding='utf8').read().splitlines()])\n            raw_scores = np.array([x for x in\n                                   io.open(fpath + '/STS.gs.%s.txt' % dataset,\n                                           encoding='utf8')\n                                   .read().splitlines()])\n            not_empty_idx = raw_scores != ''\n\n            gs_scores = [float(x) for x in raw_scores[not_empty_idx]]\n            sent1 = np.array([s.split() for s in sent1],dtype=object)[not_empty_idx]\n            sent2 = np.array([s.split() for s in sent2],dtype=object)[not_empty_idx]\n            # sort data by length to minimize padding in batcher\n            sorted_data = sorted(zip(sent1, sent2, gs_scores),\n                                 key=lambda z: (len(z[0]), len(z[1]), z[2]))\n            sent1, sent2, gs_scores = map(list, zip(*sorted_data))\n\n            self.data[dataset] = (sent1, sent2, gs_scores)\n            self.samples += sent1 + sent2\n\n    def do_prepare(self, params, prepare):\n        if 'similarity' in params:\n            self.similarity = params.similarity\n        else:  # Default similarity is cosine\n            self.similarity = lambda s1, s2: np.nan_to_num(cosine(np.nan_to_num(s1), np.nan_to_num(s2)))\n        return prepare(params, self.samples)\n\n    def run(self, params, batcher):\n        results = {}\n        all_sys_scores = []\n        all_gs_scores = []\n\n        for dataset in self.datasets:\n            sys_scores = []\n            input1, input2, gs_scores = self.data[dataset]\n            for ii in tqdm(range(0, len(gs_scores), params.batch_size), desc=f\"Evaluating\"):\n                batch1 = input1[ii:ii + params.batch_size]\n                batch2 = input2[ii:ii + params.batch_size]\n\n                # we assume get_batch already throws out the faulty ones\n                if len(batch1) == len(batch2) and len(batch1) > 0:\n                    enc1 = batcher(params, batch1)\n                    enc2 = batcher(params, batch2)\n\n                    for kk in range(enc2.shape[0]):\n                        sys_score = self.similarity(enc1[kk], enc2[kk])\n                        sys_scores.append(sys_score)\n            all_sys_scores.extend(sys_scores)\n            all_gs_scores.extend(gs_scores)\n            results[dataset] = {'pearson': pearsonr(sys_scores, gs_scores),\n                                'spearman': spearmanr(sys_scores, gs_scores),\n                                'nsamples': len(sys_scores)}\n            logging.debug('%s : pearson = %.4f, spearman = %.4f' %\n                          (dataset, results[dataset]['pearson'][0],\n                           results[dataset]['spearman'][0]))\n        \n        weights = [results[dset]['nsamples'] for dset in results.keys()]\n        list_prs = np.array([results[dset]['pearson'][0] for\n                            dset in results.keys()])\n        list_spr = np.array([results[dset]['spearman'][0] for\n                            dset in results.keys()])\n\n        avg_pearson = np.average(list_prs)\n        avg_spearman = np.average(list_spr)\n        wavg_pearson = np.average(list_prs, weights=weights)\n        wavg_spearman = np.average(list_spr, weights=weights)\n        all_pearson = pearsonr(all_sys_scores, all_gs_scores)\n        all_spearman = spearmanr(all_sys_scores, all_gs_scores)\n        results['all'] = {'pearson': {'all': all_pearson[0],\n                                      'mean': avg_pearson,\n                                      'wmean': wavg_pearson},\n                          'spearman': {'all': all_spearman[0],\n                                       'mean': avg_spearman,\n                                       'wmean': wavg_spearman}}\n        logging.debug('ALL : Pearson = %.4f, \\\n            Spearman = %.4f' % (all_pearson[0], all_spearman[0]))\n        logging.debug('ALL (weighted average) : Pearson = %.4f, \\\n            Spearman = %.4f' % (wavg_pearson, wavg_spearman))\n        logging.debug('ALL (average) : Pearson = %.4f, \\\n            Spearman = %.4f\\n' % (avg_pearson, avg_spearman))\n\n        return results\n\n\nclass STS12Eval(STSEval):\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : STS12 *****\\n\\n')\n        self.seed = seed\n        self.datasets = ['MSRpar', 'MSRvid', 'SMTeuroparl',\n                         'surprise.OnWN', 'surprise.SMTnews']\n        self.loadFile(taskpath)\n\n\nclass STS13Eval(STSEval):\n    # STS13 here does not contain the \"SMT\" subtask due to LICENSE issue\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : STS13 (-SMT) *****\\n\\n')\n        self.seed = seed\n        self.datasets = ['FNWN', 'headlines', 'OnWN']\n        self.loadFile(taskpath)\n\n\nclass STS14Eval(STSEval):\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : STS14 *****\\n\\n')\n        self.seed = seed\n        self.datasets = ['deft-forum', 'deft-news', 'headlines',\n                         'images', 'OnWN', 'tweet-news']\n        self.loadFile(taskpath)\n\n\nclass STS15Eval(STSEval):\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : STS15 *****\\n\\n')\n        self.seed = seed\n        self.datasets = ['answers-forums', 'answers-students',\n                         'belief', 'headlines', 'images']\n        self.loadFile(taskpath)\n\n\nclass STS16Eval(STSEval):\n    def __init__(self, taskpath, seed=1111):\n        logging.debug('***** Transfer task : STS16 *****\\n\\n')\n        self.seed = seed\n        self.datasets = ['answer-answer', 'headlines', 'plagiarism',\n                         'postediting', 'question-question']\n        self.loadFile(taskpath)\n\n\nclass STSBenchmarkEval(STSEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('\\n\\n***** Transfer task : STSBenchmark*****\\n\\n')\n        self.seed = seed\n        self.samples = []\n        #train = self.loadFile(os.path.join(task_path, 'sts-train.csv'))\n        #dev = self.loadFile(os.path.join(task_path, 'sts-dev.csv'))\n        #test = self.loadFile(os.path.join(task_path, 'sts-test.csv'))\n        #self.datasets = ['train', 'dev', 'test']\n        #self.data = {'train': train, 'dev': dev, 'test': test}\n        test = self.loadFile(os.path.join(task_path, 'sts-test.csv'))\n        self.datasets = ['test']\n        self.data = {'test': test}\n\n    def loadFile(self, fpath):\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                text = line.strip().split('\\t')\n                sick_data['X_A'].append(text[5].split())\n                sick_data['X_B'].append(text[6].split())\n                sick_data['y'].append(text[4])\n\n        sick_data['y'] = [float(s) for s in sick_data['y']]\n        self.samples += sick_data['X_A'] + sick_data[\"X_B\"]\n        return (sick_data['X_A'], sick_data[\"X_B\"], sick_data['y'])\n\nclass STSBenchmarkEvalDev(STSEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('\\n\\n***** Transfer task : STSBenchmark*****\\n\\n')\n        self.seed = seed\n        self.samples = []\n        #train = self.loadFile(os.path.join(task_path, 'sts-train.csv'))\n        #dev = self.loadFile(os.path.join(task_path, 'sts-dev.csv'))\n        #test = self.loadFile(os.path.join(task_path, 'sts-test.csv'))\n        #self.datasets = ['train', 'dev', 'test']\n        #self.data = {'train': train, 'dev': dev, 'test': test}\n        dev = self.loadFile(os.path.join(task_path, 'sts-dev.csv'))\n        self.datasets = ['dev']\n        self.data = {'dev': dev}\n\n    def loadFile(self, fpath):\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                text = line.strip().split('\\t')\n                sick_data['X_A'].append(text[5].split())\n                sick_data['X_B'].append(text[6].split())\n                sick_data['y'].append(text[4])\n\n        sick_data['y'] = [float(s) for s in sick_data['y']]\n        self.samples += sick_data['X_A'] + sick_data[\"X_B\"]\n        return (sick_data['X_A'], sick_data[\"X_B\"], sick_data['y'])\n\nclass STSBenchmarkFinetune(SICKEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('\\n\\n***** Transfer task : STSBenchmark*****\\n\\n')\n        self.seed = seed\n        train = self.loadFile(os.path.join(task_path, 'sts-train.csv'))\n        dev = self.loadFile(os.path.join(task_path, 'sts-dev.csv'))\n        test = self.loadFile(os.path.join(task_path, 'sts-test.csv'))\n        self.sick_data = {'train': train, 'dev': dev, 'test': test}\n\n    def loadFile(self, fpath):\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                text = line.strip().split('\\t')\n                sick_data['X_A'].append(text[5].split())\n                sick_data['X_B'].append(text[6].split())\n                sick_data['y'].append(text[4])\n\n        sick_data['y'] = [float(s) for s in sick_data['y']]\n        return sick_data\n        \nclass SICKRelatednessEval(STSEval):\n    def __init__(self, task_path, seed=1111):\n        logging.debug('\\n\\n***** Transfer task : SICKRelatedness*****\\n\\n')\n        self.seed = seed\n        self.samples = []\n        #train = self.loadFile(os.path.join(task_path, 'SICK_train.txt'))\n        #dev = self.loadFile(os.path.join(task_path, 'SICK_trial.txt'))\n        #test = self.loadFile(os.path.join(task_path, 'SICK_test_annotated.txt'))\n        #self.datasets = ['train', 'dev', 'test']\n        #self.data = {'train': train, 'dev': dev, 'test': test}\n        test = self.loadFile(os.path.join(task_path, 'SICK_test_annotated.txt'))\n        self.datasets = ['test']\n        self.data = {'test': test}\n    \n    def loadFile(self, fpath):\n        skipFirstLine = True\n        sick_data = {'X_A': [], 'X_B': [], 'y': []}\n        with io.open(fpath, 'r', encoding='utf-8') as f:\n            for line in f:\n                if skipFirstLine:\n                    skipFirstLine = False\n                else:\n                    text = line.strip().split('\\t')\n                    sick_data['X_A'].append(text[1].split())\n                    sick_data['X_B'].append(text[2].split())\n                    sick_data['y'].append(text[3])\n\n        sick_data['y'] = [float(s) for s in sick_data['y']]\n        self.samples += sick_data['X_A'] + sick_data[\"X_B\"]\n        return (sick_data['X_A'], sick_data[\"X_B\"], sick_data['y'])\n",
    "SentEval/senteval/tools/__init__.py": "",
    "SentEval/senteval/tools/classifier.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n\"\"\"\nPytorch Classifier class in the style of scikit-learn\nClassifiers include Logistic Regression and MLP\n\"\"\"\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport numpy as np\nimport copy\nfrom senteval import utils\n\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\n\nclass PyTorchClassifier(object):\n    def __init__(self, inputdim, nclasses, l2reg=0., batch_size=64, seed=1111,\n                 cudaEfficient=False):\n        # fix seed\n        np.random.seed(seed)\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed(seed)\n\n        self.inputdim = inputdim\n        self.nclasses = nclasses\n        self.l2reg = l2reg\n        self.batch_size = batch_size\n        self.cudaEfficient = cudaEfficient\n\n    def prepare_split(self, X, y, validation_data=None, validation_split=None):\n        # Preparing validation data\n        assert validation_split or validation_data\n        if validation_data is not None:\n            trainX, trainy = X, y\n            devX, devy = validation_data\n        else:\n            permutation = np.random.permutation(len(X))\n            trainidx = permutation[int(validation_split * len(X)):]\n            devidx = permutation[0:int(validation_split * len(X))]\n            trainX, trainy = X[trainidx], y[trainidx]\n            devX, devy = X[devidx], y[devidx]\n\n        device = torch.device('cpu') if self.cudaEfficient else torch.device('cuda')\n\n        trainX = torch.from_numpy(trainX).to(device, dtype=torch.float32)\n        trainy = torch.from_numpy(trainy).to(device, dtype=torch.int64)\n        devX = torch.from_numpy(devX).to(device, dtype=torch.float32)\n        devy = torch.from_numpy(devy).to(device, dtype=torch.int64)\n\n        return trainX, trainy, devX, devy\n\n    def fit(self, X, y, validation_data=None, validation_split=None,\n            early_stop=True):\n        self.nepoch = 0\n        bestaccuracy = -1\n        stop_train = False\n        early_stop_count = 0\n\n        # Preparing validation data\n        trainX, trainy, devX, devy = self.prepare_split(X, y, validation_data,\n                                                        validation_split)\n\n        # Training\n        while not stop_train and self.nepoch <= self.max_epoch:\n            self.trainepoch(trainX, trainy, epoch_size=self.epoch_size)\n            accuracy = self.score(devX, devy)\n            if accuracy > bestaccuracy:\n                bestaccuracy = accuracy\n                bestmodel = copy.deepcopy(self.model)\n            elif early_stop:\n                if early_stop_count >= self.tenacity:\n                    stop_train = True\n                early_stop_count += 1\n        self.model = bestmodel\n        return bestaccuracy\n\n    def trainepoch(self, X, y, epoch_size=1):\n        self.model.train()\n        for _ in range(self.nepoch, self.nepoch + epoch_size):\n            permutation = np.random.permutation(len(X))\n            all_costs = []\n            for i in range(0, len(X), self.batch_size):\n                # forward\n                idx = torch.from_numpy(permutation[i:i + self.batch_size]).long().to(X.device)\n\n                Xbatch = X[idx]\n                ybatch = y[idx]\n\n                if self.cudaEfficient:\n                    Xbatch = Xbatch.cuda()\n                    ybatch = ybatch.cuda()\n                output = self.model(Xbatch)\n                # loss\n                loss = self.loss_fn(output, ybatch)\n                all_costs.append(loss.data.item())\n                # backward\n                self.optimizer.zero_grad()\n                loss.backward()\n                # Update parameters\n                self.optimizer.step()\n        self.nepoch += epoch_size\n\n    def score(self, devX, devy):\n        self.model.eval()\n        correct = 0\n        if not isinstance(devX, torch.cuda.FloatTensor) or self.cudaEfficient:\n            devX = torch.FloatTensor(devX).cuda()\n            devy = torch.LongTensor(devy).cuda()\n        with torch.no_grad():\n            for i in range(0, len(devX), self.batch_size):\n                Xbatch = devX[i:i + self.batch_size]\n                ybatch = devy[i:i + self.batch_size]\n                if self.cudaEfficient:\n                    Xbatch = Xbatch.cuda()\n                    ybatch = ybatch.cuda()\n                output = self.model(Xbatch)\n                pred = output.data.max(1)[1]\n                correct += pred.long().eq(ybatch.data.long()).sum().item()\n            accuracy = 1.0 * correct / len(devX)\n        return accuracy\n\n    def predict(self, devX):\n        self.model.eval()\n        if not isinstance(devX, torch.cuda.FloatTensor):\n            devX = torch.FloatTensor(devX).cuda()\n        yhat = np.array([])\n        with torch.no_grad():\n            for i in range(0, len(devX), self.batch_size):\n                Xbatch = devX[i:i + self.batch_size]\n                output = self.model(Xbatch)\n                yhat = np.append(yhat,\n                                 output.data.max(1)[1].cpu().numpy())\n        yhat = np.vstack(yhat)\n        return yhat\n\n    def predict_proba(self, devX):\n        self.model.eval()\n        probas = []\n        with torch.no_grad():\n            for i in range(0, len(devX), self.batch_size):\n                Xbatch = devX[i:i + self.batch_size]\n                vals = F.softmax(self.model(Xbatch).data.cpu().numpy())\n                if not probas:\n                    probas = vals\n                else:\n                    probas = np.concatenate(probas, vals, axis=0)\n        return probas\n\n\n\"\"\"\nMLP with Pytorch (nhid=0 --> Logistic Regression)\n\"\"\"\n\nclass MLP(PyTorchClassifier):\n    def __init__(self, params, inputdim, nclasses, l2reg=0., batch_size=64,\n                 seed=1111, cudaEfficient=False):\n        super(self.__class__, self).__init__(inputdim, nclasses, l2reg,\n                                             batch_size, seed, cudaEfficient)\n        \"\"\"\n        PARAMETERS:\n        -nhid:       number of hidden units (0: Logistic Regression)\n        -optim:      optimizer (\"sgd,lr=0.1\", \"adam\", \"rmsprop\" ..)\n        -tenacity:   how many times dev acc does not increase before stopping\n        -epoch_size: each epoch corresponds to epoch_size pass on the train set\n        -max_epoch:  max number of epoches\n        -dropout:    dropout for MLP\n        \"\"\"\n\n        self.nhid = 0 if \"nhid\" not in params else params[\"nhid\"]\n        self.optim = \"adam\" if \"optim\" not in params else params[\"optim\"]\n        self.tenacity = 5 if \"tenacity\" not in params else params[\"tenacity\"]\n        self.epoch_size = 4 if \"epoch_size\" not in params else params[\"epoch_size\"]\n        self.max_epoch = 200 if \"max_epoch\" not in params else params[\"max_epoch\"]\n        self.dropout = 0. if \"dropout\" not in params else params[\"dropout\"]\n        self.batch_size = 64 if \"batch_size\" not in params else params[\"batch_size\"]\n\n        if params[\"nhid\"] == 0:\n            self.model = nn.Sequential(\n                nn.Linear(self.inputdim, self.nclasses),\n            ).cuda()\n        else:\n            self.model = nn.Sequential(\n                nn.Linear(self.inputdim, params[\"nhid\"]),\n                nn.Dropout(p=self.dropout),\n                nn.Sigmoid(),\n                nn.Linear(params[\"nhid\"], self.nclasses),\n            ).cuda()\n\n        self.loss_fn = nn.CrossEntropyLoss().cuda()\n        self.loss_fn.size_average = False\n\n        optim_fn, optim_params = utils.get_optimizer(self.optim)\n        self.optimizer = optim_fn(self.model.parameters(), **optim_params)\n        self.optimizer.param_groups[0]['weight_decay'] = self.l2reg\n",
    "SentEval/senteval/tools/ranking.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n\"\"\"\nImage Annotation/Search for COCO with Pytorch\n\"\"\"\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport logging\nimport copy\nimport numpy as np\n\nimport torch\nfrom torch import nn\nfrom torch.autograd import Variable\nimport torch.optim as optim\n\n\nclass COCOProjNet(nn.Module):\n    def __init__(self, config):\n        super(COCOProjNet, self).__init__()\n        self.imgdim = config['imgdim']\n        self.sentdim = config['sentdim']\n        self.projdim = config['projdim']\n        self.imgproj = nn.Sequential(\n                        nn.Linear(self.imgdim, self.projdim),\n                        )\n        self.sentproj = nn.Sequential(\n                        nn.Linear(self.sentdim, self.projdim),\n                        )\n\n    def forward(self, img, sent, imgc, sentc):\n        # imgc : (bsize, ncontrast, imgdim)\n        # sentc : (bsize, ncontrast, sentdim)\n        # img : (bsize, imgdim)\n        # sent : (bsize, sentdim)\n        img = img.unsqueeze(1).expand_as(imgc).contiguous()\n        img = img.view(-1, self.imgdim)\n        imgc = imgc.view(-1, self.imgdim)\n        sent = sent.unsqueeze(1).expand_as(sentc).contiguous()\n        sent = sent.view(-1, self.sentdim)\n        sentc = sentc.view(-1, self.sentdim)\n\n        imgproj = self.imgproj(img)\n        imgproj = imgproj / torch.sqrt(torch.pow(imgproj, 2).sum(1, keepdim=True)).expand_as(imgproj)\n        imgcproj = self.imgproj(imgc)\n        imgcproj = imgcproj / torch.sqrt(torch.pow(imgcproj, 2).sum(1, keepdim=True)).expand_as(imgcproj)\n        sentproj = self.sentproj(sent)\n        sentproj = sentproj / torch.sqrt(torch.pow(sentproj, 2).sum(1, keepdim=True)).expand_as(sentproj)\n        sentcproj = self.sentproj(sentc)\n        sentcproj = sentcproj / torch.sqrt(torch.pow(sentcproj, 2).sum(1, keepdim=True)).expand_as(sentcproj)\n        # (bsize*ncontrast, projdim)\n\n        anchor1 = torch.sum((imgproj*sentproj), 1)\n        anchor2 = torch.sum((sentproj*imgproj), 1)\n        img_sentc = torch.sum((imgproj*sentcproj), 1)\n        sent_imgc = torch.sum((sentproj*imgcproj), 1)\n\n        # (bsize*ncontrast)\n        return anchor1, anchor2, img_sentc, sent_imgc\n\n    def proj_sentence(self, sent):\n        output = self.sentproj(sent)\n        output = output / torch.sqrt(torch.pow(output, 2).sum(1, keepdim=True)).expand_as(output)\n        return output # (bsize, projdim)\n\n    def proj_image(self, img):\n        output = self.imgproj(img)\n        output = output / torch.sqrt(torch.pow(output, 2).sum(1, keepdim=True)).expand_as(output)\n        return output # (bsize, projdim)\n\n\nclass PairwiseRankingLoss(nn.Module):\n    \"\"\"\n    Pairwise ranking loss\n    \"\"\"\n    def __init__(self, margin):\n        super(PairwiseRankingLoss, self).__init__()\n        self.margin = margin\n\n    def forward(self, anchor1, anchor2, img_sentc, sent_imgc):\n\n        cost_sent = torch.clamp(self.margin - anchor1 + img_sentc,\n                                min=0.0).sum()\n        cost_img = torch.clamp(self.margin - anchor2 + sent_imgc,\n                               min=0.0).sum()\n        loss = cost_sent + cost_img\n        return loss\n\n\nclass ImageSentenceRankingPytorch(object):\n    # Image Sentence Ranking on COCO with Pytorch\n    def __init__(self, train, valid, test, config):\n        # fix seed\n        self.seed = config['seed']\n        np.random.seed(self.seed)\n        torch.manual_seed(self.seed)\n        torch.cuda.manual_seed(self.seed)\n\n        self.train = train\n        self.valid = valid\n        self.test = test\n\n        self.imgdim = len(train['imgfeat'][0])\n        self.sentdim = len(train['sentfeat'][0])\n        self.projdim = config['projdim']\n        self.margin = config['margin']\n\n        self.batch_size = 128\n        self.ncontrast = 30\n        self.maxepoch = 20\n        self.early_stop = True\n\n        config_model = {'imgdim': self.imgdim,'sentdim': self.sentdim,\n                        'projdim': self.projdim}\n        self.model = COCOProjNet(config_model).cuda()\n\n        self.loss_fn = PairwiseRankingLoss(margin=self.margin).cuda()\n\n        self.optimizer = optim.Adam(self.model.parameters())\n\n    def prepare_data(self, trainTxt, trainImg, devTxt, devImg,\n                     testTxt, testImg):\n        trainTxt = torch.FloatTensor(trainTxt)\n        trainImg = torch.FloatTensor(trainImg)\n        devTxt = torch.FloatTensor(devTxt).cuda()\n        devImg = torch.FloatTensor(devImg).cuda()\n        testTxt = torch.FloatTensor(testTxt).cuda()\n        testImg = torch.FloatTensor(testImg).cuda()\n\n        return trainTxt, trainImg, devTxt, devImg, testTxt, testImg\n\n    def run(self):\n        self.nepoch = 0\n        bestdevscore = -1\n        early_stop_count = 0\n        stop_train = False\n\n        # Preparing data\n        logging.info('prepare data')\n        trainTxt, trainImg, devTxt, devImg, testTxt, testImg = \\\n            self.prepare_data(self.train['sentfeat'], self.train['imgfeat'],\n                              self.valid['sentfeat'], self.valid['imgfeat'],\n                              self.test['sentfeat'], self.test['imgfeat'])\n\n        # Training\n        while not stop_train and self.nepoch <= self.maxepoch:\n            logging.info('start epoch')\n            self.trainepoch(trainTxt, trainImg, devTxt, devImg, nepoches=1)\n            logging.info('Epoch {0} finished'.format(self.nepoch))\n\n            results = {'i2t': {'r1': 0, 'r5': 0, 'r10': 0, 'medr': 0},\n                       't2i': {'r1': 0, 'r5': 0, 'r10': 0, 'medr': 0},\n                       'dev': bestdevscore}\n            score = 0\n            for i in range(5):\n                devTxt_i = devTxt[i*5000:(i+1)*5000]\n                devImg_i = devImg[i*5000:(i+1)*5000]\n                # Compute dev ranks img2txt\n                r1_i2t, r5_i2t, r10_i2t, medr_i2t = self.i2t(devImg_i,\n                                                             devTxt_i)\n                results['i2t']['r1'] += r1_i2t / 5\n                results['i2t']['r5'] += r5_i2t / 5\n                results['i2t']['r10'] += r10_i2t / 5\n                results['i2t']['medr'] += medr_i2t / 5\n                logging.info(\"Image to text: {0}, {1}, {2}, {3}\"\n                             .format(r1_i2t, r5_i2t, r10_i2t, medr_i2t))\n                # Compute dev ranks txt2img\n                r1_t2i, r5_t2i, r10_t2i, medr_t2i = self.t2i(devImg_i,\n                                                             devTxt_i)\n                results['t2i']['r1'] += r1_t2i / 5\n                results['t2i']['r5'] += r5_t2i / 5\n                results['t2i']['r10'] += r10_t2i / 5\n                results['t2i']['medr'] += medr_t2i / 5\n                logging.info(\"Text to Image: {0}, {1}, {2}, {3}\"\n                             .format(r1_t2i, r5_t2i, r10_t2i, medr_t2i))\n                score += (r1_i2t + r5_i2t + r10_i2t +\n                          r1_t2i + r5_t2i + r10_t2i) / 5\n\n            logging.info(\"Dev mean Text to Image: {0}, {1}, {2}, {3}\".format(\n                        results['t2i']['r1'], results['t2i']['r5'],\n                        results['t2i']['r10'], results['t2i']['medr']))\n            logging.info(\"Dev mean Image to text: {0}, {1}, {2}, {3}\".format(\n                        results['i2t']['r1'], results['i2t']['r5'],\n                        results['i2t']['r10'], results['i2t']['medr']))\n\n            # early stop on Pearson\n            if score > bestdevscore:\n                bestdevscore = score\n                bestmodel = copy.deepcopy(self.model)\n            elif self.early_stop:\n                if early_stop_count >= 3:\n                    stop_train = True\n                early_stop_count += 1\n        self.model = bestmodel\n\n        # Compute test for the 5 splits\n        results = {'i2t': {'r1': 0, 'r5': 0, 'r10': 0, 'medr': 0},\n                   't2i': {'r1': 0, 'r5': 0, 'r10': 0, 'medr': 0},\n                   'dev': bestdevscore}\n        for i in range(5):\n            testTxt_i = testTxt[i*5000:(i+1)*5000]\n            testImg_i = testImg[i*5000:(i+1)*5000]\n            # Compute test ranks img2txt\n            r1_i2t, r5_i2t, r10_i2t, medr_i2t = self.i2t(testImg_i, testTxt_i)\n            results['i2t']['r1'] += r1_i2t / 5\n            results['i2t']['r5'] += r5_i2t / 5\n            results['i2t']['r10'] += r10_i2t / 5\n            results['i2t']['medr'] += medr_i2t / 5\n            # Compute test ranks txt2img\n            r1_t2i, r5_t2i, r10_t2i, medr_t2i = self.t2i(testImg_i, testTxt_i)\n            results['t2i']['r1'] += r1_t2i / 5\n            results['t2i']['r5'] += r5_t2i / 5\n            results['t2i']['r10'] += r10_t2i / 5\n            results['t2i']['medr'] += medr_t2i / 5\n\n        return bestdevscore, results['i2t']['r1'], results['i2t']['r5'], \\\n                             results['i2t']['r10'], results['i2t']['medr'], \\\n                             results['t2i']['r1'], results['t2i']['r5'], \\\n                             results['t2i']['r10'], results['t2i']['medr']\n\n    def trainepoch(self, trainTxt, trainImg, devTxt, devImg, nepoches=1):\n        self.model.train()\n        for _ in range(self.nepoch, self.nepoch + nepoches):\n            permutation = list(np.random.permutation(len(trainTxt)))\n            all_costs = []\n            for i in range(0, len(trainTxt), self.batch_size):\n                # forward\n                if i % (self.batch_size*500) == 0 and i > 0:\n                    logging.info('samples : {0}'.format(i))\n                    r1_i2t, r5_i2t, r10_i2t, medr_i2t = self.i2t(devImg,\n                                                                 devTxt)\n                    logging.info(\"Image to text: {0}, {1}, {2}, {3}\".format(\n                        r1_i2t, r5_i2t, r10_i2t, medr_i2t))\n                    # Compute test ranks txt2img\n                    r1_t2i, r5_t2i, r10_t2i, medr_t2i = self.t2i(devImg,\n                                                                 devTxt)\n                    logging.info(\"Text to Image: {0}, {1}, {2}, {3}\".format(\n                        r1_t2i, r5_t2i, r10_t2i, medr_t2i))\n                idx = torch.LongTensor(permutation[i:i + self.batch_size])\n                imgbatch = Variable(trainImg.index_select(0, idx)).cuda()\n                sentbatch = Variable(trainTxt.index_select(0, idx)).cuda()\n\n                idximgc = np.random.choice(permutation[:i] +\n                                           permutation[i + self.batch_size:],\n                                           self.ncontrast*idx.size(0))\n                idxsentc = np.random.choice(permutation[:i] +\n                                            permutation[i + self.batch_size:],\n                                            self.ncontrast*idx.size(0))\n                idximgc = torch.LongTensor(idximgc)\n                idxsentc = torch.LongTensor(idxsentc)\n                # Get indexes for contrastive images and sentences\n                imgcbatch = Variable(trainImg.index_select(0, idximgc)).view(\n                    -1, self.ncontrast, self.imgdim).cuda()\n                sentcbatch = Variable(trainTxt.index_select(0, idxsentc)).view(\n                    -1, self.ncontrast, self.sentdim).cuda()\n\n                anchor1, anchor2, img_sentc, sent_imgc = self.model(\n                    imgbatch, sentbatch, imgcbatch, sentcbatch)\n                # loss\n                loss = self.loss_fn(anchor1, anchor2, img_sentc, sent_imgc)\n                all_costs.append(loss.data.item())\n                # backward\n                self.optimizer.zero_grad()\n                loss.backward()\n                # Update parameters\n                self.optimizer.step()\n        self.nepoch += nepoches\n\n    def t2i(self, images, captions):\n        \"\"\"\n        Images: (5N, imgdim) matrix of images\n        Captions: (5N, sentdim) matrix of captions\n        \"\"\"\n        with torch.no_grad():\n            # Project images and captions\n            img_embed, sent_embed = [], []\n            for i in range(0, len(images), self.batch_size):\n                img_embed.append(self.model.proj_image(\n                    Variable(images[i:i + self.batch_size])))\n                sent_embed.append(self.model.proj_sentence(\n                    Variable(captions[i:i + self.batch_size])))\n            img_embed = torch.cat(img_embed, 0).data\n            sent_embed = torch.cat(sent_embed, 0).data\n\n            npts = int(img_embed.size(0) / 5)\n            idxs = torch.cuda.LongTensor(range(0, len(img_embed), 5))\n            ims = img_embed.index_select(0, idxs)\n\n            ranks = np.zeros(5 * npts)\n            for index in range(npts):\n\n                # Get query captions\n                queries = sent_embed[5*index: 5*index + 5]\n\n                # Compute scores\n                scores = torch.mm(queries, ims.transpose(0, 1)).cpu().numpy()\n                inds = np.zeros(scores.shape)\n                for i in range(len(inds)):\n                    inds[i] = np.argsort(scores[i])[::-1]\n                    ranks[5 * index + i] = np.where(inds[i] == index)[0][0]\n\n            # Compute metrics\n            r1 = 100.0 * len(np.where(ranks < 1)[0]) / len(ranks)\n            r5 = 100.0 * len(np.where(ranks < 5)[0]) / len(ranks)\n            r10 = 100.0 * len(np.where(ranks < 10)[0]) / len(ranks)\n            medr = np.floor(np.median(ranks)) + 1\n            return (r1, r5, r10, medr)\n\n    def i2t(self, images, captions):\n        \"\"\"\n        Images: (5N, imgdim) matrix of images\n        Captions: (5N, sentdim) matrix of captions\n        \"\"\"\n        with torch.no_grad():\n            # Project images and captions\n            img_embed, sent_embed = [], []\n            for i in range(0, len(images), self.batch_size):\n                img_embed.append(self.model.proj_image(\n                    Variable(images[i:i + self.batch_size])))\n                sent_embed.append(self.model.proj_sentence(\n                    Variable(captions[i:i + self.batch_size])))\n            img_embed = torch.cat(img_embed, 0).data\n            sent_embed = torch.cat(sent_embed, 0).data\n\n            npts = int(img_embed.size(0) / 5)\n            index_list = []\n\n            ranks = np.zeros(npts)\n            for index in range(npts):\n\n                # Get query image\n                query_img = img_embed[5 * index]\n\n                # Compute scores\n                scores = torch.mm(query_img.view(1, -1),\n                                  sent_embed.transpose(0, 1)).view(-1)\n                scores = scores.cpu().numpy()\n                inds = np.argsort(scores)[::-1]\n                index_list.append(inds[0])\n\n                # Score\n                rank = 1e20\n                for i in range(5*index, 5*index + 5, 1):\n                    tmp = np.where(inds == i)[0][0]\n                    if tmp < rank:\n                        rank = tmp\n                ranks[index] = rank\n\n            # Compute metrics\n            r1 = 100.0 * len(np.where(ranks < 1)[0]) / len(ranks)\n            r5 = 100.0 * len(np.where(ranks < 5)[0]) / len(ranks)\n            r10 = 100.0 * len(np.where(ranks < 10)[0]) / len(ranks)\n            medr = np.floor(np.median(ranks)) + 1\n            return (r1, r5, r10, medr)\n",
    "SentEval/senteval/tools/relatedness.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n\"\"\"\nSemantic Relatedness (supervised) with Pytorch\n\"\"\"\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport copy\nimport numpy as np\n\nimport torch\nfrom torch import nn\nimport torch.optim as optim\n\nfrom scipy.stats import pearsonr, spearmanr\n\n\nclass RelatednessPytorch(object):\n    # Can be used for SICK-Relatedness, and STS14\n    def __init__(self, train, valid, test, devscores, config):\n        # fix seed\n        np.random.seed(config['seed'])\n        torch.manual_seed(config['seed'])\n        assert torch.cuda.is_available(), 'torch.cuda required for Relatedness'\n        torch.cuda.manual_seed(config['seed'])\n\n        self.train = train\n        self.valid = valid\n        self.test = test\n        self.devscores = devscores\n\n        self.inputdim = train['X'].shape[1]\n        self.nclasses = config['nclasses']\n        self.seed = config['seed']\n        self.l2reg = 0.\n        self.batch_size = 64\n        self.maxepoch = 1000\n        self.early_stop = True\n\n        self.model = nn.Sequential(\n            nn.Linear(self.inputdim, self.nclasses),\n            nn.Softmax(dim=-1),\n        )\n        self.loss_fn = nn.MSELoss()\n\n        if torch.cuda.is_available():\n            self.model = self.model.cuda()\n            self.loss_fn = self.loss_fn.cuda()\n\n        self.loss_fn.size_average = False\n        self.optimizer = optim.Adam(self.model.parameters(),\n                                    weight_decay=self.l2reg)\n\n    def prepare_data(self, trainX, trainy, devX, devy, testX, testy):\n        # Transform probs to log-probs for KL-divergence\n        trainX = torch.from_numpy(trainX).float().cuda()\n        trainy = torch.from_numpy(trainy).float().cuda()\n        devX = torch.from_numpy(devX).float().cuda()\n        devy = torch.from_numpy(devy).float().cuda()\n        testX = torch.from_numpy(testX).float().cuda()\n        testY = torch.from_numpy(testy).float().cuda()\n\n        return trainX, trainy, devX, devy, testX, testy\n\n    def run(self):\n        self.nepoch = 0\n        bestpr = -1\n        early_stop_count = 0\n        r = np.arange(1, 6)\n        stop_train = False\n\n        # Preparing data\n        trainX, trainy, devX, devy, testX, testy = self.prepare_data(\n            self.train['X'], self.train['y'],\n            self.valid['X'], self.valid['y'],\n            self.test['X'], self.test['y'])\n\n        # Training\n        while not stop_train and self.nepoch <= self.maxepoch:\n            self.trainepoch(trainX, trainy, nepoches=50)\n            yhat = np.dot(self.predict_proba(devX), r)\n            pr = spearmanr(yhat, self.devscores)[0]\n            pr = 0 if pr != pr else pr  # if NaN bc std=0\n            # early stop on Pearson\n            if pr > bestpr:\n                bestpr = pr\n                bestmodel = copy.deepcopy(self.model)\n            elif self.early_stop:\n                if early_stop_count >= 3:\n                    stop_train = True\n                early_stop_count += 1\n        self.model = bestmodel\n\n        yhat = np.dot(self.predict_proba(testX), r)\n\n        return bestpr, yhat\n\n    def trainepoch(self, X, y, nepoches=1):\n        self.model.train()\n        for _ in range(self.nepoch, self.nepoch + nepoches):\n            permutation = np.random.permutation(len(X))\n            all_costs = []\n            for i in range(0, len(X), self.batch_size):\n                # forward\n                idx = torch.from_numpy(permutation[i:i + self.batch_size]).long().cuda()\n                Xbatch = X[idx]\n                ybatch = y[idx]\n                output = self.model(Xbatch)\n                # loss\n                loss = self.loss_fn(output, ybatch)\n                all_costs.append(loss.item())\n                # backward\n                self.optimizer.zero_grad()\n                loss.backward()\n                # Update parameters\n                self.optimizer.step()\n        self.nepoch += nepoches\n\n    def predict_proba(self, devX):\n        self.model.eval()\n        probas = []\n        with torch.no_grad():\n            for i in range(0, len(devX), self.batch_size):\n                Xbatch = devX[i:i + self.batch_size]\n                if len(probas) == 0:\n                    probas = self.model(Xbatch).data.cpu().numpy()\n                else:\n                    probas = np.concatenate((probas, self.model(Xbatch).data.cpu().numpy()), axis=0)\n        return probas\n",
    "SentEval/senteval/tools/validation.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n\"\"\"\nValidation and classification\n(train)            :  inner-kfold classifier\n(train, test)      :  kfold classifier\n(train, dev, test) :  split classifier\n\n\"\"\"\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport logging\nimport numpy as np\nfrom senteval.tools.classifier import MLP\n\nimport sklearn\nassert(sklearn.__version__ >= \"0.18.0\"), \\\n    \"need to update sklearn to version >= 0.18.0\"\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import StratifiedKFold\n\n\ndef get_classif_name(classifier_config, usepytorch):\n    if not usepytorch:\n        modelname = 'sklearn-LogReg'\n    else:\n        nhid = classifier_config['nhid']\n        optim = 'adam' if 'optim' not in classifier_config else classifier_config['optim']\n        bs = 64 if 'batch_size' not in classifier_config else classifier_config['batch_size']\n        modelname = 'pytorch-MLP-nhid%s-%s-bs%s' % (nhid, optim, bs)\n    return modelname\n\n# Pytorch version\nclass InnerKFoldClassifier(object):\n    \"\"\"\n    (train) split classifier : InnerKfold.\n    \"\"\"\n    def __init__(self, X, y, config):\n        self.X = X\n        self.y = y\n        self.featdim = X.shape[1]\n        self.nclasses = config['nclasses']\n        self.seed = config['seed']\n        self.devresults = []\n        self.testresults = []\n        self.usepytorch = config['usepytorch']\n        self.classifier_config = config['classifier']\n        self.modelname = get_classif_name(self.classifier_config, self.usepytorch)\n\n        self.k = 5 if 'kfold' not in config else config['kfold']\n\n    def run(self):\n        logging.info('Training {0} with (inner) {1}-fold cross-validation'\n                     .format(self.modelname, self.k))\n\n        regs = [10**t for t in range(-5, -1)] if self.usepytorch else \\\n               [2**t for t in range(-2, 4, 1)]\n        skf = StratifiedKFold(n_splits=self.k, shuffle=True, random_state=1111)\n        innerskf = StratifiedKFold(n_splits=self.k, shuffle=True,\n                                   random_state=1111)\n        count = 0\n        for train_idx, test_idx in skf.split(self.X, self.y):\n            count += 1\n            X_train, X_test = self.X[train_idx], self.X[test_idx]\n            y_train, y_test = self.y[train_idx], self.y[test_idx]\n            scores = []\n            for reg in regs:\n                regscores = []\n                for inner_train_idx, inner_test_idx in innerskf.split(X_train, y_train):\n                    X_in_train, X_in_test = X_train[inner_train_idx], X_train[inner_test_idx]\n                    y_in_train, y_in_test = y_train[inner_train_idx], y_train[inner_test_idx]\n                    if self.usepytorch:\n                        clf = MLP(self.classifier_config, inputdim=self.featdim,\n                                  nclasses=self.nclasses, l2reg=reg,\n                                  seed=self.seed)\n                        clf.fit(X_in_train, y_in_train,\n                                validation_data=(X_in_test, y_in_test))\n                    else:\n                        clf = LogisticRegression(C=reg, random_state=self.seed)\n                        clf.fit(X_in_train, y_in_train)\n                    regscores.append(clf.score(X_in_test, y_in_test))\n                scores.append(round(100*np.mean(regscores), 2))\n            optreg = regs[np.argmax(scores)]\n            logging.info('Best param found at split {0}: l2reg = {1} \\\n                with score {2}'.format(count, optreg, np.max(scores)))\n            self.devresults.append(np.max(scores))\n\n            if self.usepytorch:\n                clf = MLP(self.classifier_config, inputdim=self.featdim,\n                          nclasses=self.nclasses, l2reg=optreg,\n                          seed=self.seed)\n\n                clf.fit(X_train, y_train, validation_split=0.05)\n            else:\n                clf = LogisticRegression(C=optreg, random_state=self.seed)\n                clf.fit(X_train, y_train)\n\n            self.testresults.append(round(100*clf.score(X_test, y_test), 2))\n\n        devaccuracy = round(np.mean(self.devresults), 2)\n        testaccuracy = round(np.mean(self.testresults), 2)\n        return devaccuracy, testaccuracy\n\n\nclass KFoldClassifier(object):\n    \"\"\"\n    (train, test) split classifier : cross-validation on train.\n    \"\"\"\n    def __init__(self, train, test, config):\n        self.train = train\n        self.test = test\n        self.featdim = self.train['X'].shape[1]\n        self.nclasses = config['nclasses']\n        self.seed = config['seed']\n        self.usepytorch = config['usepytorch']\n        self.classifier_config = config['classifier']\n        self.modelname = get_classif_name(self.classifier_config, self.usepytorch)\n\n        self.k = 5 if 'kfold' not in config else config['kfold']\n\n    def run(self):\n        # cross-validation\n        logging.info('Training {0} with {1}-fold cross-validation'\n                     .format(self.modelname, self.k))\n        regs = [10**t for t in range(-5, -1)] if self.usepytorch else \\\n               [2**t for t in range(-1, 6, 1)]\n        skf = StratifiedKFold(n_splits=self.k, shuffle=True,\n                              random_state=self.seed)\n        scores = []\n\n        for reg in regs:\n            scanscores = []\n            for train_idx, test_idx in skf.split(self.train['X'],\n                                                 self.train['y']):\n                # Split data\n                X_train, y_train = self.train['X'][train_idx], self.train['y'][train_idx]\n\n                X_test, y_test = self.train['X'][test_idx], self.train['y'][test_idx]\n\n                # Train classifier\n                if self.usepytorch:\n                    clf = MLP(self.classifier_config, inputdim=self.featdim,\n                              nclasses=self.nclasses, l2reg=reg,\n                              seed=self.seed)\n                    clf.fit(X_train, y_train, validation_data=(X_test, y_test))\n                else:\n                    clf = LogisticRegression(C=reg, random_state=self.seed)\n                    clf.fit(X_train, y_train)\n                score = clf.score(X_test, y_test)\n                scanscores.append(score)\n            # Append mean score\n            scores.append(round(100*np.mean(scanscores), 2))\n\n        # evaluation\n        logging.info([('reg:' + str(regs[idx]), scores[idx])\n                      for idx in range(len(scores))])\n        optreg = regs[np.argmax(scores)]\n        devaccuracy = np.max(scores)\n        logging.info('Cross-validation : best param found is reg = {0} \\\n            with score {1}'.format(optreg, devaccuracy))\n\n        logging.info('Evaluating...')\n        if self.usepytorch:\n            clf = MLP(self.classifier_config, inputdim=self.featdim,\n                      nclasses=self.nclasses, l2reg=optreg,\n                      seed=self.seed)\n            clf.fit(self.train['X'], self.train['y'], validation_split=0.05)\n        else:\n            clf = LogisticRegression(C=optreg, random_state=self.seed)\n            clf.fit(self.train['X'], self.train['y'])\n        yhat = clf.predict(self.test['X'])\n\n        testaccuracy = clf.score(self.test['X'], self.test['y'])\n        testaccuracy = round(100*testaccuracy, 2)\n\n        return devaccuracy, testaccuracy, yhat\n\n\nclass SplitClassifier(object):\n    \"\"\"\n    (train, valid, test) split classifier.\n    \"\"\"\n    def __init__(self, X, y, config):\n        self.X = X\n        self.y = y\n        self.nclasses = config['nclasses']\n        self.featdim = self.X['train'].shape[1]\n        self.seed = config['seed']\n        self.usepytorch = config['usepytorch']\n        self.classifier_config = config['classifier']\n        self.cudaEfficient = False if 'cudaEfficient' not in config else \\\n            config['cudaEfficient']\n        self.modelname = get_classif_name(self.classifier_config, self.usepytorch)\n        self.noreg = False if 'noreg' not in config else config['noreg']\n        self.config = config\n\n    def run(self):\n        logging.info('Training {0} with standard validation..'\n                     .format(self.modelname))\n        regs = [10**t for t in range(-5, -1)] if self.usepytorch else \\\n               [2**t for t in range(-2, 4, 1)]\n        if self.noreg:\n            regs = [1e-9 if self.usepytorch else 1e9]\n        scores = []\n        for reg in regs:\n            if self.usepytorch:\n                clf = MLP(self.classifier_config, inputdim=self.featdim,\n                          nclasses=self.nclasses, l2reg=reg,\n                          seed=self.seed, cudaEfficient=self.cudaEfficient)\n\n                # TODO: Find a hack for reducing nb epoches in SNLI\n                clf.fit(self.X['train'], self.y['train'],\n                        validation_data=(self.X['valid'], self.y['valid']))\n            else:\n                clf = LogisticRegression(C=reg, random_state=self.seed)\n                clf.fit(self.X['train'], self.y['train'])\n            scores.append(round(100*clf.score(self.X['valid'],\n                                self.y['valid']), 2))\n        logging.info([('reg:'+str(regs[idx]), scores[idx])\n                      for idx in range(len(scores))])\n        optreg = regs[np.argmax(scores)]\n        devaccuracy = np.max(scores)\n        logging.info('Validation : best param found is reg = {0} with score \\\n            {1}'.format(optreg, devaccuracy))\n        clf = LogisticRegression(C=optreg, random_state=self.seed)\n        logging.info('Evaluating...')\n        if self.usepytorch:\n            clf = MLP(self.classifier_config, inputdim=self.featdim,\n                      nclasses=self.nclasses, l2reg=optreg,\n                      seed=self.seed, cudaEfficient=self.cudaEfficient)\n\n            # TODO: Find a hack for reducing nb epoches in SNLI\n            clf.fit(self.X['train'], self.y['train'],\n                    validation_data=(self.X['valid'], self.y['valid']))\n        else:\n            clf = LogisticRegression(C=optreg, random_state=self.seed)\n            clf.fit(self.X['train'], self.y['train'])\n\n        testaccuracy = clf.score(self.X['test'], self.y['test'])\n        testaccuracy = round(100*testaccuracy, 2)\n        return devaccuracy, testaccuracy\n",
    "SentEval/senteval/trec.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\n'''\nTREC question-type classification\n'''\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport os\nimport io\nimport logging\nimport numpy as np\n\nfrom senteval.tools.validation import KFoldClassifier\n\n\nclass TRECEval(object):\n    def __init__(self, task_path, seed=1111):\n        logging.info('***** Transfer task : TREC *****\\n\\n')\n        self.seed = seed\n        self.train = self.loadFile(os.path.join(task_path, 'train_5500.label'))\n        self.test = self.loadFile(os.path.join(task_path, 'TREC_10.label'))\n\n    def do_prepare(self, params, prepare):\n        samples = self.train['X'] + self.test['X']\n        return prepare(params, samples)\n\n    def loadFile(self, fpath):\n        trec_data = {'X': [], 'y': []}\n        tgt2idx = {'ABBR': 0, 'DESC': 1, 'ENTY': 2,\n                   'HUM': 3, 'LOC': 4, 'NUM': 5}\n        with io.open(fpath, 'r', encoding='latin-1') as f:\n            for line in f:\n                target, sample = line.strip().split(':', 1)\n                sample = sample.split(' ', 1)[1].split()\n                assert target in tgt2idx, target\n                trec_data['X'].append(sample)\n                trec_data['y'].append(tgt2idx[target])\n        return trec_data\n\n    def run(self, params, batcher):\n        train_embeddings, test_embeddings = [], []\n\n        # Sort to reduce padding\n        sorted_corpus_train = sorted(zip(self.train['X'], self.train['y']),\n                                     key=lambda z: (len(z[0]), z[1]))\n        train_samples = [x for (x, y) in sorted_corpus_train]\n        train_labels = [y for (x, y) in sorted_corpus_train]\n\n        sorted_corpus_test = sorted(zip(self.test['X'], self.test['y']),\n                                    key=lambda z: (len(z[0]), z[1]))\n        test_samples = [x for (x, y) in sorted_corpus_test]\n        test_labels = [y for (x, y) in sorted_corpus_test]\n\n        # Get train embeddings\n        for ii in range(0, len(train_labels), params.batch_size):\n            batch = train_samples[ii:ii + params.batch_size]\n            embeddings = batcher(params, batch)\n            train_embeddings.append(embeddings)\n        train_embeddings = np.vstack(train_embeddings)\n        logging.info('Computed train embeddings')\n\n        # Get test embeddings\n        for ii in range(0, len(test_labels), params.batch_size):\n            batch = test_samples[ii:ii + params.batch_size]\n            embeddings = batcher(params, batch)\n            test_embeddings.append(embeddings)\n        test_embeddings = np.vstack(test_embeddings)\n        logging.info('Computed test embeddings')\n\n        config_classifier = {'nclasses': 6, 'seed': self.seed,\n                             'usepytorch': params.usepytorch,\n                             'classifier': params.classifier,\n                             'kfold': params.kfold}\n        clf = KFoldClassifier({'X': train_embeddings,\n                               'y': np.array(train_labels)},\n                              {'X': test_embeddings,\n                               'y': np.array(test_labels)},\n                              config_classifier)\n        devacc, testacc, _ = clf.run()\n        logging.debug('\\nDev acc : {0} Test acc : {1} \\\n            for TREC\\n'.format(devacc, testacc))\n        return {'devacc': devacc, 'acc': testacc,\n                'ndev': len(self.train['X']), 'ntest': len(self.test['X'])}\n",
    "SentEval/senteval/utils.py": "# Copyright (c) 2017-present, Facebook, Inc.\n# All rights reserved.\n#\n# This source code is licensed under the license found in the\n# LICENSE file in the root directory of this source tree.\n#\n\nfrom __future__ import absolute_import, division, unicode_literals\n\nimport numpy as np\nimport re\nimport inspect\nfrom torch import optim\n\n\ndef create_dictionary(sentences):\n    words = {}\n    for s in sentences:\n        for word in s:\n            if word in words:\n                words[word] += 1\n            else:\n                words[word] = 1\n    words['<s>'] = 1e9 + 4\n    words['</s>'] = 1e9 + 3\n    words['<p>'] = 1e9 + 2\n    # words['<UNK>'] = 1e9 + 1\n    sorted_words = sorted(words.items(), key=lambda x: -x[1])  # inverse sort\n    id2word = []\n    word2id = {}\n    for i, (w, _) in enumerate(sorted_words):\n        id2word.append(w)\n        word2id[w] = i\n\n    return id2word, word2id\n\n\ndef cosine(u, v):\n    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))\n\n\nclass dotdict(dict):\n    \"\"\" dot.notation access to dictionary attributes \"\"\"\n    __getattr__ = dict.get\n    __setattr__ = dict.__setitem__\n    __delattr__ = dict.__delitem__\n\n\ndef get_optimizer(s):\n    \"\"\"\n    Parse optimizer parameters.\n    Input should be of the form:\n        - \"sgd,lr=0.01\"\n        - \"adagrad,lr=0.1,lr_decay=0.05\"\n    \"\"\"\n    if \",\" in s:\n        method = s[:s.find(',')]\n        optim_params = {}\n        for x in s[s.find(',') + 1:].split(','):\n            split = x.split('=')\n            assert len(split) == 2\n            assert re.match(\"^[+-]?(\\d+(\\.\\d*)?|\\.\\d+)$\", split[1]) is not None\n            optim_params[split[0]] = float(split[1])\n    else:\n        method = s\n        optim_params = {}\n\n    if method == 'adadelta':\n        optim_fn = optim.Adadelta\n    elif method == 'adagrad':\n        optim_fn = optim.Adagrad\n    elif method == 'adam':\n        optim_fn = optim.Adam\n    elif method == 'adamax':\n        optim_fn = optim.Adamax\n    elif method == 'asgd':\n        optim_fn = optim.ASGD\n    elif method == 'rmsprop':\n        optim_fn = optim.RMSprop\n    elif method == 'rprop':\n        optim_fn = optim.Rprop\n    elif method == 'sgd':\n        optim_fn = optim.SGD\n        assert 'lr' in optim_params\n    else:\n        raise Exception('Unknown optimization method: \"%s\"' % method)\n\n    # check that we give good parameters to the optimizer\n    try:\n        expected_args = inspect.getargspec(optim_fn.__init__)[0]\n    except ValueError:\n        expected_args = inspect.getfullargspec(optim_fn.__init__)[0]\n    assert expected_args[:2] == ['self', 'params']\n    if not all(k in expected_args[2:] for k in optim_params.keys()):\n        raise Exception('Unexpected parameters: expected \"%s\", got \"%s\"' % (\n            str(expected_args[2:]), str(optim_params.keys())))\n\n    return optim_fn, optim_params\n",
}

for rel_path, content in FILES.items():
    path = PROJECT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

print(f"Wrote {len(FILES)} files to {PROJECT_DIR}")


In [ ]:
# Write Kaggle-specific config
import os
import sys
from pathlib import Path

import yaml

PROJECT_DIR = Path("/kaggle/working/token_prepending")
sys.path.insert(0, str(PROJECT_DIR))

CONFIG_NAME = os.environ.get("TP_CONFIG_NAME", "kaggle-qwen2.5-7b-vi-tp-t4x2")
MODEL_NAME_OR_PATH = os.environ.get("MODEL_NAME_OR_PATH", "Qwen/Qwen2.5-7B")

config = {
    "default_config": CONFIG_NAME,
    "gpu_config": {"cuda_visible_devices": "0,1"},
    "models": {
        CONFIG_NAME: {
            "model_name_or_path": MODEL_NAME_OR_PATH,
            "use_which_plan": "tp",
            "output_layer": int(os.environ.get("OUTPUT_LAYER", "-2")),
            "tp_starting_index": int(os.environ.get("TP_STARTING_INDEX", "1")),
            "tp_exiting_index": int(os.environ.get("TP_EXITING_INDEX", "6")),
            "batch_size": int(os.environ.get("BATCH_SIZE", "1")),
            "mode": "test",
            "task_set": "vi-sts",
            "prompt_method": os.environ.get("PROMPT_METHOD", "cot"),
            "device": "cuda",
            "cache_dir": "/kaggle/working/hf-cache",
            "vietnamese_dataset_name": os.environ.get(
                "VIETNAMESE_STS_DATASET",
                "nemixo/stsbenchmark-sts-vietnamese",
            ),
            "vietnamese_split": os.environ.get("VIETNAMESE_STS_SPLIT", "test"),
        }
    },
}

(PROJECT_DIR / "config.yaml").write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
print((PROJECT_DIR / "config.yaml").read_text())


In [ ]:
# Run default Vietnamese STS evaluation
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/token_prepending")
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable,
    "evaluate.py",
    "--config",
    CONFIG_NAME,
    "--config_file",
    "config.yaml",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## Optional English SentEval

To run `sts`, `stsb`, `transfer`, or `full`, attach/download SentEval data into `/kaggle/working/token_prepending/SentEval/data`, then edit `task_set` in the config cell. The default notebook avoids this because the repo does not bundle SentEval data.